# Final Source-Disjoint Generalisation Evaluation

This notebook performs the **final generalisation test** of the thesis by comparing:

```text
Frozen Setup F1
        vs
Fine-tuned Setup F1 + QLoRA
```

on the completely unseen source-disjoint database constructed in the preceding notebook.

At this stage, all design choices are frozen:

- the participant-centric evidence pipeline;
- the Structured R1 reasoning format;
- the selected participation representation (`speaks` only);
- the selected local temporal representation (response offsets only);
- the full global temporal branch;
- the revised semantic policy;
- the Setup F1 semantic payload;
- the trained QLoRA adapter.

No prompt development, feature selection, or fine-tuning decision is made using these final test cases.

---

## Final unseen evaluation set

The evaluation database contains:

```text
17 NORMAL
17 LAG
17 WRONG PARTNER
17 SILENT PARTNER
-----------------
68 total cases
```

All 17 source conversations are source-disjoint from the conversations used during the earlier development, ablation, and adapter-training stages.

This notebook therefore tests whether the selected frozen reasoner and the targeted fine-tuned adapter generalise beyond the conversations used to design the system.

---

# Evaluation Objective

The fine-tuning research question is:

> **Can lightweight targeted fine-tuning improve the semantic fidelity of the unified Setup F1 reasoner for Wrong Partner detection while preserving its NORMAL, temporal-LAG, Silent Partner, structured-reasoning, and final-classification behaviour?**

The final evaluation therefore considers two complementary outcomes:

1. **classification generalisation** across all four case families;
2. **semantic fidelity** on Wrong Partner cases.

The fine-tuned adapter is not considered successful based on overall accuracy alone. Its intended effect is specifically to improve the semantic interpretation of Wrong Partner cases without substantially damaging the rest of the unified reasoner.

---

# Final Classification Results

The final source-disjoint classification results are:

| Case family | Frozen F1 | Fine-tuned F1 |
|---|---:|---:|
| NORMAL | 11/17 (64.71%) | **15/17 (88.24%)** |
| LAG | **12/17 (70.59%)** | 10/17 (58.82%) |
| Wrong Partner | 16/17 (94.12%) | 16/17 (94.12%) |
| Silent Partner | 17/17 (100%) | 17/17 (100%) |
| **Overall** | **56/68 (82.35%)** | **58/68 (85.29%)** |

The fine-tuned model therefore improves overall source-disjoint accuracy from:

```text
82.35% → 85.29%
```

while producing a clear redistribution of class-specific behaviour:

- NORMAL preservation improves substantially;
- Wrong Partner classification is preserved;
- Silent Partner performance remains perfect;
- LAG performance decreases.

This class-level trade-off is important and is retained explicitly rather than being hidden by the overall accuracy increase.

---

# Wrong Partner Semantic Assessment

The central target of the QLoRA experiment is the semantic assessment of Wrong Partner cases.

The final source-disjoint semantic-assessment distributions are:

| Model | COMPATIBLE | INCOMPATIBLE | LIMITED |
|---|---:|---:|---:|
| Frozen F1 | 7 | 6 | 4 |
| Fine-tuned F1 | 3 | **12** | 2 |

The number of unseen Wrong Partner cases explicitly recognised as semantically incompatible therefore doubles:

```text
Frozen F1:
6 / 17 → INCOMPATIBLE

Fine-tuned F1:
12 / 17 → INCOMPATIBLE
```

At the same time:

```text
COMPATIBLE:
7 → 3

LIMITED:
4 → 2
```

This is the key semantic-fidelity result of the final evaluation.

The adapter does not merely preserve Wrong Partner classification accuracy; it changes the structured semantic assessment in the intended direction, making the reasoner's observable semantic judgement more consistent with the actual anomaly type.

---

# Interpretation

The final unseen evaluation therefore provides evidence for the original fine-tuning hypothesis.

The fine-tuned F1 reasoner:

- improves overall classification from **82.35% to 85.29%**;
- substantially improves NORMAL preservation;
- preserves Wrong Partner classification at **16/17**;
- preserves perfect Silent Partner classification at **17/17**;
- doubles Wrong Partner `INCOMPATIBLE` assessments from **6/17 to 12/17**;
- but reduces LAG accuracy from **12/17 to 10/17**.

The most important result is not the modest gain in overall accuracy.

It is the combination of:

```text
preserved Wrong Partner classification
+
substantially improved Wrong Partner semantic assessment
```

on completely unseen source conversations.

This supports the conclusion that lightweight targeted QLoRA adaptation can improve the **semantic fidelity** of the unified Setup F1 reasoner, although the reduced LAG performance shows that the adaptation is not perfectly branch-isolated and that some cross-branch trade-offs remain.

---

# Role in the Thesis

This notebook is the final experimental stage of the repository:

```text
development and isolated branches
        ↓
consolidation formats
        ↓
controlled branch ablations
        ↓
semantic policy refinement
        ↓
Setup F1 selection
        ↓
targeted QLoRA fine-tuning
        ↓
final source-disjoint unseen evaluation
```

Unlike the earlier development-set experiments and the adapter-development held-out comparison, this stage uses source conversations that were not part of the system-design process.

It therefore provides the final evidence used to discuss **generalisation**, **semantic fidelity**, and the remaining trade-offs introduced by targeted fine-tuning.

> **Reproducibility note:** every code cell, saved output, execution count, code-cell metadata field, inference result, comparison table, semantic-assessment distribution, statistical test, and diagnostic output is preserved exactly as executed. Only this introductory Markdown cell has been rewritten for the public repository.

## 1. Install the exact runtime dependencies

In [ ]:
# Install the exact runtime dependencies used by the F1 experiment
# plus PEFT for LoRA/QLoRA fine-tuning.
!pip uninstall -y transformers
!pip install -U "transformers>=4.57.0" accelerate bitsandbytes sentencepiece peft
!pip install -U qwen-omni-utils decord ffmpeg-python scikit-learn scipy


Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 142.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 66.7 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.19.1
    Uninstalling peft-0.19.1:
      Successfully uninstalled peft-0.19.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 151.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 161.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 80.0 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninsta

## 2. Mount Drive and audit all read-only inputs and isolated outputs

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import Counter
import copy
import gc
import hashlib
import json
import time

import pandas as pd
from IPython.display import display

# ============================================================
# READ-ONLY SOURCE ARTIFACTS FROM THE ORIGINAL F1 EXPERIMENT
# ============================================================

SOURCE_OUT_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec"
)

# OUT_DIR remains the original source root because the exact saved
# prompt_template.txt used to reconstruct F1 lives under this root.
# No new prediction output is written there by this notebook.
OUT_DIR = SOURCE_OUT_DIR

REFERENCE_BASE_STATS_PATH = (
    SOURCE_OUT_DIR
    / "frozen_reference_base_statistics.json"
)

REFERENCE_SHIFT_STATS_PATH = (
    SOURCE_OUT_DIR
    / "frozen_reference_global_shift_statistics.json"
)

MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

MODEL_PATH = Path(
    "/content/drive/MyDrive/Qwen2.5-Omni-7B"
)

ORIGINAL_F1_CACHE_PATH = (
    SOURCE_OUT_DIR
    / "structured_r1_improved_semantic_payload_ablation"
    / "f1"
    / "predictions_cache.json"
)

ORIGINAL_FINE_TUNING_ROOT = (
    SOURCE_OUT_DIR
    / "structured_r1_f1_semantic_targeted_finetuning"
)

SPLIT_SEED = 42
LORA_RANK = 8
LORA_ALPHA = 16
PRESERVATION_LAMBDA = 0.50
LEARNING_RATE = 2e-5

ORIGINAL_TRAINING_RUN_DIR = (
    ORIGINAL_FINE_TUNING_ROOT
    / (
        f"qlora_r{LORA_RANK}_"
        f"lambda_{str(PRESERVATION_LAMBDA).replace('.', '_')}_"
        f"seed_{SPLIT_SEED}"
    )
)

BEST_ADAPTER_DIR = (
    ORIGINAL_TRAINING_RUN_DIR
    / "best_adapter"
)

ORIGINAL_SPLIT_MANIFEST_PATH = (
    ORIGINAL_FINE_TUNING_ROOT
    / "grouped_split_manifest.json"
)

# ============================================================
# NEW, ISOLATED FINAL-UNSEEN INPUT AND OUTPUT LOCATIONS
# ============================================================

FINAL_UNSEEN_DATABASE_DIR = Path(
    "/content/drive/MyDrive/"
    "final_test_completely_unseen_database"
)

FINAL_DATABASE_PATH = (
    FINAL_UNSEEN_DATABASE_DIR
    / "final_unseen_all_68_cases_with_temporal_and_semantic_summaries.json"
)

FINAL_EVALUATION_ROOT = (
    FINAL_UNSEEN_DATABASE_DIR
    / "f1_frozen_vs_finetuned_final_unseen_evaluation"
)

FROZEN_RESULTS_DIR = (
    FINAL_EVALUATION_ROOT
    / "frozen_f1"
)

FINE_TUNED_RESULTS_DIR = (
    FINAL_EVALUATION_ROOT
    / "fine_tuned_f1"
)

PAIRED_RESULTS_DIR = (
    FINAL_EVALUATION_ROOT
    / "paired_comparison"
)

for directory in [
    FINAL_EVALUATION_ROOT,
    FROZEN_RESULTS_DIR,
    FINE_TUNED_RESULTS_DIR,
    PAIRED_RESULTS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

required_paths = {
    "Frozen base statistics": REFERENCE_BASE_STATS_PATH,
    "Frozen global-shift statistics": REFERENCE_SHIFT_STATS_PATH,
    "Final unseen 68-case database": FINAL_DATABASE_PATH,
    "Local Qwen checkpoint": MODEL_PATH,
    "Original F1 cache for prompt-hash audit": ORIGINAL_F1_CACHE_PATH,
    "Saved best fine-tuned adapter": BEST_ADAPTER_DIR,
    "Original split manifest": ORIGINAL_SPLIT_MANIFEST_PATH,
}

print("=" * 100)
print("FINAL-UNSEEN ARTIFACT PATH AUDIT")
print("=" * 100)

for name, path in required_paths.items():
    print(f"{name}: {path}")
    print("  exists:", path.exists())

for name, path in required_paths.items():
    assert path.exists(), f"Required artifact not found: {name}\n{path}"

assert (BEST_ADAPTER_DIR / "adapter_config.json").exists()

assert any(
    path.exists()
    for path in [
        BEST_ADAPTER_DIR / "adapter_model.safetensors",
        BEST_ADAPTER_DIR / "adapter_model.bin",
    ]
)

print("\nAll source artifacts exist.")
print("All new results will be written only under:")
print(FINAL_EVALUATION_ROOT)


Mounted at /content/drive
FINAL-UNSEEN ARTIFACT PATH AUDIT
Frozen base statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_base_statistics.json
  exists: True
Frozen global-shift statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_global_shift_statistics.json
  exists: True
Final unseen 68-case database: /content/drive/MyDrive/final_test_completely_unseen_database/final_unseen_all_68_cases_with_temporal_and_semantic_summaries.json
  exists: True
Local Qwen checkpoint: /content/drive/MyDrive/Qwen2.5-Omni-7B
  exists: True
Original F1 cache for prompt-hash audit: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_payload_ablation/f1/predictions_cache.json
  exists: True
Saved best fine-tuned adapter: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning/qlora_r8_lambda_0_5_seed_42/best_adapter
  

## 3. Load the frozen NORMAL reference statistics

In [ ]:
# ============================================================
# LOAD FROZEN STATISTICS AND RETAIN NORMAL ONLY
# ============================================================

def load_json_list(
    path,
):
    data = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


    assert isinstance(
        data,
        list,
    ), (
        f"Expected a JSON list in {path}"
    )


    return data


frozen_base_records_all_profiles = (
    load_json_list(
        REFERENCE_BASE_STATS_PATH
    )
)


frozen_shift_records_all_profiles = (
    load_json_list(
        REFERENCE_SHIFT_STATS_PATH
    )
)


base_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_base_records_all_profiles
]


shift_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_shift_records_all_profiles
]


assert len(
    base_profiles
) == len(
    set(
        base_profiles
    )
)


assert len(
    shift_profiles
) == len(
    set(
        shift_profiles
    )
)


assert "NORMAL" in base_profiles
assert "NORMAL" in shift_profiles


normal_base_matches = [
    record

    for record
    in frozen_base_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


normal_shift_matches = [
    record

    for record
    in frozen_shift_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


assert len(
    normal_base_matches
) == 1


assert len(
    normal_shift_matches
) == 1


frozen_normal_base_statistics = copy.deepcopy(
    normal_base_matches[0]
)


frozen_normal_shift_statistics = copy.deepcopy(
    normal_shift_matches[0]
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# This is the only frozen reference object that should later
# be passed to the binary-only prompt constructor.
FROZEN_NORMAL_REFERENCE = {
    "base_statistics": (
        frozen_normal_base_statistics
    ),

    "global_shift_statistics": (
        frozen_normal_shift_statistics
    ),
}


normal_base_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_base_statistics.items()

    if key != "reference_profile"
])


normal_shift_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_shift_statistics.items()

    if key != "reference_profile"
])


print("=" * 88)
print("FROZEN REFERENCE FILES LOADED")
print("=" * 88)

print(
    "Profiles stored in base-statistics file:",
    base_profiles,
)

print(
    "Profiles stored in global-shift file:",
    shift_profiles,
)


print("\n" + "=" * 88)
print("FROZEN NORMAL BASE TEMPORAL STATISTICS")
print("=" * 88)

display(
    normal_base_display_df
)


print("\n" + "=" * 88)
print("FROZEN NORMAL GLOBAL-SHIFT STATISTICS")
print("=" * 88)

display(
    normal_shift_display_df
)


print(
    "\nOnly NORMAL frozen references retained:",
    list(
        FROZEN_NORMAL_REFERENCE.keys()
    ),
)

FROZEN REFERENCE FILES LOADED
Profiles stored in base-statistics file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']
Profiles stored in global-shift file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']

FROZEN NORMAL BASE TEMPORAL STATISTICS


,metric,value
0,num_original_A_turns,11.800000
1,num_original_B_turns,11.080000
2,num_removed_A_backchannels,1.740000
3,num_removed_B_backchannels,1.660000
4,num_filtered_A_turns,10.060000
5,num_filtered_B_turns,9.420000
6,num_offsets,5.760000
7,num_negative,2.040000
8,num_positive,3.720000
9,offset_mean,0.302128



FROZEN NORMAL GLOBAL-SHIFT STATISTICS


,metric,value
0,best_B_correction_shift_seconds__mean,-0.230000
1,best_B_correction_shift_seconds__median,-0.000000
2,estimated_B_lateness_seconds__mean,0.500000
3,estimated_B_lateness_seconds__median,0.000000
4,alignment_score_gain_vs_zero__mean,0.023376
5,alignment_score_gain_vs_zero__median,0.009105
6,best_num_bilateral_events__mean,11.940000
7,best_event_coverage__mean,0.595175



Only NORMAL frozen references retained: ['base_statistics', 'global_shift_statistics']


## 4. Load and strictly audit the new 68-case final-unseen database

In [ ]:

# ============================================================
# LOAD AND STRICTLY AUDIT THE NEW 68-CASE FINAL-UNSEEN DATABASE
# ============================================================

def load_json_list(path):
    data = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )

    assert isinstance(data, list), (
        f"Expected a JSON list in {path}"
    )

    return data


consolidation_cases = load_json_list(
    FINAL_DATABASE_PATH
)

assert len(consolidation_cases) == 68


def get_case_family(case):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()

    if variant == "normal":
        return "normal"

    if variant == "wrong_partner":
        return "wrong_partner"

    if variant == "silent_partner":
        return "silent_partner"

    if variant.startswith("lag"):
        return "lag"

    raise ValueError(
        "Unknown case_variant: "
        f"{case.get('case_variant')}"
    )


case_ids = [
    str(case["case_id"])
    for case in consolidation_cases
]

assert len(case_ids) == len(set(case_ids))

family_counts = Counter(
    get_case_family(case)
    for case in consolidation_cases
)

expected_family_counts = {
    "normal": 17,
    "wrong_partner": 17,
    "lag": 17,
    "silent_partner": 17,
}

assert dict(family_counts) == expected_family_counts

binary_label_counts = Counter(
    str(case["gold_binary_label"])
    for case in consolidation_cases
)

assert binary_label_counts["NORMAL"] == 17
assert binary_label_counts["ANOMALOUS"] == 51

source_group_counts = Counter(
    str(case["source_group_id"])
    for case in consolidation_cases
)

assert len(source_group_counts) == 17
assert set(source_group_counts.values()) == {4}

SEGMENT_NAMES = [
    "segment_0_0_to_60_seconds",
    "segment_1_60_to_120_seconds",
]

num_participant_records_checked = 0
num_semantic_slots_checked = 0

for case in consolidation_cases:
    for role, turns_key in [
        (
            "participant_A",
            "participant_A_filtered_turns",
        ),
        (
            "participant_B",
            "participant_B_filtered_turns",
        ),
    ]:
        participant = case[role]
        turns = case[turns_key]

        assert isinstance(participant, dict)
        assert isinstance(turns, list)

        assert isinstance(
            participant.get("speaks"),
            bool,
        )

        assert (
            participant["speaks"]
            == (len(turns) > 0)
        ), (
            "Participant-level speaks disagrees with final "
            f'VAD turns in {case["case_id"]} / {role}'
        )

        participant_semantics = (
            case["semantic_summaries"][role]
        )

        for segment_name in SEGMENT_NAMES:
            segment_record = (
                participant_semantics[
                    segment_name
                ]
            )

            coarse_summary = (
                segment_record.get(
                    "coarse_summary"
                )
            )

            focused_summary = (
                segment_record.get(
                    "focused_summary"
                )
            )

            assert isinstance(
                coarse_summary,
                dict,
            )

            assert isinstance(
                focused_summary,
                dict,
            )

            assert (
                "speaks"
                not in focused_summary
            ), (
                "Legacy focused-summary speaks field found in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )

            num_semantic_slots_checked += 1

        num_participant_records_checked += 1


print("=" * 100)
print("FINAL-UNSEEN DATABASE AUDIT PASSED")
print("=" * 100)

print("Database:", FINAL_DATABASE_PATH)
print("Total unique cases:", len(consolidation_cases))
print("Source groups:", len(source_group_counts))
print("Participant records checked:", num_participant_records_checked)
print("Semantic participant-segment slots checked:", num_semantic_slots_checked)

display(
    pd.DataFrame([
        {
            "case_family": family,
            "num_cases": count,
        }
        for family, count
        in sorted(family_counts.items())
    ])
)

display(
    pd.DataFrame([
        {
            "gold_binary_label": label,
            "num_cases": count,
        }
        for label, count
        in sorted(binary_label_counts.items())
    ])
)


FINAL-UNSEEN DATABASE AUDIT PASSED
Database: /content/drive/MyDrive/final_test_completely_unseen_database/final_unseen_all_68_cases_with_temporal_and_semantic_summaries.json
Total unique cases: 68
Source groups: 17
Participant records checked: 136
Semantic participant-segment slots checked: 272


,case_family,num_cases
0,lag,17
1,normal,17
2,silent_partner,17
3,wrong_partner,17


,gold_binary_label,num_cases
0,ANOMALOUS,51
1,NORMAL,17


## 5. Build the exact frozen NORMAL reference text

In [ ]:
# ============================================================
# BUILD AND PRINT THE SELECTED NORMAL-ONLY REFERENCE TEXT
#
# Only the frozen NORMAL profile is used.
#
# Included base metrics:
#   - num_offsets
#   - offset_mean
#   - offset_median
#   - offset_max
#   - offset_p75
#   - offset_p90
#   - percent_above_1_5
#   - clean_overlap_seconds
#
# Included global metrics:
#   - best_B_correction_shift_seconds__mean
#   - best_B_correction_shift_seconds__median
#   - estimated_B_lateness_seconds__mean
#   - estimated_B_lateness_seconds__median
#   - alignment_score_gain_vs_zero__mean
#   - alignment_score_gain_vs_zero__median
#   - best_num_bilateral_events__mean
#   - best_event_coverage__mean
#
# No other frozen statistics are exposed.
# ============================================================

import json
import math
import numbers


# ============================================================
# STRICT NORMAL PROFILE AUDIT
# ============================================================

assert isinstance(
    frozen_normal_base_statistics,
    dict,
)


assert isinstance(
    frozen_normal_shift_statistics,
    dict,
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# ============================================================
# NUMERIC VALIDATION HELPER
# ============================================================

def require_finite_numeric(
    record,
    field_name,
):
    """
    Read one required numeric field and verify
    that it exists and contains a finite number.
    """

    assert field_name in record, (
        f"Missing required field: {field_name}"
    )


    value = record[
        field_name
    ]


    assert isinstance(
        value,
        numbers.Real,
    ) and not isinstance(
        value,
        bool,
    ), (
        f"Expected numeric value for {field_name}, "
        f"found {type(value).__name__}: {value}"
    )


    value = float(
        value
    )


    assert math.isfinite(
        value
    ), (
        f"Non-finite value for {field_name}: {value}"
    )


    return value


def normalize_negative_zero(
    value,
):
    """
    Convert -0.0 to 0.0 for cleaner prompt text.
    """

    if abs(
        value
    ) < 1e-12:

        return 0.0


    return value


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL BASE STATISTICS
# ============================================================

normal_num_offsets = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "num_offsets",
    )
)


normal_offset_mean = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_mean",
    )
)


normal_offset_median = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_median",
    )
)


normal_offset_max = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_max",
    )
)


normal_offset_p75 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p75",
    )
)


normal_offset_p90 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p90",
    )
)


normal_percent_above_1_5 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "percent_above_1_5",
    )
)


normal_clean_overlap_seconds = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "clean_overlap_seconds",
    )
)


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL GLOBAL-SHIFT STATISTICS
#
# IMPORTANT:
# These fields use a double underscore before
# "mean" and "median".
# ============================================================

normal_best_shift_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__mean",
    )
)


normal_best_shift_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__median",
    )
)


normal_lateness_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__mean",
    )
)


normal_lateness_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__median",
    )
)


normal_alignment_gain_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__mean",
    )
)


normal_alignment_gain_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__median",
    )
)


normal_bilateral_events_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_num_bilateral_events__mean",
    )
)


normal_event_coverage_fraction = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_event_coverage__mean",
    )
)


# ============================================================
# CLEAN DISPLAY VALUES
# ============================================================

normal_best_shift_mean = (
    normalize_negative_zero(
        normal_best_shift_mean
    )
)


normal_best_shift_median = (
    normalize_negative_zero(
        normal_best_shift_median
    )
)


normal_lateness_mean = (
    normalize_negative_zero(
        normal_lateness_mean
    )
)


normal_lateness_median = (
    normalize_negative_zero(
        normal_lateness_median
    )
)


normal_event_coverage_percent = (
    100.0
    * normal_event_coverage_fraction
)


# ============================================================
# STORE THE EXACT VALUES USED IN THE PROMPT
# ============================================================

normal_base_reference_values = {
    "num_signed_offsets": (
        normal_num_offsets
    ),

    "mean_signed_offset_seconds": (
        normal_offset_mean
    ),

    "median_signed_offset_seconds": (
        normal_offset_median
    ),

    "maximum_signed_offset_seconds": (
        normal_offset_max
    ),

    "p75_signed_offset_seconds": (
        normal_offset_p75
    ),

    "p90_signed_offset_seconds": (
        normal_offset_p90
    ),

    "percent_offsets_above_1_5_seconds": (
        normal_percent_above_1_5
    ),

    "filtered_clean_overlap_seconds": (
        normal_clean_overlap_seconds
    ),
}


normal_global_reference_values = {
    "mean_best_B_correction_shift_seconds": (
        normal_best_shift_mean
    ),

    "median_best_B_correction_shift_seconds": (
        normal_best_shift_median
    ),

    "mean_estimated_B_lateness_seconds": (
        normal_lateness_mean
    ),

    "median_estimated_B_lateness_seconds": (
        normal_lateness_median
    ),

    "mean_alignment_score_gain_vs_zero": (
        normal_alignment_gain_mean
    ),

    "median_alignment_score_gain_vs_zero": (
        normal_alignment_gain_median
    ),

    "mean_num_bilateral_alignment_events": (
        normal_bilateral_events_mean
    ),

    "mean_bilateral_event_coverage_percent": (
        normal_event_coverage_percent
    ),
}


assert len(
    normal_base_reference_values
) == 8


assert len(
    normal_global_reference_values
) == 8


# ============================================================
# BUILD NORMAL LOCAL-TEMPORAL REFERENCE TEXT
# ============================================================

normal_base_reference_text = f"""
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around {normal_num_offsets:.2f}.
- Mean signed offset is around {normal_offset_mean:.2f} seconds.
- Median signed offset is around {normal_offset_median:.2f} seconds.
- Maximum signed offset is around {normal_offset_max:.2f} seconds.
- P75 signed offset is around {normal_offset_p75:.2f} seconds.
- P90 signed offset is around {normal_offset_p90:.2f} seconds.
- Approximately {normal_percent_above_1_5:.1f}% of offsets are above 1.5 seconds.
- Filtered clean overlap is around {normal_clean_overlap_seconds:.2f} seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.
""".strip()


# ============================================================
# BUILD NORMAL GLOBAL-ALIGNMENT REFERENCE TEXT
# ============================================================

normal_global_reference_text = f"""
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around {normal_best_shift_mean:.2f} seconds.
- Median best B correction shift is around {normal_best_shift_median:.2f} seconds.
- Mean estimated B lateness is around {normal_lateness_mean:.2f} seconds.
- Median estimated B lateness is around {normal_lateness_median:.2f} seconds.
- Mean alignment score gain versus zero shift is around {normal_alignment_gain_mean:.3f}.
- Median alignment score gain versus zero shift is around {normal_alignment_gain_median:.3f}.
- Mean number of bilateral alignment events is around {normal_bilateral_events_mean:.2f}.
- Mean bilateral event coverage is around {normal_event_coverage_percent:.2f}%.

These values were calculated only from the frozen NORMAL reference conversations.
They describe typical global-alignment behavior in the frozen NORMAL set and are not hard classification thresholds.
""".strip()


# ============================================================
# PRINT BOTH PROMPT SECTIONS
# ============================================================

print("=" * 88)
print("normal_base_reference_text")
print("=" * 88)

print(
    normal_base_reference_text
)


print("\n" + "=" * 88)
print("normal_global_reference_text")
print("=" * 88)

print(
    normal_global_reference_text
)


# ============================================================
# PRINT THE EXACT STRUCTURED VALUES USED
# ============================================================

print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL BASE VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_base_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL GLOBAL VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_global_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


# ============================================================
# STRICT CONTENT AUDIT
# ============================================================

combined_reference_text = (
    normal_base_reference_text
    + "\n"
    + normal_global_reference_text
)


# No anomaly-specific labels or profiles.
for forbidden_text in [
    "LAG +1",
    "LAG +2",
    "LAG +3",
    "wrong_partner",
    "silent_partner",
    "anomaly_type",
]:

    assert (
        forbidden_text.lower()
        not in combined_reference_text.lower()
    )


# Unwanted frozen reference fields must not appear.
for excluded_text in [
    "original A turns",
    "original B turns",
    "removed A backchannels",
    "removed B backchannels",
    "filtered A turns",
    "filtered B turns",
    "number of negative",
    "number of positive",
    "minimum signed offset",
    "clean overlap percentage",
]:

    assert (
        excluded_text.lower()
        not in combined_reference_text.lower()
    )


# All required selected metrics must appear.
for required_text in [
    "Number of signed offsets",
    "Mean signed offset",
    "Median signed offset",
    "Maximum signed offset",
    "P75 signed offset",
    "P90 signed offset",
    "offsets are above 1.5 seconds",
    "Filtered clean overlap",
    "Mean best B correction shift",
    "Median best B correction shift",
    "Mean estimated B lateness",
    "Median estimated B lateness",
    "Mean alignment score gain",
    "Median alignment score gain",
    "Mean number of bilateral alignment events",
    "Mean bilateral event coverage",
]:

    assert (
        required_text
        in combined_reference_text
    )


print("\n" + "=" * 88)
print("SELECTED NORMAL-ONLY REFERENCE TEXT READY")
print("=" * 88)

print(
    "Base reference metrics included:",
    len(
        normal_base_reference_values
    ),
)

print(
    "Global reference metrics included:",
    len(
        normal_global_reference_values
    ),
)

print(
    "No additional frozen statistics were exposed."
)

print(
    "No explicit anomaly profile or anomaly type was included."
)

normal_base_reference_text
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around 5.76.
- Mean signed offset is around 0.30 seconds.
- Median signed offset is around 0.26 seconds.
- Maximum signed offset is around 1.05 seconds.
- P75 signed offset is around 0.59 seconds.
- P90 signed offset is around 0.83 seconds.
- Approximately 6.7% of offsets are above 1.5 seconds.
- Filtered clean overlap is around 6.25 seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.

normal_global_reference_text
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around -0.23 seconds.
- Median best B correction shift is around 0.00 seconds.
- Mean estimated B lateness is around 0.50 seconds.
- Median estimated B lateness is around 0.00 seconds.
- Mean alignment score gain versus zero shift is around 0.023.

## 6. Load the exact Structured R1 helpers

In [ ]:

# ============================================================
# SHARED STRUCTURED-REASONING EXPERIMENT HELPERS
#
# The complete original Structured R1 projection is retained here
# as an internal source projection. The manual semantic-ablation helper defined later creates a deep-copied
# model-facing view that removes filtered turns, overlap and semantic summaries,
# retains participant-level `speaks`, and preserves local offsets and global features.
#
# The three source prompts are loaded from their exact saved
# prompt_template.txt files.
#
# The only prompt change is replacement of the original binary
# OUTPUT block with one common structured-reasoning JSON schema.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import copy
import hashlib
import json
import re
import time

import pandas as pd
import torch

from tqdm.auto import tqdm


import difflib

MAX_NEW_TOKENS_REASONING = 256
MAX_NEW_TOKENS = MAX_NEW_TOKENS_REASONING

LABELS = [
    "NORMAL",
    "ANOMALOUS",
]


# ============================================================
# EXACT SEMANTIC FIELDS PASSED TO THE MODEL
# ============================================================

COARSE_FIELDS = [
    "speech_content_summary",
    "apparent_topic",
]


FOCUSED_FIELDS = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


SEGMENT_MAP = {
    "segment_0_0_to_60_seconds": (
        "segment_0"
    ),

    "segment_1_60_to_120_seconds": (
        "segment_1"
    ),
}


# ============================================================
# EXACT TEMPORAL FIELDS PASSED TO THE MODEL
# ============================================================

LOCAL_FEATURE_FIELDS = [
    "clean_overlap_seconds",
    "clean_overlap_percent",

    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",

    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",

    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]


GLOBAL_FEATURE_FIELDS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage",
]


# ============================================================
# FIELDS THAT MUST NEVER APPEAR IN THE MODEL PROMPT
# ============================================================

FORBIDDEN_PROMPT_KEYS = [
    "case_id",
    "source_group_id",
    "pair_index",

    "gold_binary_label",
    "gold_anomaly_type",

    "case_variant",
    "pairing_type",

    "conversation_id",
    "participant_id",
    "metadata_path",

    "num_raw_vad_entries",

    "A_source_conversation",
    "B_source_conversation",

    "semantic_summary_source",
    "semantic_summary_coverage",
]


# ============================================================
# GENERAL HELPERS
# ============================================================

def canonical_json(
    value,
):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def sha256_text(
    text,
):
    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


def select_exact_fields(
    record,
    fields,
    context,
):
    assert isinstance(
        record,
        dict,
    ), (
        f"Expected dictionary for {context}"
    )


    missing_fields = [
        field

        for field in fields

        if field not in record
    ]


    assert not missing_fields, (
        f"Missing fields in {context}: "
        f"{missing_fields}"
    )


    return {
        field: copy.deepcopy(
            record[
                field
            ]
        )

        for field in fields
    }


# ============================================================
# TURN PROJECTION
#
# Database format:
#   {"start": 1.2, "end": 3.4}
#
# Prompt format:
#   [1.2, 3.4]
# ============================================================

def compact_turns(
    turns,
    context,
):
    assert isinstance(
        turns,
        list,
    ), (
        f"Expected list for {context}"
    )


    compact = []


    for index, turn in enumerate(
        turns
    ):

        assert isinstance(
            turn,
            dict,
        ), (
            f"Invalid turn in {context} "
            f"at index {index}"
        )


        assert "start" in turn
        assert "end" in turn


        start = float(
            turn[
                "start"
            ]
        )


        end = float(
            turn[
                "end"
            ]
        )


        assert end >= start


        compact.append([
            start,
            end,
        ])


    return compact


# ============================================================
# SEMANTIC INPUT PROJECTION
#
# Participant IDs and other metadata are deliberately excluded.
# ============================================================

def build_semantic_input(
    case,
):
    semantic_input = {}


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        role_output = {}


        for (
            database_segment_name,
            prompt_segment_name,
        ) in SEGMENT_MAP.items():

            segment_record = (
                participant_semantics[
                    database_segment_name
                ]
            )


            coarse_summary = (
                select_exact_fields(
                    segment_record[
                        "coarse_summary"
                    ],

                    COARSE_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "coarse_summary"
                    ),
                )
            )


            focused_summary = (
                select_exact_fields(
                    segment_record[
                        "focused_summary"
                    ],

                    FOCUSED_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "focused_summary"
                    ),
                )
            )


            # Legacy focused-summary speaks must not exist.
            assert (
                "speaks"
                not in focused_summary
            )


            role_output[
                prompt_segment_name
            ] = {
                "coarse_summary": (
                    coarse_summary
                ),

                "focused_summary": (
                    focused_summary
                ),
            }


        semantic_input[
            role
        ] = role_output


    return semantic_input


# ============================================================
# BUILD THE EXACT MODEL INPUT
# ============================================================

def build_binary_model_input(
    case,
):
    local_features = (
        select_exact_fields(
            case[
                "local_temporal_features"
            ],

            LOCAL_FEATURE_FIELDS,

            "local_temporal_features",
        )
    )


    global_features_raw = (
        select_exact_fields(
            case[
                "global_shift_features"
            ],

            GLOBAL_FEATURE_FIELDS,

            "global_shift_features",
        )
    )


    # Stored in the database as a fraction.
    # Presented in the prompt as a percentage so that it is
    # directly comparable with the frozen NORMAL percentage.
    event_coverage_fraction = float(
        global_features_raw.pop(
            "best_event_coverage"
        )
    )


    global_features = {
        **global_features_raw,

        "best_event_coverage_percent": round(
            100.0
            * event_coverage_fraction,

            6,
        ),
    }


    payload = {
        "analysis_duration_seconds": float(
            case[
                "analysis_duration_seconds"
            ]
        ),

        "participant_A": {
            "speaks": bool(
                case[
                    "participant_A"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_A_filtered_turns"
                ],

                "participant_A_filtered_turns",
            ),
        },

        "participant_B": {
            "speaks": bool(
                case[
                    "participant_B"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_B_filtered_turns"
                ],

                "participant_B_filtered_turns",
            ),
        },

        "local_temporal_features": (
            local_features
        ),

        "global_shift_features": (
            global_features
        ),

        "semantic_summaries": (
            build_semantic_input(
                case
            )
        ),
    }


    assert set(
        payload
    ) == {
        "analysis_duration_seconds",
        "participant_A",
        "participant_B",
        "local_temporal_features",
        "global_shift_features",
        "semantic_summaries",
    }


    return payload


# ============================================================
# BUILD THE FINAL PROMPT FOR ONE CASE
# ============================================================

def build_binary_prompt_from_template(
    case,
    prompt_template,
):
    payload = build_binary_model_input(
        case
    )


    prompt = (
        prompt_template
        .format(
            normal_base_reference_text=(
                normal_base_reference_text
            ),

            normal_global_reference_text=(
                normal_global_reference_text
            ),

            duration_seconds=(
                payload[
                    "analysis_duration_seconds"
                ]
            ),

            participant_A_speaks=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_A_turns=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            participant_B_speaks=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_B_turns=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            local_temporal_features=(
                json.dumps(
                    payload[
                        "local_temporal_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            global_shift_features=(
                json.dumps(
                    payload[
                        "global_shift_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            semantic_summaries=(
                json.dumps(
                    payload[
                        "semantic_summaries"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),
        )
    )


    # Strict leakage audit.
    prompt_lower = prompt.lower()


    for forbidden_key in (
        FORBIDDEN_PROMPT_KEYS
    ):

        assert (
            forbidden_key.lower()
            not in prompt_lower
        ), (
            "Forbidden field name leaked into prompt: "
            f"{forbidden_key}"
        )


    return (
        prompt,
        payload,
    )


# ============================================================
# TEXT-ONLY QWEN INFERENCE
#
# The existing qwen_text_only helper used 700 output tokens.
# Here only 64 are needed because the expected output is:
#
#   {"label": "NORMAL"}
#
# or:
#
#   {"label": "ANOMALOUS"}
# ============================================================

def qwen_text_only_binary(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS,
):
    messages = [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ]


    inputs = (
        processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",

            processor_kwargs={
                "padding": True,
            },
        )
    )


    inputs = {
        key: (
            value.to(
                model.device
            )
            if hasattr(
                value,
                "to",
            )
            else value
        )

        for key, value
        in inputs.items()
    }


    input_token_count = int(
        inputs[
            "input_ids"
        ].shape[-1]
    )


    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,

            max_new_tokens=(
                max_new_tokens
            ),

            do_sample=False,

            use_cache=True,
        )


    generated_ids = output_ids[
        :,
        inputs[
            "input_ids"
        ].shape[1]:,
    ]


    raw_output = (
        processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
    )


    return (
        raw_output,
        input_token_count,
    )


# ============================================================
# ROBUST JSON PARSING
# ============================================================

def extract_first_json_object(
    text,
):
    cleaned = text.strip()


    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )


    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned,
    )


    try:

        return json.loads(
            cleaned
        )

    except Exception:

        pass


    decoder = json.JSONDecoder()


    possible_starts = [
        index

        for index, character
        in enumerate(
            cleaned
        )

        if character == "{"
    ]


    for start in possible_starts:

        try:

            parsed, _ = (
                decoder.raw_decode(
                    cleaned[
                        start:
                    ]
                )
            )


            return parsed

        except Exception:

            continue


    return None

# ============================================================
# STRUCTURED REASONING SCHEMA
# ============================================================

REASONING_SCHEMA_KEYS = [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
    "label",
]


REASONING_ALLOWED_VALUES = {
    "participation_assessment": {
        "VALID",
        "INVALID",
    },

    "local_temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "global_temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "semantic_assessment": {
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    },

    "decisive_dimension": {
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    },

    "label": {
        "NORMAL",
        "ANOMALOUS",
    },
}


OLD_BINARY_OUTPUT_BLOCK = """
============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include reasoning.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


STRUCTURED_REASONING_OUTPUT_BLOCK = """
============================================================
STRUCTURED REASONING OUTPUT
============================================================

Preserve all evidence definitions, comparison rules, decision criteria,
and the final binary decision policy stated above.

Before selecting the final label, expose the following fixed structured
assessments.

These fields are diagnostic outputs only.

They must not introduce new evidence, new thresholds, a new voting rule,
or any change to the existing decision policy.

Use the available evidence exactly as instructed above.

Assessment meanings:

- participation_assessment:
  - VALID
  - INVALID

- local_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- global_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- semantic_assessment:
  - COMPATIBLE
  - INCOMPATIBLE
  - LIMITED

- decisive_dimension:
  - PARTICIPATION
  - TEMPORAL
  - SEMANTIC
  - NONE

The local and global temporal assessments must reflect the reliability
rules already defined in the prompt.

The combined temporal assessment must be based on the existing temporal
decision policy.

The final label must follow the existing final decision policy exactly.

Do not infer or output an anomaly subtype.
Do not infer or output a delay magnitude.
Do not provide free-form reasoning.

============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "participation_assessment": "VALID or INVALID",
  "local_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "global_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "semantic_assessment": "COMPATIBLE or INCOMPATIBLE or LIMITED",
  "decisive_dimension": "PARTICIPATION or TEMPORAL or SEMANTIC or NONE",
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


def build_structured_reasoning_prompt_template(
    source_prompt_template,
):
    assert isinstance(
        source_prompt_template,
        str,
    )


    assert source_prompt_template.endswith(
        OLD_BINARY_OUTPUT_BLOCK
    ), (
        "The saved source prompt does not end with the exact "
        "original binary OUTPUT block."
    )


    reasoning_prompt_template = (
        source_prompt_template[
            :-len(
                OLD_BINARY_OUTPUT_BLOCK
            )
        ]
        + STRUCTURED_REASONING_OUTPUT_BLOCK
    )


    reconstructed_source_prompt = (
        reasoning_prompt_template[
            :-len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            )
        ]
        + OLD_BINARY_OUTPUT_BLOCK
    )


    assert (
        reconstructed_source_prompt
        == source_prompt_template
    ), (
        "Unexpected source-prompt modification."
    )


    return reasoning_prompt_template


def parse_structured_reasoning_prediction(
    raw_output,
):
    parsed = extract_first_json_object(
        raw_output
    )


    normalized = {
        key: None
        for key in REASONING_SCHEMA_KEYS
    }


    schema_errors = []


    if isinstance(
        parsed,
        dict,
    ):

        for key in REASONING_SCHEMA_KEYS:

            if key in parsed:

                normalized[
                    key
                ] = str(
                    parsed[
                        key
                    ]
                ).strip().upper()


        for key in REASONING_SCHEMA_KEYS:

            if normalized[
                key
            ] not in REASONING_ALLOWED_VALUES[
                key
            ]:

                schema_errors.append(
                    (
                        f"{key}: "
                        f"{normalized[key]}"
                    )
                )


        exact_keys = (
            set(
                parsed.keys()
            )
            == set(
                REASONING_SCHEMA_KEYS
            )
        )


        schema_exact = (
            exact_keys
            and
            not schema_errors
        )


        label = normalized[
            "label"
        ]


        if label in LABELS:

            return {
                "prediction": label,

                "parse_mode": (
                    "structured_json"
                ),

                "schema_exact": bool(
                    schema_exact
                ),

                "parsed_output": parsed,

                "schema_errors": (
                    schema_errors
                ),

                **{
                    key: normalized[
                        key
                    ]

                    for key in (
                        REASONING_SCHEMA_KEYS
                    )

                    if key != "label"
                },
            }


    plain_output = (
        raw_output
        .strip()
        .strip('"')
        .strip("'")
        .upper()
    )


    if plain_output in LABELS:

        return {
            "prediction": plain_output,

            "parse_mode": (
                "exact_plaintext_fallback"
            ),

            "schema_exact": False,

            "parsed_output": None,

            "schema_errors": [
                "Structured reasoning fields missing."
            ],

            **{
                key: None

                for key in (
                    REASONING_SCHEMA_KEYS
                )

                if key != "label"
            },
        }


    return {
        "prediction": None,

        "parse_mode": "invalid",

        "schema_exact": False,

        "parsed_output": parsed,

        "schema_errors": (
            schema_errors
            if schema_errors
            else [
                "No valid structured JSON prediction."
            ]
        ),

        **{
            key: normalized[
                key
            ]

            for key in (
                REASONING_SCHEMA_KEYS
            )

            if key != "label"
        },
    }


# ============================================================
# FULL-SEMANTICS PAYLOAD AUDIT
# ============================================================

def collect_nested_keys_reasoning(
    value,
):
    keys = set()


    if isinstance(
        value,
        dict,
    ):

        for key, nested_value in value.items():

            keys.add(
                str(key)
            )


            keys.update(
                collect_nested_keys_reasoning(
                    nested_value
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for item in value:

            keys.update(
                collect_nested_keys_reasoning(
                    item
                )
            )


    return keys


FULL_SEMANTIC_REQUIRED_KEYS = [
    "semantic_summaries",
    "coarse_summary",
    "focused_summary",

    "speech_content_summary",
    "apparent_topic",

    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


# ============================================================
# EXACT PROMPT CONFIGURATION
# ============================================================

def prepare_reasoning_experiment(
    *,
    experiment_version,
    source_experiment_name,
    experiment_title,
    required_source_markers,
    forbidden_source_markers,
    temporal_profiles_used,
    assessment_policy,
):
    source_experiment_dir = (
        OUT_DIR
        / source_experiment_name
    )


    source_prompt_path = (
        source_experiment_dir
        / "prompt_template.txt"
    )


    assert source_prompt_path.exists(), (
        "Saved source prompt not found:\n"
        f"{source_prompt_path}"
    )


    source_prompt_template = (
        source_prompt_path.read_text(
            encoding="utf-8"
        )
    )


    for marker in required_source_markers:

        assert marker in source_prompt_template, (
            "Required source-prompt marker missing: "
            f"{marker}"
        )


    for marker in forbidden_source_markers:

        assert marker not in source_prompt_template, (
            "Forbidden source-prompt marker found: "
            f"{marker}"
        )


    for semantic_marker in [
        "Coarse semantic information:",
        "Focused semantic information:",
        "- detailed_speech_summary",
        "- main_topic",
        "- secondary_topics",
        "- key_semantic_details",
    ]:

        assert semantic_marker in source_prompt_template, (
            "The source prompt is not a full-semantics prompt. "
            f"Missing: {semantic_marker}"
        )


    reasoning_prompt_template = (
        build_structured_reasoning_prompt_template(
            source_prompt_template
        )
    )


    experiment_dir = (
        OUT_DIR
        / experiment_version
    )


    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    paths = {
        "prediction_cache": (
            experiment_dir
            / "predictions_cache.json"
        ),

        "predictions_csv": (
            experiment_dir
            / "predictions_all_68.csv"
        ),

        "metrics_json": (
            experiment_dir
            / "metrics.json"
        ),

        "confusion_matrix_csv": (
            experiment_dir
            / "confusion_matrix.csv"
        ),

        "family_metrics_csv": (
            experiment_dir
            / "metrics_by_case_family.csv"
        ),

        "variant_metrics_csv": (
            experiment_dir
            / "metrics_by_case_variant.csv"
        ),

        "errors_csv": (
            experiment_dir
            / "classification_errors.csv"
        ),

        "reasoning_assessments_csv": (
            experiment_dir
            / "reasoning_assessments.csv"
        ),

        "reasoning_inconsistencies_csv": (
            experiment_dir
            / "reasoning_inconsistencies.csv"
        ),

        "assessment_distributions_csv": (
            experiment_dir
            / "assessment_distributions.csv"
        ),

        "prompt_template": (
            experiment_dir
            / "prompt_template.txt"
        ),

        "source_prompt_copy": (
            experiment_dir
            / "source_prompt_template.txt"
        ),

        "prompt_diff": (
            experiment_dir
            / "prompt_diff_vs_source.txt"
        ),

        "experiment_manifest": (
            experiment_dir
            / "experiment_manifest.json"
        ),
    }


    source_prompt_sha256 = sha256_text(
        source_prompt_template
    )


    reasoning_prompt_sha256 = sha256_text(
        reasoning_prompt_template
    )


    reasoning_schema_sha256 = sha256_text(
        STRUCTURED_REASONING_OUTPUT_BLOCK
    )


    normal_reference_sha256 = sha256_text(
        normal_base_reference_text
        + "\n"
        + normal_global_reference_text
    )


    paths[
        "prompt_template"
    ].write_text(
        reasoning_prompt_template,
        encoding="utf-8",
    )


    paths[
        "source_prompt_copy"
    ].write_text(
        source_prompt_template,
        encoding="utf-8",
    )


    prompt_diff_lines = difflib.unified_diff(
        source_prompt_template.splitlines(),
        reasoning_prompt_template.splitlines(),

        fromfile=(
            f"{source_experiment_name}/prompt_template.txt"
        ),

        tofile=(
            f"{experiment_version}/prompt_template.txt"
        ),

        lineterm="",
    )


    paths[
        "prompt_diff"
    ].write_text(
        "\n".join(
            prompt_diff_lines
        ),
        encoding="utf-8",
    )


    example_prompt, example_payload = (
        build_binary_prompt_from_template(
            consolidation_cases[0],
            reasoning_prompt_template,
        )
    )


    payload_keys = collect_nested_keys_reasoning(
        example_payload
    )


    for required_key in FULL_SEMANTIC_REQUIRED_KEYS:

        assert required_key in payload_keys, (
            "Full-semantic input field missing: "
            f"{required_key}"
        )


    assert (
        "STRUCTURED REASONING OUTPUT"
        in example_prompt
    )


    assert (
        "Do not include reasoning."
        not in reasoning_prompt_template[
            -len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            ):
        ]
    )


    manifest = {
        "experiment_version": (
            experiment_version
        ),

        "experiment_title": (
            experiment_title
        ),

        "source_experiment": (
            source_experiment_name
        ),

        "source_prompt_path": str(
            source_prompt_path
        ),

        "source_prompt_sha256": (
            source_prompt_sha256
        ),

        "reasoning_prompt_sha256": (
            reasoning_prompt_sha256
        ),

        "reasoning_schema_sha256": (
            reasoning_schema_sha256
        ),

        "normal_reference_sha256": (
            normal_reference_sha256
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            temporal_profiles_used
        ),

        "assessment_policy": (
            assessment_policy
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "model_id": (
            MODEL_ID
        ),

        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },

        "only_prompt_change": (
            "The exact original binary OUTPUT block was "
            "replaced by the common structured-reasoning "
            "OUTPUT block."
        ),

        "all_model_facing_evidence_unchanged": (
            True
        ),

        "all_pre_output_prompt_text_unchanged": (
            True
        ),
    }


    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    config = {
        **manifest,

        "experiment_dir": (
            experiment_dir
        ),

        "source_prompt_template": (
            source_prompt_template
        ),

        "reasoning_prompt_template": (
            reasoning_prompt_template
        ),

        "paths": paths,
    }


    print("=" * 88)
    print(
        f"{experiment_title} — CONFIGURATION READY"
    )
    print("=" * 88)

    print(
        "Experiment version:",
        experiment_version,
    )

    print(
        "Source experiment:",
        source_experiment_name,
    )

    print(
        "Source prompt SHA256:",
        source_prompt_sha256,
    )

    print(
        "Reasoning prompt SHA256:",
        reasoning_prompt_sha256,
    )

    print(
        "Semantic input:",
        "coarse_and_focused",
    )

    print(
        "Focused summaries used:",
        True,
    )

    print(
        "Temporal profiles:",
        temporal_profiles_used,
    )

    print(
        "Assessment policy:",
        assessment_policy,
    )

    print(
        "Only source-prompt change:",
        (
            "binary OUTPUT block -> "
            "structured reasoning OUTPUT block"
        ),
    )

    print(
        "Example prompt characters:",
        len(
            example_prompt
        ),
    )

    print(
        "Prediction cache:",
        paths[
            "prediction_cache"
        ],
    )

    print(
        "Existing cache:",
        paths[
            "prediction_cache"
        ].exists(),
    )


    return config


# ============================================================
# CHECKPOINTED INFERENCE
# ============================================================

def reasoning_utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def reasoning_atomic_write_json(
    path,
    data,
):
    path = Path(
        path
    )


    temporary_path = path.with_suffix(
        path.suffix
        + ".tmp"
    )


    temporary_path.write_text(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    temporary_path.replace(
        path
    )


def create_reasoning_cache(
    config,
):
    return {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "experiment_title": (
            config[
                "experiment_title"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "model_id": (
            MODEL_ID
        ),

        "source_prompt_sha256": (
            config[
                "source_prompt_sha256"
            ]
        ),

        "reasoning_prompt_sha256": (
            config[
                "reasoning_prompt_sha256"
            ]
        ),

        "reasoning_schema_sha256": (
            config[
                "reasoning_schema_sha256"
            ]
        ),

        "normal_reference_sha256": (
            config[
                "normal_reference_sha256"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "created_at_utc": (
            reasoning_utc_now()
        ),

        "updated_at_utc": (
            reasoning_utc_now()
        ),

        "records": {},
    }


def run_reasoning_experiment(
    config,
):
    prediction_cache_path = (
        config[
            "paths"
        ][
            "prediction_cache"
        ]
    )


    if prediction_cache_path.exists():

        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )


        for key in [
            "experiment_version",
            "source_experiment",
            "model_id",
            "source_prompt_sha256",
            "reasoning_prompt_sha256",
            "reasoning_schema_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "focused_summaries_used",
            "temporal_profiles_used",
            "assessment_policy",
            "max_new_tokens",
        ]:

            expected_value = (
                create_reasoning_cache(
                    config
                )[
                    key
                ]
            )


            assert (
                prediction_cache[
                    key
                ]
                == expected_value
            ), (
                f"Cache mismatch for {key}"
            )


        print(
            "Resuming cache:",
            prediction_cache_path,
        )

        print(
            "Existing records:",
            len(
                prediction_cache[
                    "records"
                ]
            ),
        )


    else:

        prediction_cache = (
            create_reasoning_cache(
                config
            )
        )


        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )


        print(
            "Created cache:",
            prediction_cache_path,
        )


    ordered_cases = sorted(
        consolidation_cases,

        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )


    assert len(
        ordered_cases
    ) == 68


    for case in tqdm(
        ordered_cases,
        desc=(
            config[
                "experiment_version"
            ]
        ),
    ):

        case_id = str(
            case[
                "case_id"
            ]
        )


        prompt, model_input_payload = (
            build_binary_prompt_from_template(
                case,
                config[
                    "reasoning_prompt_template"
                ],
            )
        )


        current_payload_keys = (
            collect_nested_keys_reasoning(
                model_input_payload
            )
        )


        for required_key in (
            FULL_SEMANTIC_REQUIRED_KEYS
        ):

            assert required_key in (
                current_payload_keys
            ), (
                f"Missing full-semantic field "
                f"for case {case_id}: "
                f"{required_key}"
            )


        prompt_sha256 = sha256_text(
            prompt
        )


        input_payload_sha256 = sha256_text(
            canonical_json(
                model_input_payload
            )
        )


        existing_record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )


        if (
            existing_record is not None
            and
            existing_record.get(
                "prediction"
            ) in LABELS
        ):

            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )


            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == input_payload_sha256
            )


            continue


        started = time.perf_counter()


        try:

            raw_output, input_token_count = (
                qwen_text_only_binary(
                    prompt,
                    max_new_tokens=(
                        MAX_NEW_TOKENS_REASONING
                    ),
                )
            )


            parsed_result = (
                parse_structured_reasoning_prediction(
                    raw_output
                )
            )


            generation_error = None


        except Exception as exc:

            raw_output = ""


            input_token_count = None


            parsed_result = {
                "prediction": None,
                "parse_mode": "generation_error",
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
                "participation_assessment": None,
                "local_temporal_assessment": None,
                "global_temporal_assessment": None,
                "temporal_assessment": None,
                "semantic_assessment": None,
                "decisive_dimension": None,
            }


            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )


            if torch.cuda.is_available():

                torch.cuda.empty_cache()


        elapsed_seconds = (
            time.perf_counter()
            - started
        )


        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_binary_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "prompt_sha256": (
                prompt_sha256
            ),

            "input_payload_sha256": (
                input_payload_sha256
            ),

            "semantic_input": (
                "coarse_and_focused"
            ),

            "focused_summaries_used": True,

            "input_token_count": (
                input_token_count
            ),

            "max_new_tokens": (
                MAX_NEW_TOKENS_REASONING
            ),

            "raw_output": (
                raw_output
            ),

            "prediction": (
                parsed_result[
                    "prediction"
                ]
            ),

            "participation_assessment": (
                parsed_result[
                    "participation_assessment"
                ]
            ),

            "local_temporal_assessment": (
                parsed_result[
                    "local_temporal_assessment"
                ]
            ),

            "global_temporal_assessment": (
                parsed_result[
                    "global_temporal_assessment"
                ]
            ),

            "temporal_assessment": (
                parsed_result[
                    "temporal_assessment"
                ]
            ),

            "semantic_assessment": (
                parsed_result[
                    "semantic_assessment"
                ]
            ),

            "decisive_dimension": (
                parsed_result[
                    "decisive_dimension"
                ]
            ),

            "parse_mode": (
                parsed_result[
                    "parse_mode"
                ]
            ),

            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),

            "schema_errors": (
                parsed_result[
                    "schema_errors"
                ]
            ),

            "parsed_output": (
                parsed_result[
                    "parsed_output"
                ]
            ),

            "generation_error": (
                generation_error
            ),

            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),

            "completed_at_utc": (
                reasoning_utc_now()
            ),
        }


        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()


        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )


    print("\n" + "=" * 88)
    print(
        f"{config['experiment_title']} — INFERENCE COMPLETE"
    )
    print("=" * 88)

    print(
        "Cached records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )

    print(
        "Prediction cache:",
        prediction_cache_path,
    )


    return prediction_cache


# ============================================================
# EVALUATION
# ============================================================

def summarize_reasoning_group(
    group,
):
    valid_group = group[
        group[
            "valid_prediction"
        ]
    ]


    return {
        "total_cases": int(
            len(
                group
            )
        ),

        "valid_predictions": int(
            len(
                valid_group
            )
        ),

        "invalid_predictions": int(
            len(
                group
            )
            -
            len(
                valid_group
            )
        ),

        "predicted_NORMAL": int(
            (
                valid_group[
                    "prediction"
                ]
                == "NORMAL"
            ).sum()
        ),

        "predicted_ANOMALOUS": int(
            (
                valid_group[
                    "prediction"
                ]
                == "ANOMALOUS"
            ).sum()
        ),

        "correct_predictions": int(
            group[
                "correct"
            ].sum()
        ),

        "accuracy_on_valid": (
            float(
                (
                    valid_group[
                        "gold_label"
                    ]
                    ==
                    valid_group[
                        "prediction"
                    ]
                ).mean()
            )
            if len(
                valid_group
            )
            else float(
                "nan"
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            group[
                "correct"
            ].mean()
        ),

        "exact_schema_rate": float(
            group[
                "schema_exact"
            ].mean()
        ),
    }


def evaluate_reasoning_experiment(
    config,
):
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        matthews_corrcoef,
        precision_score,
        recall_score,
    )

    from IPython.display import display


    prediction_cache = json.loads(
        config[
            "paths"
        ][
            "prediction_cache"
        ].read_text(
            encoding="utf-8"
        )
    )


    assert (
        prediction_cache[
            "experiment_version"
        ]
        == config[
            "experiment_version"
        ]
    )


    assert (
        prediction_cache[
            "reasoning_prompt_sha256"
        ]
        == config[
            "reasoning_prompt_sha256"
        ]
    )


    result_rows = []


    for case in consolidation_cases:

        case_id = str(
            case[
                "case_id"
            ]
        )


        record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )


        result_rows.append({
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "prediction": (
                None
                if record is None
                else record.get(
                    "prediction"
                )
            ),

            "participation_assessment": (
                None
                if record is None
                else record.get(
                    "participation_assessment"
                )
            ),

            "local_temporal_assessment": (
                None
                if record is None
                else record.get(
                    "local_temporal_assessment"
                )
            ),

            "global_temporal_assessment": (
                None
                if record is None
                else record.get(
                    "global_temporal_assessment"
                )
            ),

            "temporal_assessment": (
                None
                if record is None
                else record.get(
                    "temporal_assessment"
                )
            ),

            "semantic_assessment": (
                None
                if record is None
                else record.get(
                    "semantic_assessment"
                )
            ),

            "decisive_dimension": (
                None
                if record is None
                else record.get(
                    "decisive_dimension"
                )
            ),

            "parse_mode": (
                "missing"
                if record is None
                else record.get(
                    "parse_mode"
                )
            ),

            "schema_exact": (
                False
                if record is None
                else bool(
                    record.get(
                        "schema_exact",
                        False,
                    )
                )
            ),

            "schema_errors": (
                None
                if record is None
                else json.dumps(
                    record.get(
                        "schema_errors",
                        [],
                    ),
                    ensure_ascii=False,
                )
            ),

            "input_token_count": (
                None
                if record is None
                else record.get(
                    "input_token_count"
                )
            ),

            "elapsed_seconds": (
                None
                if record is None
                else record.get(
                    "elapsed_seconds"
                )
            ),

            "raw_output": (
                ""
                if record is None
                else record.get(
                    "raw_output",
                    "",
                )
            ),

            "generation_error": (
                None
                if record is None
                else record.get(
                    "generation_error"
                )
            ),
        })


    results_df = pd.DataFrame(
        result_rows
    )


    results_df[
        "valid_prediction"
    ] = results_df[
        "prediction"
    ].isin(
        LABELS
    )


    results_df[
        "correct"
    ] = (
        results_df[
            "valid_prediction"
        ]
        &
        (
            results_df[
                "gold_label"
            ]
            ==
            results_df[
                "prediction"
            ]
        )
    )


    results_df[
        "normal_with_invalid_participation"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "participation_assessment"
            ]
            == "INVALID"
        )
    )


    results_df[
        "normal_with_anomalous_temporal"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "temporal_assessment"
            ]
            == "ANOMALOUS"
        )
    )


    results_df[
        "normal_with_incompatible_semantics"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "semantic_assessment"
            ]
            == "INCOMPATIBLE"
        )
    )


    results_df[
        "reasoning_inconsistency"
    ] = (
        results_df[
            [
                "normal_with_invalid_participation",
                "normal_with_anomalous_temporal",
                "normal_with_incompatible_semantics",
            ]
        ].any(
            axis=1
        )
    )


    assert len(
        results_df
    ) == 68


    assert (
        results_df[
            "case_id"
        ].is_unique
    )


    valid_df = results_df[
        results_df[
            "valid_prediction"
        ]
    ].copy()


    invalid_df = results_df[
        ~results_df[
            "valid_prediction"
        ]
    ].copy()


    assert len(
        valid_df
    ) > 0


    y_true = valid_df[
        "gold_label"
    ]


    y_pred = valid_df[
        "prediction"
    ]


    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            "NORMAL",
            "ANOMALOUS",
        ],
    )


    confusion_df = pd.DataFrame(
        confusion,

        index=[
            "Gold NORMAL",
            "Gold ANOMALOUS",
        ],

        columns=[
            "Pred NORMAL",
            "Pred ANOMALOUS",
        ],
    )


    true_normal = int(
        confusion[
            0,
            0,
        ]
    )


    false_anomalous = int(
        confusion[
            0,
            1,
        ]
    )


    normal_recall = (
        true_normal
        /
        (
            true_normal
            + false_anomalous
        )
        if (
            true_normal
            + false_anomalous
        )
        else float(
            "nan"
        )
    )


    metrics = {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "total_cases": int(
            len(
                results_df
            )
        ),

        "valid_predictions": int(
            len(
                valid_df
            )
        ),

        "invalid_predictions": int(
            len(
                invalid_df
            )
        ),

        "exact_json_schema_rate": float(
            results_df[
                "schema_exact"
            ].mean()
        ),

        "reasoning_inconsistency_count": int(
            results_df[
                "reasoning_inconsistency"
            ].sum()
        ),

        "accuracy_valid_predictions": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            results_df[
                "correct"
            ].mean()
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "anomalous_precision": float(
            precision_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "anomalous_recall": float(
            recall_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "anomalous_f1": float(
            f1_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "normal_recall_specificity": float(
            normal_recall
        ),

        "matthews_correlation_coefficient": float(
            matthews_corrcoef(
                y_true,
                y_pred,
            )
        ),

        "confusion_matrix_label_order": [
            "NORMAL",
            "ANOMALOUS",
        ],

        "confusion_matrix": (
            confusion.tolist()
        ),
    }


    family_rows = []


    for (
        family_name,
        family_group,
    ) in results_df.groupby(
        "case_family",
        sort=True,
    ):

        family_rows.append({
            "case_family": (
                family_name
            ),

            **summarize_reasoning_group(
                family_group
            ),
        })


    family_metrics_df = pd.DataFrame(
        family_rows
    )


    variant_rows = []


    for (
        variant_name,
        variant_group,
    ) in results_df.groupby(
        "case_variant",
        sort=True,
    ):

        variant_rows.append({
            "case_variant": (
                variant_name
            ),

            **summarize_reasoning_group(
                variant_group
            ),
        })


    variant_metrics_df = pd.DataFrame(
        variant_rows
    )


    classification_report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=[
                "NORMAL",
                "ANOMALOUS",
            ],
            output_dict=True,
            zero_division=0,
        )
    ).T


    error_df = results_df[
        (
            ~results_df[
                "valid_prediction"
            ]
        )
        |
        (
            results_df[
                "gold_label"
            ] != results_df[
                "prediction"
            ]
        )
    ].copy()


    reasoning_inconsistency_df = (
        results_df[
            results_df[
                "reasoning_inconsistency"
            ]
        ].copy()
    )


    assessment_rows = []


    for assessment_field in [
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
    ]:

        counts = (
            results_df[
                assessment_field
            ]
            .fillna(
                "MISSING"
            )
            .value_counts(
                dropna=False
            )
        )


        for value, count in counts.items():

            assessment_rows.append({
                "assessment_field": (
                    assessment_field
                ),

                "assessment_value": (
                    value
                ),

                "count": int(
                    count
                ),
            })


    assessment_distributions_df = (
        pd.DataFrame(
            assessment_rows
        )
    )


    source_group_rows = []


    for (
        source_group_id,
        source_group,
    ) in results_df.groupby(
        "source_group_id"
    ):

        source_group_rows.append({
            "source_group_id": (
                source_group_id
            ),

            "num_cases": int(
                len(
                    source_group
                )
            ),

            "all_predictions_valid": bool(
                source_group[
                    "valid_prediction"
                ].all()
            ),

            "all_cases_correct": bool(
                source_group[
                    "valid_prediction"
                ].all()
                and
                source_group[
                    "correct"
                ].all()
            ),
        })


    source_group_df = pd.DataFrame(
        source_group_rows
    )


    assert len(
        source_group_df
    ) == 17


    assert set(
        source_group_df[
            "num_cases"
        ]
    ) == {
        4
    }


    metrics[
        "source_group_exact_match_rate"
    ] = float(
        source_group_df[
            "all_cases_correct"
        ].mean()
    )


    paths = config[
        "paths"
    ]


    results_df.to_csv(
        paths[
            "predictions_csv"
        ],
        index=False,
    )


    reasoning_columns = [
        "case_id",
        "source_group_id",
        "case_family",
        "case_variant",
        "gold_label",
        "prediction",
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
        "schema_exact",
        "correct",
    ]


    results_df[
        reasoning_columns
    ].to_csv(
        paths[
            "reasoning_assessments_csv"
        ],
        index=False,
    )


    confusion_df.to_csv(
        paths[
            "confusion_matrix_csv"
        ]
    )


    family_metrics_df.to_csv(
        paths[
            "family_metrics_csv"
        ],
        index=False,
    )


    variant_metrics_df.to_csv(
        paths[
            "variant_metrics_csv"
        ],
        index=False,
    )


    error_df.to_csv(
        paths[
            "errors_csv"
        ],
        index=False,
    )


    reasoning_inconsistency_df.to_csv(
        paths[
            "reasoning_inconsistencies_csv"
        ],
        index=False,
    )


    assessment_distributions_df.to_csv(
        paths[
            "assessment_distributions_csv"
        ],
        index=False,
    )


    paths[
        "metrics_json"
    ].write_text(
        json.dumps(
            metrics,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    print("=" * 88)
    print(
        f"{config['experiment_title']} — RESULTS"
    )
    print("=" * 88)

    print(
        "Total cases:",
        metrics[
            "total_cases"
        ],
    )

    print(
        "Valid predictions:",
        metrics[
            "valid_predictions"
        ],
    )

    print(
        "Invalid predictions:",
        metrics[
            "invalid_predictions"
        ],
    )

    print(
        "Exact structured-schema rate:",
        (
            f'{metrics["exact_json_schema_rate"]:.4f}'
        ),
    )

    print(
        "Accuracy:",
        (
            f'{metrics["accuracy_valid_predictions"]:.4f}'
        ),
    )

    print(
        "Balanced accuracy:",
        (
            f'{metrics["balanced_accuracy"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS precision:",
        (
            f'{metrics["anomalous_precision"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS recall:",
        (
            f'{metrics["anomalous_recall"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS F1:",
        (
            f'{metrics["anomalous_f1"]:.4f}'
        ),
    )

    print(
        "NORMAL recall / specificity:",
        (
            f'{metrics["normal_recall_specificity"]:.4f}'
        ),
    )

    print(
        "MCC:",
        (
            f'{metrics["matthews_correlation_coefficient"]:.4f}'
        ),
    )

    print(
        "Matched source-group exact rate:",
        (
            f'{metrics["source_group_exact_match_rate"]:.4f}'
        ),
    )

    print(
        "Reasoning inconsistencies:",
        metrics[
            "reasoning_inconsistency_count"
        ],
    )


    print("\nCONFUSION MATRIX")

    display(
        confusion_df
    )


    print("\nCLASSIFICATION REPORT")

    display(
        classification_report_df
    )


    print("\nPER CASE FAMILY")

    display(
        family_metrics_df
    )


    print("\nPER EXACT CASE VARIANT")

    display(
        variant_metrics_df
    )


    print("\nASSESSMENT DISTRIBUTIONS")

    display(
        assessment_distributions_df
    )


    print(
        "\nSaved cache:",
        paths[
            "prediction_cache"
        ],
    )

    print(
        "Saved prompt:",
        paths[
            "prompt_template"
        ],
    )

    print(
        "Saved prompt diff:",
        paths[
            "prompt_diff"
        ],
    )

    print(
        "Saved predictions:",
        paths[
            "predictions_csv"
        ],
    )

    print(
        "Saved reasoning assessments:",
        paths[
            "reasoning_assessments_csv"
        ],
    )

    print(
        "Saved errors:",
        paths[
            "errors_csv"
        ],
    )


    if len(
        invalid_df
    ):

        print(
            "\nWARNING: Invalid predictions were excluded "
            "from the confusion matrix."
        )


        display(
            invalid_df[
                [
                    "case_id",
                    "case_family",
                    "case_variant",
                    "parse_mode",
                    "raw_output",
                    "generation_error",
                ]
            ]
        )


    else:

        print(
            "\nAll 68 cases produced valid "
            "binary predictions."
        )


    return {
        "metrics": metrics,
        "results_df": results_df,
        "confusion_df": confusion_df,
        "family_metrics_df": family_metrics_df,
        "variant_metrics_df": variant_metrics_df,
        "error_df": error_df,
        "reasoning_inconsistency_df": (
            reasoning_inconsistency_df
        ),
        "assessment_distributions_df": (
            assessment_distributions_df
        ),
    }


## 7. Load the exact improved semantic-policy workflow

In [ ]:

# ============================================================
# STRUCTURED R1 — IMPROVED SEMANTIC POLICY
# MANUAL PROMPT WORKFLOW HELPERS
#
# No automatic prompt merging or rewriting is performed.
#
# The notebook prints:
#   1. the exact targeted WRONG_PARTNER semantic prompt,
#   2. the exact full Structured R1 baseline prompt,
#      with semantics but without filtered turns/overlap.
#
# The final combined prompt must then be supplied manually.
#
# Model-facing evidence is fixed to:
#   - participant `speaks` only,
#   - complete local offset distribution,
#   - all five global temporal features,
#   - complete coarse + focused semantic summaries,
#   - no filtered turns,
#   - no overlap.
# ============================================================

from collections import Counter
from IPython.display import display

import copy
import json
import os
import re
import time


IMPROVED_SEMANTIC_BASELINE_EXPERIMENT_NAME = (
    "manual_ablation_l1_no_overlap_evidence_speaks_only"
)

IMPROVED_SEMANTIC_BASELINE_PROMPT_PATH = (
    OUT_DIR
    / IMPROVED_SEMANTIC_BASELINE_EXPERIMENT_NAME
    / "prompt_template.txt"
)

IMPROVED_SEMANTIC_EXPERIMENT_VERSION = (
    "structured_r1_improved_semantic_policy_"
    "speaks_offsets_all_global_no_overlap"
)

IMPROVED_SEMANTIC_EXPERIMENT_TITLE = (
    "Structured R1 — Improved Semantic Policy — "
    "Speaks + Offsets + All Global + Coarse/Focused — No Overlap"
)

IMPROVED_SEMANTIC_ABLATION_ID = (
    "R1_IMPROVED_SEMANTIC_POLICY_"
    "SPEAKS_OFFSETS_ALL_GLOBAL_NO_OVERLAP"
)

IMPROVED_SEMANTIC_INSPECTED_EXPERIMENTS = set()


LOCAL_OVERLAP_FIELDS_IMPROVED = [
    "clean_overlap_seconds",
    "clean_overlap_percent",
]

LOCAL_OFFSET_DISTRIBUTION_FIELDS_IMPROVED = [
    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",
    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",
    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]

GLOBAL_MODEL_FEATURE_FIELDS_IMPROVED = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage_percent",
]

assert (
    LOCAL_OVERLAP_FIELDS_IMPROVED
    + LOCAL_OFFSET_DISTRIBUTION_FIELDS_IMPROVED
    == LOCAL_FEATURE_FIELDS
)


# ============================================================
# EXACT TARGETED WRONG-PARTNER PROMPT
#
# Extracted from the retained coarse + full-focused semantic-only
# validation experiment. It is printed for manual comparison only.
# ============================================================

TARGETED_WRONG_PARTNER_PROMPT_TEMPLATE = 'You are determining whether two people belong to the SAME real\n120-second dyadic conversation.\n\nYou are given semantic summaries from two synchronized 60-second\nsegments for Participant A and Participant B.\n\nYou must use ONLY the supplied semantic fields.\n\nFor each participant segment, the input contains:\n\nCOARSE SUMMARY:\n- speech_content_summary\n- apparent_topic\n\nFOCUSED SUMMARY:\n- detailed_speech_summary\n- main_topic\n- secondary_topics\n- key_semantic_details\n- summary_specificity\n- unclear_content\n- confidence\n\nThe coarse and focused fields are two independently generated semantic\ndescriptions of the SAME participant segment. Interpret them together.\nDo not treat differences between a participant\'s own coarse and focused\nsummaries as evidence of wrong partner.\n\nDo not use visual engagement, interaction style, speaking amount,\nlistening behavior, or any other information.\n\nThe summaries were independently generated and may sometimes be broad,\nimperfect, or uncertain.\n\nIMPORTANT INTERPRETATION\n\nNORMAL:\n- The two participants\' content can plausibly belong to the same conversation.\n- They may discuss different aspects of a shared subject.\n- One participant\'s content may plausibly respond to, elaborate on, or provide\n  context for the other participant\'s content.\n- A conversation may naturally change topic between Segment 0 and Segment 1.\n- Exact word or topic-label matching is not required.\n\nWRONG_PARTNER:\n- The participants repeatedly discuss unrelated concrete subjects.\n- There is no plausible shared conversational context across the 120 seconds.\n- Both synchronized segments show semantic mismatch, or one segment shows a\n  strong concrete mismatch while the other provides no credible compatibility.\n- The fact that both participants discuss personal experiences, preferences,\n  opinions, daily life, products, or general topics is NOT by itself evidence\n  that they belong to the same conversation.\n\nGENERIC OR WEAK SUMMARIES\n\nBroad labels such as:\n- personal experiences\n- personal preferences\n- general discussion\n- daily life\n- opinions\n- lifestyle\n- personal well-being\n\nmust not be treated as evidence of compatibility unless the concrete content\nalso provides a plausible semantic connection.\n\nIf one segment is vague or generic, treat that segment as\nINSUFFICIENT_EVIDENCE rather than as evidence for NORMAL.\n\nEvaluate the complete 120-second pattern. Do not use a simple vote between\nthe two segments.\n\nReturn ONLY valid JSON with exactly this schema:\n\n{{\n  "label": "NORMAL or ANOMALOUS",\n  "anomaly_type": "none or wrong_partner",\n  "confidence": 0.0,\n  "segment_0_assessment": "compatible / mismatch / insufficient_evidence",\n  "segment_1_assessment": "compatible / mismatch / insufficient_evidence",\n  "cross_segment_assessment": "short assessment of the complete 120-second semantic relationship",\n  "reasoning": "brief evidence-based explanation"\n}}\n\nSEMANTIC INPUT:\n\n{semantic_input}'


WP_DATABASE_SEGMENTS = [
    "segment_0_0_to_60_seconds",
    "segment_1_60_to_120_seconds",
]

WP_COARSE_FIELDS_EXACT = [
    "speech_content_summary",
    "apparent_topic",
]

WP_FOCUSED_FIELDS_EXACT = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


def build_exact_wp_semantic_input(case):
    combined = {
        "participant_A": {},
        "participant_B": {},
    }

    for role in [
        "participant_A",
        "participant_B",
    ]:
        participant_semantics = case[
            "semantic_summaries"
        ][role]

        for segment_name in WP_DATABASE_SEGMENTS:
            segment_record = participant_semantics[
                segment_name
            ]

            coarse = select_exact_fields(
                segment_record[
                    "coarse_summary"
                ],
                WP_COARSE_FIELDS_EXACT,
                f"{role}/{segment_name}/coarse_summary",
            )

            focused = select_exact_fields(
                segment_record[
                    "focused_summary"
                ],
                WP_FOCUSED_FIELDS_EXACT,
                f"{role}/{segment_name}/focused_summary",
            )

            assert "speaks" not in focused

            combined[role][segment_name] = {
                **coarse,
                "focused_summary": focused,
            }

    return combined


def print_targeted_wrong_partner_prompt(
    *,
    case_index=0,
    prefer_wrong_partner=True,
):
    wp_cases = sorted(
        [
            case
            for case in consolidation_cases
            if get_case_family(case) in {
                "normal",
                "wrong_partner",
            }
        ],
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert len(wp_cases) == 34

    family_counts = Counter(
        get_case_family(case)
        for case in wp_cases
    )

    assert family_counts == {
        "normal": 17,
        "wrong_partner": 17,
    }

    if prefer_wrong_partner:
        candidate_cases = [
            case
            for case in wp_cases
            if get_case_family(case)
            == "wrong_partner"
        ]
    else:
        candidate_cases = wp_cases

    assert 0 <= case_index < len(candidate_cases)

    case = candidate_cases[
        case_index
    ]

    semantic_input = (
        build_exact_wp_semantic_input(
            case
        )
    )

    rendered_prompt = (
        TARGETED_WRONG_PARTNER_PROMPT_TEMPLATE
        .format(
            semantic_input=json.dumps(
                semantic_input,
                indent=2,
                ensure_ascii=False,
            )
        )
    )

    print("=" * 100)
    print(
        "EXACT TARGETED NORMAL vs WRONG_PARTNER PROMPT"
    )
    print("=" * 100)
    print(
        "Inspection case ID:",
        case[
            "case_id"
        ],
    )
    print(
        "Case metadata is printed outside the model prompt only."
    )
    print(
        "Temporal evidence supplied:",
        False,
    )
    print(
        "Participant speaks supplied:",
        False,
    )
    print(
        "Semantic evidence supplied:",
        "coarse + full focused",
    )
    print("\n" + "=" * 100)
    print("EXACT RENDERED TARGETED PROMPT")
    print("=" * 100)
    print(
        rendered_prompt
    )

    return {
        "case": case,
        "semantic_input": semantic_input,
        "prompt": rendered_prompt,
    }


# ============================================================
# LOAD THE FROZEN FULL STRUCTURED R1 BASELINE PROMPT
#
# Exact L1 No-Overlap baseline:
# speaks + offsets + all global + coarse/focused semantics.
# ============================================================

def load_full_r1_no_overlap_prompt_template():
    assert (
        IMPROVED_SEMANTIC_BASELINE_PROMPT_PATH
        .exists()
    ), (
        "The validated full Structured R1 L1 No-Overlap prompt "
        "was not found:\n"
        f"{IMPROVED_SEMANTIC_BASELINE_PROMPT_PATH}\n\n"
        "Run the L1 configuration/inspection cell in the local "
        "temporal ablation notebook first so prompt_template.txt "
        "is available."
    )

    template = (
        IMPROVED_SEMANTIC_BASELINE_PROMPT_PATH
        .read_text(
            encoding="utf-8"
        )
    )

    required_markers = [
        "Whether each participant speaks",
        "PARTICIPATION VALIDITY",
        "LOCAL TEMPORAL FEATURES",
        "GLOBAL ALIGNMENT-SHIFT FEATURES",
        "SEMANTIC EVIDENCE",
        "STRUCTURED REASONING OUTPUT",
        '"participation_assessment"',
        '"local_temporal_assessment"',
        '"global_temporal_assessment"',
        '"temporal_assessment"',
        '"semantic_assessment"',
        '"decisive_dimension"',
        "{participant_A_speaks}",
        "{participant_B_speaks}",
        "{local_temporal_features}",
        "{global_shift_features}",
        "{semantic_summaries}",
    ]

    for marker in required_markers:
        assert marker in template, (
            "The saved full Structured R1 prompt is missing: "
            f"{marker}"
        )

    forbidden_markers = [
        "Filtered turns:",
        "{participant_A_turns}",
        "{participant_B_turns}",
        "Use the actual filtered turn lists",
        "TURN FORMAT AND BACKCHANNEL FILTERING",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "FILTERED OVERLAP",
    ]

    template_lower = template.lower()

    for marker in forbidden_markers:
        assert marker.lower() not in template_lower, (
            "Filtered-turn or overlap content exists in the "
            f"saved baseline prompt: {marker}"
        )

    return template


# ============================================================
# FIXED MODEL-FACING INPUT
# ============================================================

def build_improved_semantic_model_input(case):
    payload = copy.deepcopy(
        build_binary_model_input(
            case
        )
    )

    for role in [
        "participant_A",
        "participant_B",
    ]:
        participant_payload = payload[
            role
        ]

        assert "speaks" in participant_payload
        assert "filtered_turns" in participant_payload

        participant_payload.pop(
            "filtered_turns"
        )

        assert set(
            participant_payload
        ) == {
            "speaks"
        }

    local_features = payload[
        "local_temporal_features"
    ]

    for field in (
        LOCAL_OVERLAP_FIELDS_IMPROVED
    ):
        local_features.pop(
            field
        )

    assert set(
        local_features
    ) == set(
        LOCAL_OFFSET_DISTRIBUTION_FIELDS_IMPROVED
    )

    assert set(
        payload[
            "global_shift_features"
        ]
    ) == set(
        GLOBAL_MODEL_FEATURE_FIELDS_IMPROVED
    )

    assert (
        "semantic_summaries"
        in payload
    )

    semantic_keys = (
        collect_nested_keys_reasoning(
            payload[
                "semantic_summaries"
            ]
        )
    )

    for required_key in [
        "speech_content_summary",
        "apparent_topic",
        "detailed_speech_summary",
        "main_topic",
        "secondary_topics",
        "key_semantic_details",
        "summary_specificity",
        "unclear_content",
        "confidence",
    ]:
        assert any(
            key.endswith(
                required_key
            )
            for key in semantic_keys
        ), (
            "Missing semantic field: "
            f"{required_key}"
        )

    return payload


def render_full_r1_prompt_template(
    prompt_template,
    payload,
):
    prompt = prompt_template.format(
        normal_base_reference_text=(
            normal_base_reference_text
        ),
        normal_global_reference_text=(
            normal_global_reference_text
        ),
        duration_seconds=(
            payload[
                "analysis_duration_seconds"
            ]
        ),
        participant_A_speaks=canonical_json(
            payload[
                "participant_A"
            ][
                "speaks"
            ]
        ),
        participant_B_speaks=canonical_json(
            payload[
                "participant_B"
            ][
                "speaks"
            ]
        ),
        local_temporal_features=json.dumps(
            payload[
                "local_temporal_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
        global_shift_features=json.dumps(
            payload[
                "global_shift_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
        semantic_summaries=json.dumps(
            payload[
                "semantic_summaries"
            ],
            indent=2,
            ensure_ascii=False,
        ),
    )

    prompt_lower = prompt.lower()

    for forbidden_key in FORBIDDEN_PROMPT_KEYS:
        assert (
            forbidden_key.lower()
            not in prompt_lower
        )

    for forbidden_marker in [
        "filtered turns:",
        "participant_a_filtered_turns",
        "participant_b_filtered_turns",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "filtered overlap",
    ]:
        assert (
            forbidden_marker
            not in prompt_lower
        )

    return prompt


def print_full_r1_no_overlap_prompt(
    *,
    case_index=0,
):
    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert (
        0
        <= case_index
        < len(
            ordered_cases
        )
    )

    case = ordered_cases[
        case_index
    ]

    template = (
        load_full_r1_no_overlap_prompt_template()
    )

    payload = (
        build_improved_semantic_model_input(
            case
        )
    )

    prompt = (
        render_full_r1_prompt_template(
            template,
            payload,
        )
    )

    export_dir = (
        OUT_DIR
        / "improved_semantic_policy_manual_prompts"
    )

    export_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    export_path = (
        export_dir
        / "full_structured_r1_no_overlap_rendered_prompt.txt"
    )

    export_path.write_text(
        prompt,
        encoding="utf-8",
    )

    print("=" * 100)
    print(
        "EXACT FULL STRUCTURED R1 BASELINE PROMPT"
    )
    print(
        "Speaks + Offsets + All Global + Coarse/Focused"
    )
    print(
        "No Filtered Turns · No Overlap"
    )
    print("=" * 100)
    print(
        "Inspection case ID:",
        case[
            "case_id"
        ],
    )
    print(
        "Filtered turns supplied:",
        False,
    )
    print(
        "Overlap evidence supplied:",
        False,
    )
    print(
        "Semantic evidence supplied:",
        "coarse + focused",
    )
    print("\nEXACT MODEL-FACING PAYLOAD")
    print(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
    )
    print("\n" + "=" * 100)
    print("EXACT RENDERED STRUCTURED R1 PROMPT")
    print("=" * 100)
    print(
        prompt
    )
    print("\nSaved rendered prompt:", export_path)

    return {
        "case": case,
        "payload": payload,
        "prompt": prompt,
        "template": template,
        "export_path": export_path,
    }


# ============================================================
# MANUAL COMBINED-PROMPT RENDERER
# ============================================================

def render_manual_improved_semantic_prompt(
    prompt_template,
    payload,
):
    assert isinstance(
        prompt_template,
        str,
    )

    assert (
        prompt_template.strip()
    )

    prompt = prompt_template.format(
        normal_base_reference_text=(
            normal_base_reference_text
        ),
        normal_global_reference_text=(
            normal_global_reference_text
        ),
        duration_seconds=(
            payload[
                "analysis_duration_seconds"
            ]
        ),
        participant_A_speaks=canonical_json(
            payload[
                "participant_A"
            ][
                "speaks"
            ]
        ),
        participant_B_speaks=canonical_json(
            payload[
                "participant_B"
            ][
                "speaks"
            ]
        ),
        local_temporal_features=json.dumps(
            payload[
                "local_temporal_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
        global_shift_features=json.dumps(
            payload[
                "global_shift_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
        semantic_summaries=json.dumps(
            payload[
                "semantic_summaries"
            ],
            indent=2,
            ensure_ascii=False,
        ),
    )

    prompt_lower = (
        prompt.lower()
    )

    for forbidden_key in FORBIDDEN_PROMPT_KEYS:
        assert (
            forbidden_key.lower()
            not in prompt_lower
        ), (
            "Forbidden field leaked into prompt: "
            f"{forbidden_key}"
        )

    for forbidden_marker in [
        "filtered turns:",
        "participant_a_filtered_turns",
        "participant_b_filtered_turns",
        "{participant_a_turns}",
        "{participant_b_turns}",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "filtered overlap",
    ]:
        assert (
            forbidden_marker
            not in prompt_lower
        ), (
            "Forbidden turns/overlap marker: "
            f"{forbidden_marker}"
        )

    return prompt


# ============================================================
# MANUAL PROMPT VALIDATOR
# ============================================================

def validate_manual_improved_semantic_prompt(
    prompt_template,
):
    assert isinstance(
        prompt_template,
        str,
    ), (
        "Set R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE "
        "to one complete prompt string."
    )

    template = (
        prompt_template.strip()
    )

    assert template

    required_placeholders = [
        "{duration_seconds}",
        "{participant_A_speaks}",
        "{participant_B_speaks}",
        "{local_temporal_features}",
        "{global_shift_features}",
        "{semantic_summaries}",
    ]

    for placeholder in required_placeholders:
        assert placeholder in template, (
            "Missing placeholder: "
            f"{placeholder}"
        )

    required_sections = [
        "PARTICIPATION VALIDITY",
        "LOCAL TEMPORAL FEATURES",
        "GLOBAL ALIGNMENT-SHIFT FEATURES",
        "SEMANTIC EVIDENCE",
        "FINAL COMBINED DECISION",
        "CURRENT CASE",
        "STRUCTURED REASONING OUTPUT",
        "OUTPUT",
    ]

    for marker in required_sections:
        assert marker in template, (
            "Missing required section: "
            f"{marker}"
        )

    required_schema_markers = [
        '"participation_assessment"',
        '"local_temporal_assessment"',
        '"global_temporal_assessment"',
        '"temporal_assessment"',
        '"semantic_assessment"',
        '"decisive_dimension"',
        '"label"',
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    ]

    for marker in required_schema_markers:
        assert marker in template, (
            "Missing structured-output marker: "
            f"{marker}"
        )

    semantic_policy_markers = [
        "plausible",
        "concrete",
        "insufficient",
        "generic",
        "same conversation",
    ]

    template_lower = (
        template.lower()
    )

    for marker in semantic_policy_markers:
        assert marker in template_lower, (
            "The manual semantic policy appears incomplete. "
            f"Missing concept: {marker}"
        )

    forbidden_markers = [
        "filtered turns:",
        "{participant_A_turns}",
        "{participant_B_turns}",
        "TURN FORMAT AND BACKCHANNEL FILTERING",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "FILTERED OVERLAP",
    ]

    for marker in forbidden_markers:
        assert marker.lower() not in template_lower, (
            "Filtered-turn or overlap content remains: "
            f"{marker}"
        )

    return template


def build_manual_improved_semantic_prompt(
    case,
    config,
):
    payload = (
        build_improved_semantic_model_input(
            case
        )
    )

    prompt = (
        render_manual_improved_semantic_prompt(
            config[
                "reasoning_prompt_template"
            ],
            payload,
        )
    )

    return (
        prompt,
        payload,
    )


# ============================================================
# PREPARE CONFIG
# ============================================================

def prepare_manual_improved_semantic_experiment(
    *,
    manual_prompt_template,
):
    prompt_template = (
        validate_manual_improved_semantic_prompt(
            manual_prompt_template
        )
    )

    baseline_template = (
        load_full_r1_no_overlap_prompt_template()
    )

    experiment_dir = (
        OUT_DIR
        / IMPROVED_SEMANTIC_EXPERIMENT_VERSION
    )

    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    paths = {
        "prediction_cache": (
            experiment_dir
            / "predictions_cache.json"
        ),
        "predictions_csv": (
            experiment_dir
            / "predictions_all_68.csv"
        ),
        "metrics_json": (
            experiment_dir
            / "metrics.json"
        ),
        "confusion_matrix_csv": (
            experiment_dir
            / "confusion_matrix.csv"
        ),
        "family_metrics_csv": (
            experiment_dir
            / "metrics_by_case_family.csv"
        ),
        "variant_metrics_csv": (
            experiment_dir
            / "metrics_by_case_variant.csv"
        ),
        "errors_csv": (
            experiment_dir
            / "classification_errors.csv"
        ),
        "reasoning_assessments_csv": (
            experiment_dir
            / "reasoning_assessments.csv"
        ),
        "reasoning_inconsistencies_csv": (
            experiment_dir
            / "reasoning_inconsistencies.csv"
        ),
        "assessment_distributions_csv": (
            experiment_dir
            / "assessment_distributions.csv"
        ),
        "prompt_template": (
            experiment_dir
            / "prompt_template.txt"
        ),
        "baseline_prompt_copy": (
            experiment_dir
            / "baseline_prompt_template.txt"
        ),
        "targeted_wp_prompt_copy": (
            experiment_dir
            / "targeted_wrong_partner_prompt_template.txt"
        ),
        "experiment_manifest": (
            experiment_dir
            / "experiment_manifest.json"
        ),
    }

    reasoning_prompt_sha256 = (
        sha256_text(
            prompt_template
        )
    )

    baseline_prompt_sha256 = (
        sha256_text(
            baseline_template
        )
    )

    targeted_wp_prompt_sha256 = (
        sha256_text(
            TARGETED_WRONG_PARTNER_PROMPT_TEMPLATE
        )
    )

    normal_reference_sha256 = (
        sha256_text(
            normal_base_reference_text
            + "\n"
            + normal_global_reference_text
        )
    )

    paths[
        "prompt_template"
    ].write_text(
        prompt_template,
        encoding="utf-8",
    )

    paths[
        "baseline_prompt_copy"
    ].write_text(
        baseline_template,
        encoding="utf-8",
    )

    paths[
        "targeted_wp_prompt_copy"
    ].write_text(
        TARGETED_WRONG_PARTNER_PROMPT_TEMPLATE,
        encoding="utf-8",
    )

    example_prompt, example_payload = (
        build_manual_improved_semantic_prompt(
            consolidation_cases[0],
            {
                "reasoning_prompt_template": (
                    prompt_template
                )
            },
        )
    )

    manifest = {
        "experiment_version": (
            IMPROVED_SEMANTIC_EXPERIMENT_VERSION
        ),
        "experiment_title": (
            IMPROVED_SEMANTIC_EXPERIMENT_TITLE
        ),
        "ablation_id": (
            IMPROVED_SEMANTIC_ABLATION_ID
        ),
        "source_experiment": (
            IMPROVED_SEMANTIC_BASELINE_EXPERIMENT_NAME
        ),
        "model_id": MODEL_ID,
        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),
        "reasoning_prompt_sha256": (
            reasoning_prompt_sha256
        ),
        "baseline_prompt_sha256": (
            baseline_prompt_sha256
        ),
        "targeted_wp_prompt_sha256": (
            targeted_wp_prompt_sha256
        ),
        "normal_reference_sha256": (
            normal_reference_sha256
        ),
        "semantic_input": (
            "coarse_and_focused"
        ),
        "focused_summaries_used": True,
        "filtered_turns_supplied": False,
        "overlap_supplied": False,
        "participant_model_input": (
            "speaks_only"
        ),
        "local_model_input": (
            "complete_offset_distribution_only"
        ),
        "global_model_input": (
            "all_five_global_features"
        ),
        "temporal_profiles_used": (
            "frozen_NORMAL_only"
        ),
        "assessment_policy": (
            "structured_R1_with_targeted_semantic_policy"
        ),
        "schema_keys": list(
            REASONING_SCHEMA_KEYS
        ),
        "prompt_definition_method": (
            "manual_complete_template"
        ),
        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },
    }

    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    config = {
        **manifest,
        "experiment_dir": (
            experiment_dir
        ),
        "reasoning_prompt_template": (
            prompt_template
        ),
        "paths": paths,
    }

    print("=" * 100)
    print(
        "IMPROVED SEMANTIC POLICY — CONFIGURATION READY"
    )
    print("=" * 100)
    print(
        "Experiment version:",
        config[
            "experiment_version"
        ],
    )
    print(
        "Cases:",
        len(
            consolidation_cases
        ),
    )
    print(
        "Semantic input:",
        config[
            "semantic_input"
        ],
    )
    print(
        "Filtered turns supplied:",
        False,
    )
    print(
        "Overlap supplied:",
        False,
    )
    print(
        "Prompt SHA256:",
        reasoning_prompt_sha256,
    )
    print(
        "Example prompt characters:",
        len(
            example_prompt
        ),
    )
    print(
        "Prediction cache:",
        paths[
            "prediction_cache"
        ],
    )
    print(
        "Existing cache:",
        paths[
            "prediction_cache"
        ].exists(),
    )

    return config


# ============================================================
# REQUIRED INSPECTION
# ============================================================

def inspect_manual_improved_semantic_prompt(
    config,
    *,
    case_index=0,
):
    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert (
        0
        <= case_index
        < len(
            ordered_cases
        )
    )

    case = ordered_cases[
        case_index
    ]

    prompt, payload = (
        build_manual_improved_semantic_prompt(
            case,
            config,
        )
    )

    print("=" * 100)
    print(
        "MANUAL PROMPT INSPECTION — "
        "STRUCTURED R1 IMPROVED SEMANTIC POLICY"
    )
    print("=" * 100)
    print(
        "Inspection case ID:",
        case[
            "case_id"
        ],
    )
    print(
        "Gold label is intentionally NOT printed inside the model prompt."
    )
    print(
        "Participant evidence supplied:",
        "speaks only",
    )
    print(
        "Local evidence supplied:",
        "complete offset distribution only",
    )
    print(
        "Global evidence supplied:",
        "all five global temporal features",
    )
    print(
        "Semantic evidence supplied:",
        "coarse + focused",
    )
    print(
        "Filtered turns supplied:",
        False,
    )
    print(
        "Overlap supplied:",
        False,
    )
    print("\n" + "=" * 100)
    print("EXACT MODEL-FACING PAYLOAD")
    print("=" * 100)
    print(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
    )
    print("\n" + "=" * 100)
    print("EXACT RENDERED MODEL PROMPT")
    print("=" * 100)
    print(
        prompt
    )

    IMPROVED_SEMANTIC_INSPECTED_EXPERIMENTS.add(
        config[
            "experiment_version"
        ]
    )

    print(
        "\nInspection unlocked:",
        config[
            "experiment_version"
        ]
        in IMPROVED_SEMANTIC_INSPECTED_EXPERIMENTS,
    )

    return {
        "case": case,
        "payload": payload,
        "prompt": prompt,
    }


# ============================================================
# CACHE
# ============================================================

def create_improved_semantic_cache(
    config,
):
    return {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),
        "experiment_title": (
            config[
                "experiment_title"
            ]
        ),
        "ablation_id": (
            config[
                "ablation_id"
            ]
        ),
        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),
        "model_id": (
            config[
                "model_id"
            ]
        ),
        "reasoning_prompt_sha256": (
            config[
                "reasoning_prompt_sha256"
            ]
        ),
        "baseline_prompt_sha256": (
            config[
                "baseline_prompt_sha256"
            ]
        ),
        "targeted_wp_prompt_sha256": (
            config[
                "targeted_wp_prompt_sha256"
            ]
        ),
        "normal_reference_sha256": (
            config[
                "normal_reference_sha256"
            ]
        ),
        "semantic_input": (
            config[
                "semantic_input"
            ]
        ),
        "focused_summaries_used": True,
        "filtered_turns_supplied": False,
        "overlap_supplied": False,
        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),
        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),
        "schema_keys": list(
            REASONING_SCHEMA_KEYS
        ),
        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),
        "created_at_utc": (
            reasoning_utc_now()
        ),
        "updated_at_utc": (
            reasoning_utc_now()
        ),
        "records": {},
    }


# ============================================================
# INFERENCE
# ============================================================

def run_manual_improved_semantic_experiment(
    config,
    *,
    print_each_case_prompt=False,
):
    assert (
        config[
            "experiment_version"
        ]
        in IMPROVED_SEMANTIC_INSPECTED_EXPERIMENTS
    ), (
        "Run the exact manual prompt-inspection cell "
        "before inference."
    )

    prediction_cache_path = (
        config[
            "paths"
        ][
            "prediction_cache"
        ]
    )

    expected_cache = (
        create_improved_semantic_cache(
            config
        )
    )

    if prediction_cache_path.exists():
        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )

        for key in [
            "experiment_version",
            "ablation_id",
            "source_experiment",
            "model_id",
            "reasoning_prompt_sha256",
            "baseline_prompt_sha256",
            "targeted_wp_prompt_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "focused_summaries_used",
            "filtered_turns_supplied",
            "overlap_supplied",
            "temporal_profiles_used",
            "assessment_policy",
            "schema_keys",
            "max_new_tokens",
        ]:
            assert (
                prediction_cache[
                    key
                ]
                == expected_cache[
                    key
                ]
            ), (
                "Cache mismatch for "
                f"{key}"
            )

        print(
            "Resuming cache:",
            prediction_cache_path,
        )
        print(
            "Existing records:",
            len(
                prediction_cache[
                    "records"
                ]
            ),
        )

    else:
        prediction_cache = expected_cache

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

        print(
            "Created cache:",
            prediction_cache_path,
        )

    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert len(
        ordered_cases
    ) == 68

    for case in tqdm(
        ordered_cases,
        desc=(
            config[
                "experiment_version"
            ]
        ),
    ):
        case_id = str(
            case[
                "case_id"
            ]
        )

        prompt, payload = (
            build_manual_improved_semantic_prompt(
                case,
                config,
            )
        )

        if print_each_case_prompt:
            print("\n" + "=" * 100)
            print("CASE:", case_id)
            print("=" * 100)
            print(prompt)

        prompt_sha256 = (
            sha256_text(
                prompt
            )
        )

        payload_sha256 = (
            sha256_text(
                canonical_json(
                    payload
                )
            )
        )

        existing_record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )

        if (
            existing_record is not None
            and existing_record.get(
                "prediction"
            ) in LABELS
        ):
            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )
            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == payload_sha256
            )
            continue

        started = (
            time.perf_counter()
        )

        try:
            raw_output, input_token_count = (
                qwen_text_only_binary(
                    prompt,
                    max_new_tokens=(
                        MAX_NEW_TOKENS_REASONING
                    ),
                )
            )

            parsed_result = (
                parse_structured_reasoning_prediction(
                    raw_output
                )
            )

            generation_error = None

        except Exception as exc:
            raw_output = ""
            input_token_count = None

            parsed_result = {
                "prediction": None,
                "parse_mode": (
                    "generation_error"
                ),
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
                "participation_assessment": None,
                "local_temporal_assessment": None,
                "global_temporal_assessment": None,
                "temporal_assessment": None,
                "semantic_assessment": None,
                "decisive_dimension": None,
            }

            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        elapsed_seconds = (
            time.perf_counter()
            - started
        )

        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": case_id,
            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),
            "case_family": (
                get_case_family(
                    case
                )
            ),
            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),
            "gold_binary_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),
            "prompt_sha256": (
                prompt_sha256
            ),
            "input_payload_sha256": (
                payload_sha256
            ),
            "semantic_input": (
                "coarse_and_focused"
            ),
            "focused_summaries_used": True,
            "input_token_count": (
                input_token_count
            ),
            "max_new_tokens": (
                MAX_NEW_TOKENS_REASONING
            ),
            "raw_output": (
                raw_output
            ),
            "prediction": (
                parsed_result[
                    "prediction"
                ]
            ),
            "participation_assessment": (
                parsed_result[
                    "participation_assessment"
                ]
            ),
            "local_temporal_assessment": (
                parsed_result[
                    "local_temporal_assessment"
                ]
            ),
            "global_temporal_assessment": (
                parsed_result[
                    "global_temporal_assessment"
                ]
            ),
            "temporal_assessment": (
                parsed_result[
                    "temporal_assessment"
                ]
            ),
            "semantic_assessment": (
                parsed_result[
                    "semantic_assessment"
                ]
            ),
            "decisive_dimension": (
                parsed_result[
                    "decisive_dimension"
                ]
            ),
            "parse_mode": (
                parsed_result[
                    "parse_mode"
                ]
            ),
            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),
            "schema_errors": (
                parsed_result[
                    "schema_errors"
                ]
            ),
            "parsed_output": (
                parsed_result[
                    "parsed_output"
                ]
            ),
            "generation_error": (
                generation_error
            ),
            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),
            "completed_at_utc": (
                reasoning_utc_now()
            ),
        }

        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

    print("\n" + "=" * 100)
    print(
        "IMPROVED SEMANTIC POLICY — INFERENCE COMPLETE"
    )
    print("=" * 100)
    print(
        "Cached records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )
    print(
        "Prediction cache:",
        prediction_cache_path,
    )

    return prediction_cache


## 8. Reconstruct the exact improved semantic prompt

In [ ]:
# STEP 3 — MANUALLY DEFINE THE COMPLETE COMBINED PROMPT
#
# Do not generate this automatically.
#
# Start from the exact full Structured R1 prompt printed above.
# Keep its participation, local temporal, global temporal,
# final-decision and structured-output logic fixed.
#
# Replace only the semantic reasoning policy with a carefully
# integrated version of the targeted wrong-partner logic.
#
# The final prompt must retain all placeholders:
#   {duration_seconds}
#   {participant_A_speaks}
#   {participant_B_speaks}
#   {local_temporal_features}
#   {global_shift_features}
#   {semantic_summaries}

# ============================================================
# STEP 3 — MANUALLY DEFINE THE IMPROVED SEMANTIC R1 PROMPT
#
# Surgical intervention:
# - keep the complete Structured R1 baseline unchanged
# - replace only the SEMANTIC EVIDENCE policy
# ============================================================

import difflib


# Load the exact full Structured R1 baseline template:
# - speaks only
# - complete local offset distribution
# - all five global features
# - coarse + focused semantics
# - no filtered turns
# - no overlap
ORIGINAL_FULL_R1_PROMPT_TEMPLATE = (
    load_full_r1_no_overlap_prompt_template()
)


SEMANTIC_SECTION_HEADER = """
============================================================
SEMANTIC EVIDENCE
============================================================
""".strip()


FINAL_DECISION_SECTION_HEADER = """
============================================================
FINAL COMBINED DECISION
============================================================
""".strip()


# ============================================================
# VERIFY THAT THE TWO SECTION BOUNDARIES ARE UNIQUE
# ============================================================

assert (
    ORIGINAL_FULL_R1_PROMPT_TEMPLATE.count(
        SEMANTIC_SECTION_HEADER
    )
    == 1
), (
    "Expected exactly one SEMANTIC EVIDENCE section."
)


assert (
    ORIGINAL_FULL_R1_PROMPT_TEMPLATE.count(
        FINAL_DECISION_SECTION_HEADER
    )
    == 1
), (
    "Expected exactly one FINAL COMBINED DECISION section."
)


(
    PROMPT_BEFORE_SEMANTIC,
    semantic_header_found,
    PROMPT_FROM_SEMANTIC_ONWARD,
) = ORIGINAL_FULL_R1_PROMPT_TEMPLATE.partition(
    SEMANTIC_SECTION_HEADER
)


assert semantic_header_found == SEMANTIC_SECTION_HEADER


(
    ORIGINAL_SEMANTIC_POLICY_BODY,
    final_decision_header_found,
    PROMPT_AFTER_FINAL_DECISION_HEADER,
) = PROMPT_FROM_SEMANTIC_ONWARD.partition(
    FINAL_DECISION_SECTION_HEADER
)


assert (
    final_decision_header_found
    == FINAL_DECISION_SECTION_HEADER
)


ORIGINAL_SEMANTIC_POLICY_BODY = (
    ORIGINAL_SEMANTIC_POLICY_BODY.strip()
)


# ============================================================
# IMPROVED SEMANTIC POLICY
#
# Adapted from the successful targeted wrong-partner pipeline.
#
# Crucially:
# - semantic_assessment uses semantic summaries only
# - coarse and focused summaries are interpreted together
# - specific unrelated content becomes INCOMPATIBLE
# - vague content becomes LIMITED, not COMPATIBLE
# - temporal and participation evidence cannot influence the
#   semantic_assessment
# ============================================================

IMPROVED_SEMANTIC_POLICY_BODY = r"""
The 120-second interval is divided into:

- Segment 0: 0 to 60 seconds
- Segment 1: 60 to 120 seconds

For each participant and synchronized segment, you receive:

COARSE SUMMARY:

- speech_content_summary
- apparent_topic

FOCUSED SUMMARY:

- detailed_speech_summary
- main_topic
- secondary_topics
- key_semantic_details
- summary_specificity
- unclear_content
- confidence

The coarse and focused fields are two independently generated semantic
descriptions of the SAME participant segment.

Interpret the coarse and focused fields together.

Do not treat differences between one participant's own coarse and
focused summaries as evidence of semantic incompatibility.

The summaries were independently generated and may sometimes be broad,
imperfect, repetitive, incomplete, or uncertain.

The semantic assessment must be based ONLY on the supplied semantic
summaries.

When assigning semantic_assessment:

- do not use the participation evidence,
- do not use the local temporal evidence,
- do not use the global temporal evidence,
- do not use whether the temporal pattern appears NORMAL or ANOMALOUS,
- and do not allow another reasoning branch to soften or strengthen
  the semantic assessment.

The semantic assessment must independently answer:

Can the two participants' supplied content plausibly belong to the same
real 120-second dyadic conversation?

Internally evaluate:

1. The semantic relationship between Participant A and Participant B
   in Segment 0.
2. The semantic relationship between Participant A and Participant B
   in Segment 1.
3. The complete cross-segment semantic relationship across the full
   120 seconds.

Do not expose these three internal checks as additional output fields.

Do not use a simple vote between Segment 0 and Segment 1.

Evaluate the complete 120-second semantic pattern.

============================================================
SEMANTIC COMPATIBILITY
============================================================

Semantic evidence is COMPATIBLE when the two participants' content can
plausibly belong to the same conversation.

A compatible relationship may include:

- a shared concrete subject,
- compatible people, events, places, experiences, or arguments,
- complementary accounts of the same situation,
- a plausible question-and-response relationship,
- one participant responding to or reacting to the other,
- one participant supplying context for the other,
- one participant elaborating on a detail introduced by the other,
- different aspects of one shared subject,
- or a coherent topic transition between Segment 0 and Segment 1.

The participants do not need to use identical words.

Exact topic-label matching is not required.

The participants may describe different parts of the same event or
story.

One participant may provide information that is not repeated by the
other participant, provided that the information still fits a plausible
shared conversational context.

A real conversation may naturally change topic during the 120-second
interval.

A topic change does not by itself establish semantic incompatibility.

Assign:

semantic_assessment = COMPATIBLE

only when the concrete content provides credible evidence that the two
participants can plausibly belong to the same conversation.

============================================================
SEMANTIC INCOMPATIBILITY
============================================================

Semantic evidence is INCOMPATIBLE when the participant summaries contain
specific, concrete content that does not support one plausible shared
conversation.

Strong semantic incompatibility may include:

- the participants repeatedly discussing unrelated concrete subjects,
- incompatible people, events, places, activities, products, stories,
  situations, or arguments,
- one participant describing a concrete subject while the other
  describes a clearly unrelated concrete subject,
- both synchronized segments showing semantic mismatch,
- or one synchronized segment showing a strong concrete mismatch while
  the other segment provides no credible evidence of compatibility.

Specific but unrelated content is INCOMPATIBLE.

It must not be treated as LIMITED merely because the model cannot find
a connection.

Do not invent an indirect relationship merely to make the participants
compatible.

Do not assume that two unrelated stories belong to the same conversation
only because both are personal stories.

Do not assume compatibility only because both participants discuss:

- personal experiences,
- personal preferences,
- opinions,
- daily life,
- family,
- relationships,
- products,
- health,
- lifestyle,
- personal well-being,
- or other broad categories.

The fact that two summaries fall under the same broad category is not
sufficient evidence that they describe the same conversation.

Assign:

semantic_assessment = INCOMPATIBLE

when the concrete semantic evidence strongly indicates that the two
participants do not belong to one plausible shared conversation.

============================================================
GENERIC, WEAK, OR UNCERTAIN SEMANTIC EVIDENCE
============================================================

Broad or generic labels such as:

- personal experiences
- personal preferences
- general discussion
- daily life
- opinions
- lifestyle
- personal well-being
- family matters
- emotional topics

must not be treated as evidence of compatibility unless the concrete
summary content also provides a plausible semantic connection.

If one synchronized segment is vague, generic, unclear, incomplete, or
low-confidence, treat that segment internally as insufficient evidence.

An insufficient segment must not automatically support COMPATIBLE.

However, one insufficient segment does not automatically require the
complete semantic assessment to be LIMITED.

Use the other synchronized segment and the complete 120-second pattern:

- If the available concrete evidence clearly supports one plausible
  shared conversation, assign COMPATIBLE.
- If the available concrete evidence shows a strong mismatch and the
  weak segment provides no credible compatibility, assign INCOMPATIBLE.
- If the available summaries are too vague, weak, contradictory, or
  uncertain to establish either compatibility or incompatibility,
  assign LIMITED.

Assign:

semantic_assessment = LIMITED

only when the semantic evidence is genuinely insufficient to determine
whether the two participants belong to the same conversation.

Do not use LIMITED for specific but unrelated content.

============================================================
SEMANTIC ASSESSMENT RULE
============================================================

Use exactly these meanings:

- COMPATIBLE:
  The concrete content supports a plausible shared conversation.

- INCOMPATIBLE:
  The concrete content provides strong evidence of unrelated participant
  records that do not plausibly form one conversation.

- LIMITED:
  The summaries are too generic, unclear, incomplete, contradictory, or
  uncertain to support either conclusion reliably.

The semantic assessment is an independent diagnostic assessment.

Determine it from the semantic summaries before applying the final
combined decision policy.

Semantic COMPATIBLE must not cancel reliable temporal failure.

Semantic INCOMPATIBLE must not be cancelled by apparently normal temporal
coordination.

Semantic LIMITED means that the semantic branch is inconclusive; it does
not itself establish either NORMAL or ANOMALOUS.
""".strip()


# ============================================================
# CONSTRUCT THE COMPLETE PROMPT
#
# Everything before and after SEMANTIC EVIDENCE remains exactly
# as it was in the original full Structured R1 baseline.
# ============================================================

R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE = (
    PROMPT_BEFORE_SEMANTIC
    + SEMANTIC_SECTION_HEADER
    + "\n\n"
    + IMPROVED_SEMANTIC_POLICY_BODY
    + "\n\n"
    + FINAL_DECISION_SECTION_HEADER
    + PROMPT_AFTER_FINAL_DECISION_HEADER
).strip()


# ============================================================
# AUDIT: VERIFY THAT ONLY THE SEMANTIC SECTION CHANGED
# ============================================================

assert (
    R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE
    != ORIGINAL_FULL_R1_PROMPT_TEMPLATE
)


# Prefix before semantic section must be exactly identical.
assert (
    R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE[
        :len(PROMPT_BEFORE_SEMANTIC)
    ]
    == PROMPT_BEFORE_SEMANTIC
)


# Everything following the FINAL COMBINED DECISION header must
# remain exactly identical.
improved_after_final = (
    R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE.split(
        FINAL_DECISION_SECTION_HEADER,
        1,
    )[1]
)


original_after_final = (
    ORIGINAL_FULL_R1_PROMPT_TEMPLATE.split(
        FINAL_DECISION_SECTION_HEADER,
        1,
    )[1]
)


assert improved_after_final == original_after_final


# Required model-facing placeholders must remain present.
REQUIRED_MANUAL_PROMPT_PLACEHOLDERS = [
    "{duration_seconds}",
    "{participant_A_speaks}",
    "{participant_B_speaks}",
    "{local_temporal_features}",
    "{global_shift_features}",
    "{semantic_summaries}",
]


for placeholder in REQUIRED_MANUAL_PROMPT_PLACEHOLDERS:
    assert (
        placeholder
        in R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE
    ), (
        f"Missing required placeholder: {placeholder}"
    )


# No filtered turns or overlap may reappear.
prompt_lower = (
    R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE.lower()
)


FORBIDDEN_MANUAL_PROMPT_MARKERS = [
    "filtered turns:",
    "{participant_a_turns}",
    "{participant_b_turns}",
    "turn format and backchannel filtering",
    "clean_overlap_seconds",
    "clean_overlap_percent",
    "filtered overlap",
]


for marker in FORBIDDEN_MANUAL_PROMPT_MARKERS:
    assert marker not in prompt_lower, (
        f"Forbidden content found: {marker}"
    )


# Verify the critical semantic-policy concepts.
REQUIRED_SEMANTIC_POLICY_MARKERS = [
    "specific but unrelated content is INCOMPATIBLE",
    "semantic_assessment = COMPATIBLE",
    "semantic_assessment = INCOMPATIBLE",
    "semantic_assessment = LIMITED",
    "same conversation",
    "plausible",
    "concrete",
    "generic",
    "insufficient",
]


for marker in REQUIRED_SEMANTIC_POLICY_MARKERS:
    assert (
        marker.lower()
        in prompt_lower
    ), (
        f"Missing semantic-policy marker: {marker}"
    )


# ============================================================
# DISPLAY THE EXACT CHANGE
# ============================================================

semantic_policy_diff = "\n".join(
    difflib.unified_diff(
        ORIGINAL_SEMANTIC_POLICY_BODY.splitlines(),
        IMPROVED_SEMANTIC_POLICY_BODY.splitlines(),
        fromfile="original_R1_semantic_policy",
        tofile="improved_R1_semantic_policy",
        lineterm="",
    )
)


print("=" * 100)
print("R1 IMPROVED SEMANTIC MANUAL PROMPT DEFINED")
print("=" * 100)

print(
    "Original complete prompt characters:",
    len(
        ORIGINAL_FULL_R1_PROMPT_TEMPLATE
    ),
)

print(
    "Improved complete prompt characters:",
    len(
        R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE
    ),
)

print(
    "Original semantic-policy characters:",
    len(
        ORIGINAL_SEMANTIC_POLICY_BODY
    ),
)

print(
    "Improved semantic-policy characters:",
    len(
        IMPROVED_SEMANTIC_POLICY_BODY
    ),
)

print(
    "All non-semantic prompt sections preserved exactly:",
    True,
)

print(
    "Filtered turns present:",
    "filtered turns:" in prompt_lower,
)

print(
    "Overlap present:",
    "filtered overlap" in prompt_lower,
)

print("\n" + "=" * 100)
print("SEMANTIC POLICY DIFF")
print("=" * 100)

print(
    semantic_policy_diff
)

R1 IMPROVED SEMANTIC MANUAL PROMPT DEFINED
Original complete prompt characters: 18363
Improved complete prompt characters: 24003
Original semantic-policy characters: 1872
Improved semantic-policy characters: 7512
All non-semantic prompt sections preserved exactly: True
Filtered turns present: False
Overlap present: False

SEMANTIC POLICY DIFF
--- original_R1_semantic_policy
+++ improved_R1_semantic_policy
@@ -3,14 +3,14 @@
 - Segment 0: 0 to 60 seconds
 - Segment 1: 60 to 120 seconds
 
-For each participant and segment, you receive:
-
-Coarse semantic information:
+For each participant and synchronized segment, you receive:
+
+COARSE SUMMARY:
 
 - speech_content_summary
 - apparent_topic
 
-Focused semantic information:
+FOCUSED SUMMARY:
 
 - detailed_speech_summary
 - main_topic
@@ -20,48 +20,217 @@
 - unclear_content
 - confidence
 
-The semantic summaries were independently generated and may be broad,
-imperfect, repetitive, or uncertain.
-
-Evaluate semantic compatibility rather th

## 9. Load the exact semantic-payload ablation framework

In [ ]:
# ============================================================
# SEMANTIC PAYLOAD ABLATION FRAMEWORK
#
# The full S-CF prompt defined above is the frozen policy baseline.
# Every ablation changes only:
#   1. the semantic fields placed in {semantic_summaries};
#   2. the prompt's inventory of the semantic fields actually supplied.
#
# Participation, temporal evidence, frozen references, semantic
# compatibility criteria, final-decision policy, and output schema
# remain unchanged.
# ============================================================

from collections import Counter
from pathlib import Path

import copy
import difflib
import json
import os
import re
import time


SEMANTIC_ABLATION_ROOT = (
    FINAL_EVALUATION_ROOT
    / "frozen_f1"
    / "structured_r1_improved_semantic_payload_ablation"
)

SEMANTIC_ABLATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


SEMANTIC_ABLATION_ALL_COARSE_FIELDS = [
    "speech_content_summary",
    "apparent_topic",
]

SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


SEMANTIC_ABLATION_SPECS = {
    "S-CF": {
        "title": "Coarse + Focused baseline",
        "round": "group_level",
        "coarse_fields": list(
            SEMANTIC_ABLATION_ALL_COARSE_FIELDS
        ),
        "focused_fields": list(
            SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS
        ),
        "removed_fields": [],
    },

    "S-F": {
        "title": "Focused summaries only",
        "round": "group_level",
        "coarse_fields": [],
        "focused_fields": list(
            SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS
        ),
        "removed_fields": [
            "coarse_summary/speech_content_summary",
            "coarse_summary/apparent_topic",
        ],
    },

    "S-C": {
        "title": "Coarse summaries only",
        "round": "group_level",
        "coarse_fields": list(
            SEMANTIC_ABLATION_ALL_COARSE_FIELDS
        ),
        "focused_fields": [],
        "removed_fields": [
            "focused_summary/detailed_speech_summary",
            "focused_summary/main_topic",
            "focused_summary/secondary_topics",
            "focused_summary/key_semantic_details",
            "focused_summary/summary_specificity",
            "focused_summary/unclear_content",
            "focused_summary/confidence",
        ],
    },

    "F1": {
        "title": "Without detailed_speech_summary",
        "round": "focused_leave_one_group_out",
        "coarse_fields": list(
            SEMANTIC_ABLATION_ALL_COARSE_FIELDS
        ),
        "focused_fields": [
            field
            for field in SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS
            if field != "detailed_speech_summary"
        ],
        "removed_fields": [
            "focused_summary/detailed_speech_summary",
        ],
    },

    "F2": {
        "title": "Without main_topic and secondary_topics",
        "round": "focused_leave_one_group_out",
        "coarse_fields": list(
            SEMANTIC_ABLATION_ALL_COARSE_FIELDS
        ),
        "focused_fields": [
            field
            for field in SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS
            if field not in {
                "main_topic",
                "secondary_topics",
            }
        ],
        "removed_fields": [
            "focused_summary/main_topic",
            "focused_summary/secondary_topics",
        ],
    },

    "F3": {
        "title": "Without key_semantic_details",
        "round": "focused_leave_one_group_out",
        "coarse_fields": list(
            SEMANTIC_ABLATION_ALL_COARSE_FIELDS
        ),
        "focused_fields": [
            field
            for field in SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS
            if field != "key_semantic_details"
        ],
        "removed_fields": [
            "focused_summary/key_semantic_details",
        ],
    },

    "F4": {
        "title": "Without reliability metadata",
        "round": "focused_leave_one_group_out",
        "coarse_fields": list(
            SEMANTIC_ABLATION_ALL_COARSE_FIELDS
        ),
        "focused_fields": [
            field
            for field in SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS
            if field not in {
                "summary_specificity",
                "unclear_content",
                "confidence",
            }
        ],
        "removed_fields": [
            "focused_summary/summary_specificity",
            "focused_summary/unclear_content",
            "focused_summary/confidence",
        ],
    },
}


SEMANTIC_ABLATION_INSPECTED_EXPERIMENTS = set()
SEMANTIC_ABLATION_CONFIGS = {}
SEMANTIC_ABLATION_EVALUATIONS = {}


# ============================================================
# VALIDATE THE ABLATION SPECIFICATIONS
# ============================================================

assert set(
    SEMANTIC_ABLATION_SPECS
) == {
    "S-CF",
    "S-F",
    "S-C",
    "F1",
    "F2",
    "F3",
    "F4",
}


for ablation_id, spec in (
    SEMANTIC_ABLATION_SPECS.items()
):
    assert (
        spec["coarse_fields"]
        or
        spec["focused_fields"]
    ), (
        f"{ablation_id} removes all semantic evidence."
    )

    assert set(
        spec["coarse_fields"]
    ).issubset(
        SEMANTIC_ABLATION_ALL_COARSE_FIELDS
    )

    assert set(
        spec["focused_fields"]
    ).issubset(
        SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS
    )


# ============================================================
# PROJECT THE FULL SEMANTIC PAYLOAD TO ONE ABLATION
# ============================================================

def project_semantic_summaries_for_ablation(
    semantic_summaries,
    ablation_id,
):
    assert ablation_id in SEMANTIC_ABLATION_SPECS

    spec = SEMANTIC_ABLATION_SPECS[
        ablation_id
    ]

    projected = {}

    for role in [
        "participant_A",
        "participant_B",
    ]:
        role_input = semantic_summaries[
            role
        ]

        role_output = {}

        for segment_name, segment_input in (
            role_input.items()
        ):
            segment_output = {}

            if spec["coarse_fields"]:
                segment_output[
                    "coarse_summary"
                ] = select_exact_fields(
                    segment_input[
                        "coarse_summary"
                    ],
                    spec["coarse_fields"],
                    (
                        f"{ablation_id}/{role}/"
                        f"{segment_name}/coarse_summary"
                    ),
                )

            if spec["focused_fields"]:
                segment_output[
                    "focused_summary"
                ] = select_exact_fields(
                    segment_input[
                        "focused_summary"
                    ],
                    spec["focused_fields"],
                    (
                        f"{ablation_id}/{role}/"
                        f"{segment_name}/focused_summary"
                    ),
                )

            assert segment_output

            role_output[
                segment_name
            ] = segment_output

        projected[
            role
        ] = role_output

    return projected


# ============================================================
# BUILD THE MODEL-FACING PAYLOAD
# ============================================================

def build_semantic_ablation_model_input(
    case,
    ablation_id,
):
    payload = copy.deepcopy(
        build_improved_semantic_model_input(
            case
        )
    )

    payload[
        "semantic_summaries"
    ] = project_semantic_summaries_for_ablation(
        payload[
            "semantic_summaries"
        ],
        ablation_id,
    )

    # The non-semantic representation must remain frozen.
    for role in [
        "participant_A",
        "participant_B",
    ]:
        assert set(
            payload[role]
        ) == {
            "speaks"
        }

    assert set(
        payload[
            "local_temporal_features"
        ]
    ) == set(
        LOCAL_OFFSET_DISTRIBUTION_FIELDS_IMPROVED
    )

    assert set(
        payload[
            "global_shift_features"
        ]
    ) == set(
        GLOBAL_MODEL_FEATURE_FIELDS_IMPROVED
    )

    return payload


# ============================================================
# BUILD AN EVIDENCE INVENTORY MATCHING THE ACTUAL PAYLOAD
# ============================================================

def build_semantic_evidence_inventory_text(
    spec,
):
    lines = []

    if spec["coarse_fields"]:
        lines.extend([
            "COARSE SUMMARY:",
            "",
            *[
                f"- {field}"
                for field in spec[
                    "coarse_fields"
                ]
            ],
        ])

    if (
        spec["coarse_fields"]
        and
        spec["focused_fields"]
    ):
        lines.append("")

    if spec["focused_fields"]:
        lines.extend([
            "FOCUSED SUMMARY:",
            "",
            *[
                f"- {field}"
                for field in spec[
                    "focused_fields"
                ]
            ],
        ])

    lines.append("")

    if (
        spec["coarse_fields"]
        and
        spec["focused_fields"]
    ):
        lines.extend([
            (
                "The supplied coarse and focused fields are semantic "
                "descriptions of the SAME participant segment."
            ),
            "",
            (
                "Interpret all supplied coarse and focused fields together."
            ),
            "",
            (
                "Do not treat differences between one participant's own "
                "coarse and focused summaries as evidence of semantic "
                "incompatibility."
            ),
        ])

    elif spec["focused_fields"]:
        lines.extend([
            (
                "Only the supplied focused semantic fields are available "
                "for each participant segment."
            ),
            "",
            (
                "Interpret the supplied focused fields together as one "
                "semantic description of that participant segment."
            ),
        ])

    else:
        lines.extend([
            (
                "Only the supplied coarse semantic fields are available "
                "for each participant segment."
            ),
            "",
            (
                "Interpret the supplied coarse fields together as one "
                "semantic description of that participant segment."
            ),
        ])

    lines.extend([
        "",
        (
            "The summaries were independently generated and may sometimes "
            "be broad, imperfect, repetitive, incomplete, or uncertain."
        ),
    ])

    return "\n".join(lines).strip()


# ============================================================
# KEEP THE IMPROVED SEMANTIC DECISION POLICY FROZEN
# ============================================================

SEMANTIC_POLICY_CORE_MARKER = (
    "The semantic assessment must be based ONLY on the supplied semantic\n"
    "summaries."
)

assert (
    IMPROVED_SEMANTIC_POLICY_BODY.count(
        SEMANTIC_POLICY_CORE_MARKER
    )
    == 1
)

SEMANTIC_POLICY_CORE = (
    SEMANTIC_POLICY_CORE_MARKER
    + IMPROVED_SEMANTIC_POLICY_BODY.split(
        SEMANTIC_POLICY_CORE_MARKER,
        1,
    )[1]
)


# ============================================================
# UPDATE ONLY THE TOP-LEVEL AVAILABLE-EVIDENCE INVENTORY
# ============================================================

def replace_available_evidence_inventory(
    prompt_prefix,
    spec,
):
    start_marker = "You receive:\n\n"
    end_marker = "\n\nUse only the supplied evidence."

    assert prompt_prefix.count(
        start_marker
    ) == 1

    assert prompt_prefix.count(
        end_marker
    ) == 1

    before, remainder = prompt_prefix.split(
        start_marker,
        1,
    )

    _, after = remainder.split(
        end_marker,
        1,
    )

    evidence_items = [
        "Whether each participant speaks during the 120-second interval.",
        "Local turn-handoff offset-distribution features.",
        "Global temporal alignment-shift features.",
    ]

    if spec["coarse_fields"]:
        evidence_items.append(
            "Coarse semantic summaries for two synchronized 60-second segments."
        )

    if spec["focused_fields"]:
        evidence_items.append(
            "Focused semantic summaries for the same two segments."
        )

    evidence_items.append(
        (
            "Frozen temporal reference statistics calculated only from\n"
            "   separate NORMAL dyadic conversations."
        )
    )

    numbered = "\n".join(
        f"{index}. {item}"
        for index, item in enumerate(
            evidence_items,
            start=1,
        )
    )

    return (
        before
        + start_marker
        + numbered
        + end_marker
        + after
    )


# ============================================================
# BUILD THE COMPLETE PROMPT TEMPLATE FOR ONE ABLATION
# ============================================================

def build_semantic_ablation_prompt_template(
    ablation_id,
):
    assert ablation_id in SEMANTIC_ABLATION_SPECS

    # The S-CF control must remain exactly identical to the
    # already executed improved-semantic baseline.
    if ablation_id == "S-CF":
        return (
            R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE
        )

    spec = SEMANTIC_ABLATION_SPECS[
        ablation_id
    ]

    updated_prefix = (
        replace_available_evidence_inventory(
            PROMPT_BEFORE_SEMANTIC,
            spec,
        )
    )

    semantic_inventory = (
        build_semantic_evidence_inventory_text(
            spec
        )
    )

    semantic_body = (
        "The 120-second interval is divided into:\n\n"
        "- Segment 0: 0 to 60 seconds\n"
        "- Segment 1: 60 to 120 seconds\n\n"
        "For each participant and synchronized segment, you receive:\n\n"
        + semantic_inventory
        + "\n\n"
        + SEMANTIC_POLICY_CORE
    )

    prompt_template = (
        updated_prefix
        + SEMANTIC_SECTION_HEADER
        + "\n\n"
        + semantic_body
        + "\n\n"
        + FINAL_DECISION_SECTION_HEADER
        + PROMPT_AFTER_FINAL_DECISION_HEADER
    ).strip()

    return prompt_template


# ============================================================
# VALIDATE THAT EACH PROMPT MATCHES ITS SEMANTIC PAYLOAD
# ============================================================

def validate_semantic_ablation_prompt(
    ablation_id,
    prompt_template,
):
    assert ablation_id in SEMANTIC_ABLATION_SPECS

    template = validate_manual_improved_semantic_prompt(
        prompt_template
    )

    spec = SEMANTIC_ABLATION_SPECS[
        ablation_id
    ]

    semantic_section = template.split(
        SEMANTIC_SECTION_HEADER,
        1,
    )[1].split(
        FINAL_DECISION_SECTION_HEADER,
        1,
    )[0]

    # Every supplied field must be documented in the semantic section.
    for field in (
        spec["coarse_fields"]
        + spec["focused_fields"]
    ):
        assert field in semantic_section, (
            f"{ablation_id}: supplied field missing from prompt: {field}"
        )

    # Every removed field must be absent from the evidence inventory.
    inventory = semantic_section.split(
        SEMANTIC_POLICY_CORE_MARKER,
        1,
    )[0]

    for field in (
        set(SEMANTIC_ABLATION_ALL_COARSE_FIELDS)
        - set(spec["coarse_fields"])
    ):
        assert field not in inventory, (
            f"{ablation_id}: removed coarse field remains in inventory: {field}"
        )

    for field in (
        set(SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS)
        - set(spec["focused_fields"])
    ):
        assert field not in inventory, (
            f"{ablation_id}: removed focused field remains in inventory: {field}"
        )

    # The policy after the evidence inventory must remain frozen.
    assert (
        semantic_section.split(
            SEMANTIC_POLICY_CORE_MARKER,
            1,
        )[1].strip()
        ==
        IMPROVED_SEMANTIC_POLICY_BODY.split(
            SEMANTIC_POLICY_CORE_MARKER,
            1,
        )[1].strip()
    )

    prompt_lower = template.lower()

    for forbidden_marker in [
        "filtered turns:",
        "{participant_a_turns}",
        "{participant_b_turns}",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "filtered overlap",
    ]:
        assert forbidden_marker not in prompt_lower

    return template


# ============================================================
# RENDER ONE MODEL-FACING PROMPT
# ============================================================

def build_semantic_ablation_prompt(
    case,
    config,
):
    payload = build_semantic_ablation_model_input(
        case,
        config[
            "semantic_ablation_id"
        ],
    )

    prompt = render_manual_improved_semantic_prompt(
        config[
            "reasoning_prompt_template"
        ],
        payload,
    )

    return prompt, payload


# ============================================================
# PREPARE ONE ABLATION CONFIGURATION
# ============================================================

def prepare_semantic_ablation_experiment(
    ablation_id,
):
    assert ablation_id in SEMANTIC_ABLATION_SPECS

    spec = copy.deepcopy(
        SEMANTIC_ABLATION_SPECS[
            ablation_id
        ]
    )

    prompt_template = (
        build_semantic_ablation_prompt_template(
            ablation_id
        )
    )

    prompt_template = (
        validate_semantic_ablation_prompt(
            ablation_id,
            prompt_template,
        )
    )

    slug = (
        ablation_id
        .lower()
        .replace("-", "_")
    )

    experiment_version = (
        "structured_r1_improved_semantic_payload_ablation_"
        + slug
    )

    experiment_dir = (
        SEMANTIC_ABLATION_ROOT
        / slug
    )

    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    paths = {
        "prediction_cache": experiment_dir / "predictions_cache.json",
        "predictions_csv": experiment_dir / "predictions_all_68.csv",
        "metrics_json": experiment_dir / "metrics.json",
        "confusion_matrix_csv": experiment_dir / "confusion_matrix.csv",
        "family_metrics_csv": experiment_dir / "metrics_by_case_family.csv",
        "variant_metrics_csv": experiment_dir / "metrics_by_case_variant.csv",
        "errors_csv": experiment_dir / "classification_errors.csv",
        "reasoning_assessments_csv": experiment_dir / "reasoning_assessments.csv",
        "reasoning_inconsistencies_csv": experiment_dir / "reasoning_inconsistencies.csv",
        "assessment_distributions_csv": experiment_dir / "assessment_distributions.csv",
        "prompt_template": experiment_dir / "prompt_template.txt",
        "prompt_diff": experiment_dir / "prompt_diff_vs_S_CF.txt",
        "rendered_prompt_preview": experiment_dir / "rendered_prompt_preview.txt",
        "experiment_manifest": experiment_dir / "experiment_manifest.json",
    }

    reasoning_prompt_sha256 = sha256_text(
        prompt_template
    )

    baseline_prompt_sha256 = sha256_text(
        R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE
    )

    normal_reference_sha256 = sha256_text(
        normal_base_reference_text
        + "\n"
        + normal_global_reference_text
    )

    prompt_diff = "\n".join(
        difflib.unified_diff(
            R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE.splitlines(),
            prompt_template.splitlines(),
            fromfile="S-CF_full_semantic_payload",
            tofile=f"{ablation_id}_{spec['title']}",
            lineterm="",
        )
    )

    paths[
        "prompt_template"
    ].write_text(
        prompt_template,
        encoding="utf-8",
    )

    paths[
        "prompt_diff"
    ].write_text(
        prompt_diff,
        encoding="utf-8",
    )

    manifest = {
        "experiment_version": experiment_version,
        "experiment_title": (
            "Structured R1 — Improved Semantic Policy — "
            f"Semantic Payload Ablation {ablation_id}: {spec['title']}"
        ),
        "ablation_id": (
            f"R1_IMPROVED_SEMANTIC_PAYLOAD_{ablation_id}"
        ),
        "semantic_ablation_id": ablation_id,
        "semantic_ablation_round": spec[
            "round"
        ],
        "semantic_input": ablation_id,
        "semantic_payload_title": spec[
            "title"
        ],
        "semantic_coarse_fields": list(
            spec["coarse_fields"]
        ),
        "semantic_focused_fields": list(
            spec["focused_fields"]
        ),
        "semantic_removed_fields": list(
            spec["removed_fields"]
        ),
        "focused_summaries_used": bool(
            spec["focused_fields"]
        ),
        "source_experiment": (
            IMPROVED_SEMANTIC_EXPERIMENT_VERSION
        ),
        "model_id": MODEL_ID,
        "max_new_tokens": MAX_NEW_TOKENS_REASONING,
        "reasoning_prompt_sha256": reasoning_prompt_sha256,
        "baseline_prompt_sha256": baseline_prompt_sha256,
        "normal_reference_sha256": normal_reference_sha256,
        "filtered_turns_supplied": False,
        "overlap_supplied": False,
        "participant_model_input": "speaks_only",
        "local_model_input": "complete_offset_distribution_only",
        "global_model_input": "all_five_global_features",
        "temporal_profiles_used": "frozen_NORMAL_only",
        "assessment_policy": (
            "structured_R1_with_frozen_improved_semantic_policy"
        ),
        "schema_keys": list(
            REASONING_SCHEMA_KEYS
        ),
        "prompt_definition_method": (
            "frozen_policy_with_payload_specific_manual_inspection"
        ),
        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },
    }

    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    config = {
        **manifest,
        "experiment_dir": experiment_dir,
        "reasoning_prompt_template": prompt_template,
        "paths": paths,
    }

    SEMANTIC_ABLATION_CONFIGS[
        ablation_id
    ] = config

    print("=" * 100)
    print(
        f"SEMANTIC ABLATION {ablation_id} — CONFIGURATION READY"
    )
    print("=" * 100)
    print("Title:", spec["title"])
    print("Coarse fields:", spec["coarse_fields"])
    print("Focused fields:", spec["focused_fields"])
    print("Removed fields:", spec["removed_fields"])
    print("Prompt SHA256:", reasoning_prompt_sha256)
    print("Output directory:", experiment_dir)
    print(
        "Existing cache:",
        paths["prediction_cache"].exists(),
    )

    return config


# ============================================================
# REQUIRED EXACT PROMPT INSPECTION
# ============================================================

def inspect_semantic_ablation_prompt(
    config,
    *,
    case_index=0,
    prefer_wrong_partner=True,
):
    ablation_id = config[
        "semantic_ablation_id"
    ]

    if prefer_wrong_partner:
        candidate_cases = sorted(
            [
                case
                for case in consolidation_cases
                if get_case_family(case)
                == "wrong_partner"
            ],
            key=lambda case: str(
                case["case_id"]
            ),
        )
    else:
        candidate_cases = sorted(
            consolidation_cases,
            key=lambda case: str(
                case["case_id"]
            ),
        )

    assert (
        0
        <= case_index
        < len(candidate_cases)
    )

    case = candidate_cases[
        case_index
    ]

    prompt, payload = (
        build_semantic_ablation_prompt(
            case,
            config,
        )
    )

    config[
        "paths"
    ][
        "rendered_prompt_preview"
    ].write_text(
        prompt,
        encoding="utf-8",
    )

    print("=" * 100)
    print(
        f"SEMANTIC ABLATION {ablation_id} — EXACT PROMPT INSPECTION"
    )
    print("=" * 100)
    print("Inspection case ID:", case["case_id"])
    print(
        "Gold label is intentionally NOT printed inside the model prompt."
    )
    print("Participant evidence:", "speaks only")
    print("Local evidence:", "complete offset distribution")
    print("Global evidence:", "all five global features")
    print(
        "Coarse fields supplied:",
        config["semantic_coarse_fields"],
    )
    print(
        "Focused fields supplied:",
        config["semantic_focused_fields"],
    )
    print(
        "Removed semantic fields:",
        config["semantic_removed_fields"],
    )
    print("Filtered turns supplied:", False)
    print("Overlap supplied:", False)

    print("\n" + "=" * 100)
    print("EXACT MODEL-FACING PAYLOAD")
    print("=" * 100)
    print(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
    )

    print("\n" + "=" * 100)
    print("EXACT RENDERED MODEL PROMPT")
    print("=" * 100)
    print(prompt)

    SEMANTIC_ABLATION_INSPECTED_EXPERIMENTS.add(
        config["experiment_version"]
    )

    print(
        "\nInspection unlocked:",
        config["experiment_version"]
        in SEMANTIC_ABLATION_INSPECTED_EXPERIMENTS,
    )
    print(
        "Saved rendered prompt:",
        config[
            "paths"
        ][
            "rendered_prompt_preview"
        ],
    )

    return {
        "case": case,
        "payload": payload,
        "prompt": prompt,
    }


# ============================================================
# CACHE
# ============================================================

def create_semantic_ablation_cache(
    config,
):
    return {
        "experiment_version": config[
            "experiment_version"
        ],
        "experiment_title": config[
            "experiment_title"
        ],
        "ablation_id": config[
            "ablation_id"
        ],
        "semantic_ablation_id": config[
            "semantic_ablation_id"
        ],
        "source_experiment": config[
            "source_experiment"
        ],
        "model_id": config[
            "model_id"
        ],
        "reasoning_prompt_sha256": config[
            "reasoning_prompt_sha256"
        ],
        "baseline_prompt_sha256": config[
            "baseline_prompt_sha256"
        ],
        "normal_reference_sha256": config[
            "normal_reference_sha256"
        ],
        "semantic_input": config[
            "semantic_input"
        ],
        "semantic_coarse_fields": list(
            config["semantic_coarse_fields"]
        ),
        "semantic_focused_fields": list(
            config["semantic_focused_fields"]
        ),
        "semantic_removed_fields": list(
            config["semantic_removed_fields"]
        ),
        "focused_summaries_used": bool(
            config["focused_summaries_used"]
        ),
        "filtered_turns_supplied": False,
        "overlap_supplied": False,
        "temporal_profiles_used": config[
            "temporal_profiles_used"
        ],
        "assessment_policy": config[
            "assessment_policy"
        ],
        "schema_keys": list(
            REASONING_SCHEMA_KEYS
        ),
        "max_new_tokens": MAX_NEW_TOKENS_REASONING,
        "created_at_utc": reasoning_utc_now(),
        "updated_at_utc": reasoning_utc_now(),
        "records": {},
    }


# ============================================================
# RUN ONE EXPERIMENT — MANUAL CALL ONLY
# ============================================================

def run_semantic_ablation_experiment(
    config,
    *,
    print_each_case_prompt=False,
):
    assert (
        config["experiment_version"]
        in SEMANTIC_ABLATION_INSPECTED_EXPERIMENTS
    ), (
        "Run the exact prompt-inspection cell for this ablation before inference."
    )

    prediction_cache_path = config[
        "paths"
    ][
        "prediction_cache"
    ]

    expected_cache = create_semantic_ablation_cache(
        config
    )

    if prediction_cache_path.exists():
        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )

        for key in [
            "experiment_version",
            "ablation_id",
            "semantic_ablation_id",
            "source_experiment",
            "model_id",
            "reasoning_prompt_sha256",
            "baseline_prompt_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "semantic_coarse_fields",
            "semantic_focused_fields",
            "semantic_removed_fields",
            "focused_summaries_used",
            "filtered_turns_supplied",
            "overlap_supplied",
            "temporal_profiles_used",
            "assessment_policy",
            "schema_keys",
            "max_new_tokens",
        ]:
            assert (
                prediction_cache[key]
                == expected_cache[key]
            ), (
                f"Cache mismatch for {key}. "
                "Use the experiment's own output directory or remove the incompatible cache."
            )

        print("Resuming cache:", prediction_cache_path)
        print(
            "Existing records:",
            len(prediction_cache["records"]),
        )

    else:
        prediction_cache = expected_cache

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

        print("Created cache:", prediction_cache_path)

    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case["case_id"]
        ),
    )

    assert len(ordered_cases) == 68

    for case in tqdm(
        ordered_cases,
        desc=config["experiment_version"],
    ):
        case_id = str(
            case["case_id"]
        )

        prompt, payload = (
            build_semantic_ablation_prompt(
                case,
                config,
            )
        )

        if print_each_case_prompt:
            print("\n" + "=" * 100)
            print("CASE:", case_id)
            print("=" * 100)
            print(prompt)

        prompt_sha256 = sha256_text(
            prompt
        )

        payload_sha256 = sha256_text(
            canonical_json(
                payload
            )
        )

        existing_record = (
            prediction_cache[
                "records"
            ].get(case_id)
        )

        if (
            existing_record is not None
            and existing_record.get(
                "prediction"
            ) in LABELS
        ):
            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )

            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == payload_sha256
            )

            continue

        started = time.perf_counter()

        try:
            raw_output, input_token_count = (
                qwen_text_only_binary(
                    prompt,
                    max_new_tokens=(
                        MAX_NEW_TOKENS_REASONING
                    ),
                )
            )

            parsed_result = (
                parse_structured_reasoning_prediction(
                    raw_output
                )
            )

            generation_error = None

        except Exception as exc:
            raw_output = ""
            input_token_count = None

            parsed_result = {
                "prediction": None,
                "parse_mode": "generation_error",
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
                "participation_assessment": None,
                "local_temporal_assessment": None,
                "global_temporal_assessment": None,
                "temporal_assessment": None,
                "semantic_assessment": None,
                "decisive_dimension": None,
            }

            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        elapsed_seconds = (
            time.perf_counter()
            - started
        )

        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": case_id,
            "source_group_id": str(
                case["source_group_id"]
            ),
            "case_family": get_case_family(
                case
            ),
            "case_variant": str(
                case["case_variant"]
            ),
            "gold_binary_label": str(
                case["gold_binary_label"]
            ).upper(),
            "semantic_ablation_id": config[
                "semantic_ablation_id"
            ],
            "semantic_coarse_fields": list(
                config["semantic_coarse_fields"]
            ),
            "semantic_focused_fields": list(
                config["semantic_focused_fields"]
            ),
            "prompt_sha256": prompt_sha256,
            "input_payload_sha256": payload_sha256,
            "semantic_input": config[
                "semantic_input"
            ],
            "focused_summaries_used": bool(
                config["focused_summaries_used"]
            ),
            "input_token_count": input_token_count,
            "max_new_tokens": MAX_NEW_TOKENS_REASONING,
            "raw_output": raw_output,
            "prediction": parsed_result[
                "prediction"
            ],
            "participation_assessment": parsed_result[
                "participation_assessment"
            ],
            "local_temporal_assessment": parsed_result[
                "local_temporal_assessment"
            ],
            "global_temporal_assessment": parsed_result[
                "global_temporal_assessment"
            ],
            "temporal_assessment": parsed_result[
                "temporal_assessment"
            ],
            "semantic_assessment": parsed_result[
                "semantic_assessment"
            ],
            "decisive_dimension": parsed_result[
                "decisive_dimension"
            ],
            "parse_mode": parsed_result[
                "parse_mode"
            ],
            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),
            "schema_errors": parsed_result[
                "schema_errors"
            ],
            "parsed_output": parsed_result[
                "parsed_output"
            ],
            "generation_error": generation_error,
            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),
            "completed_at_utc": reasoning_utc_now(),
        }

        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

    print("\n" + "=" * 100)
    print(
        f"SEMANTIC ABLATION {config['semantic_ablation_id']} — INFERENCE COMPLETE"
    )
    print("=" * 100)
    print(
        "Cached records:",
        len(prediction_cache["records"]),
    )
    print("Prediction cache:", prediction_cache_path)

    return prediction_cache


# ============================================================
# REGISTER AN EVALUATION FOR FINAL COMPARISON
# ============================================================

def register_semantic_ablation_evaluation(
    ablation_id,
    evaluation,
):
    assert ablation_id in SEMANTIC_ABLATION_SPECS

    SEMANTIC_ABLATION_EVALUATIONS[
        ablation_id
    ] = evaluation

    print(
        "Registered evaluation:",
        ablation_id,
    )

    return evaluation


print("Semantic ablation framework ready.")
print(
    "Experiments:",
    list(SEMANTIC_ABLATION_SPECS),
)
print(
    "Run each prompt-inspection cell manually before its inference cell."
)


Semantic ablation framework ready.
Experiments: ['S-CF', 'S-F', 'S-C', 'F1', 'F2', 'F3', 'F4']
Run each prompt-inspection cell manually before its inference cell.


## 10. Prepare F1 and verify its prompt/reference hashes against the original experiment

In [ ]:

# ============================================================
# PREPARE THE EXACT F1 PROMPT AND VERIFY IT AGAINST THE
# ORIGINAL PRE-FINE-TUNING F1 EXPERIMENT
# ============================================================

F1_CONFIG = prepare_semantic_ablation_experiment(
    "F1"
)

F1_PROMPT_INSPECTION = inspect_semantic_ablation_prompt(
    F1_CONFIG,
    case_index=0,
    prefer_wrong_partner=True,
)

ORIGINAL_F1_CACHE = json.loads(
    ORIGINAL_F1_CACHE_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    F1_CONFIG[
        "reasoning_prompt_sha256"
    ]
    ==
    ORIGINAL_F1_CACHE[
        "reasoning_prompt_sha256"
    ]
), (
    "The reconstructed F1 prompt does not match the exact "
    "original frozen F1 prompt."
)

assert (
    F1_CONFIG[
        "normal_reference_sha256"
    ]
    ==
    ORIGINAL_F1_CACHE[
        "normal_reference_sha256"
    ]
), (
    "The frozen NORMAL reference text does not match the "
    "original F1 experiment."
)

assert (
    F1_CONFIG[
        "semantic_coarse_fields"
    ]
    ==
    ORIGINAL_F1_CACHE[
        "semantic_coarse_fields"
    ]
)

assert (
    F1_CONFIG[
        "semantic_focused_fields"
    ]
    ==
    ORIGINAL_F1_CACHE[
        "semantic_focused_fields"
    ]
)

assert (
    F1_CONFIG[
        "semantic_removed_fields"
    ]
    ==
    ORIGINAL_F1_CACHE[
        "semantic_removed_fields"
    ]
)

print("=" * 100)
print("EXACT ORIGINAL F1 PROMPT/REFERENCE AUDIT PASSED")
print("=" * 100)
print(
    "F1 prompt SHA256:",
    F1_CONFIG["reasoning_prompt_sha256"],
)
print(
    "Frozen NORMAL reference SHA256:",
    F1_CONFIG["normal_reference_sha256"],
)
print(
    "New frozen-F1 output directory:",
    F1_CONFIG["experiment_dir"],
)


SEMANTIC ABLATION F1 — CONFIGURATION READY
Title: Without detailed_speech_summary
Coarse fields: ['speech_content_summary', 'apparent_topic']
Focused fields: ['main_topic', 'secondary_topics', 'key_semantic_details', 'summary_specificity', 'unclear_content', 'confidence']
Removed fields: ['focused_summary/detailed_speech_summary']
Prompt SHA256: bf50d017f027995c6eaae81adad6b04d263530d25f2f4361e3ee3d9f19791355
Output directory: /content/drive/MyDrive/final_test_completely_unseen_database/f1_frozen_vs_finetuned_final_unseen_evaluation/frozen_f1/structured_r1_improved_semantic_payload_ablation/f1
Existing cache: True
SEMANTIC ABLATION F1 — EXACT PROMPT INSPECTION
Inspection case ID: consolidation_wrong_partner_000
Gold label is intentionally NOT printed inside the model prompt.
Participant evidence: speaks only
Local evidence: complete offset distribution
Global evidence: all five global features
Coarse fields supplied: ['speech_content_summary', 'apparent_topic']
Focused fields supplied:

## 11. Load the frozen pre-fine-tuning F1 model

In [ ]:

# ============================================================
# LOAD THE FROZEN PRE-FINE-TUNING F1 MODEL
#
# Same base model, processor, 4-bit NF4 quantisation and
# deterministic generation settings used by the saved adapter
# evaluation. No LoRA adapter is attached in this section.
# ============================================================

import gc
import torch

from transformers import (
    BitsAndBytesConfig,
    Qwen2_5OmniProcessor,
    Qwen2_5OmniThinkerForConditionalGeneration,
)

assert torch.cuda.is_available(), (
    "A CUDA GPU runtime is required."
)

COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

USE_4BIT = True

MODEL_SOURCE = (
    str(MODEL_PATH)
    if MODEL_PATH.exists()
    else MODEL_ID
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

processor = Qwen2_5OmniProcessor.from_pretrained(
    MODEL_SOURCE
)

tokenizer = processor.tokenizer

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = (
    Qwen2_5OmniThinkerForConditionalGeneration
    .from_pretrained(
        MODEL_SOURCE,
        torch_dtype=COMPUTE_DTYPE,
        quantization_config=quantization_config,
        device_map={"": 0},
        low_cpu_mem_usage=True,
    )
)

model.eval()
model.config.use_cache = True

print("=" * 100)
print("FROZEN F1 BASE MODEL LOADED")
print("=" * 100)
print("Model source:", MODEL_SOURCE)
print("Model class:", type(model).__name__)
print("Compute dtype:", COMPUTE_DTYPE)
print("4-bit NF4:", USE_4BIT)
print("LoRA adapter attached:", False)


[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniThinkerForConditionalGeneration LOAD REPORT from: /content/drive/MyDrive/Qwen2.5-Omni-7B
Key                                                                                                      | Status     |  | 
---------------------------------------------------------------------------------------------------------+------------+--+-
token2wav.code2wav_bigvgan_model.resblocks.{0...17}.activations.{0, 1, 2, 3, 4, 5}.act.alpha             | UNEXPECTED |  | 
token2wav.code2wav_bigvgan_model.resblocks.{0...17}.convs2.{0, 1, 2}.bias                                | UNEXPECTED |  | 
talker.model.layers.{0...23}.mlp.up_proj.weight                                                          | UNEXPECTED |  | 
token2wav.code2wav_bigvgan_model.ups.{0, 1, 2, 3, 4, 5}.0.bias                                           | UNEXPECTED |  | 
talker.model.layers.{0...23}.self_attn.k_proj.weight                                                     | UNEXPECTED |  | 
talker.model.laye

FROZEN F1 BASE MODEL LOADED
Model source: /content/drive/MyDrive/Qwen2.5-Omni-7B
Model class: Qwen2_5OmniThinkerForConditionalGeneration
Compute dtype: torch.bfloat16
4-bit NF4: True
LoRA adapter attached: False


## 12. Run frozen F1 on all 68 final-unseen cases

In [ ]:

# ============================================================
# RUN FROZEN F1 ON ALL 68 FINAL-UNSEEN CASES
#
# Uses the exact original F1:
# - prompt
# - semantic projection
# - frozen NORMAL references
# - deterministic decoding
# - parser
# - structured output schema
# - cache/resume logic
# ============================================================

FROZEN_F1_CACHE = run_semantic_ablation_experiment(
    F1_CONFIG,
    print_each_case_prompt=False,
)

assert len(
    FROZEN_F1_CACHE["records"]
) == 68

assert set(
    FROZEN_F1_CACHE["records"].keys()
) == {
    str(case["case_id"])
    for case in consolidation_cases
}

FROZEN_F1_EVALUATION = evaluate_reasoning_experiment(
    F1_CONFIG
)

print("=" * 100)
print("FROZEN F1 FINAL-UNSEEN EVALUATION COMPLETE")
print("=" * 100)
print(
    "Cache:",
    F1_CONFIG["paths"]["prediction_cache"],
)


Created cache: /content/drive/MyDrive/final_test_completely_unseen_database/f1_frozen_vs_finetuned_final_unseen_evaluation/frozen_f1/structured_r1_improved_semantic_payload_ablation/f1/predictions_cache.json


structured_r1_improved_semantic_payload_ablation_f1:   0%|          | 0/68 [00:00<?, ?it/s]


SEMANTIC ABLATION F1 — INFERENCE COMPLETE
Cached records: 68
Prediction cache: /content/drive/MyDrive/final_test_completely_unseen_database/f1_frozen_vs_finetuned_final_unseen_evaluation/frozen_f1/structured_r1_improved_semantic_payload_ablation/f1/predictions_cache.json
Structured R1 — Improved Semantic Policy — Semantic Payload Ablation F1: Without detailed_speech_summary — RESULTS
Total cases: 68
Valid predictions: 68
Invalid predictions: 0
Exact structured-schema rate: 1.0000
Accuracy: 0.8235
Balanced accuracy: 0.7647
ANOMALOUS precision: 0.8824
ANOMALOUS recall: 0.8824
ANOMALOUS F1: 0.8824
NORMAL recall / specificity: 0.6471
MCC: 0.5294
Matched source-group exact rate: 0.4118
Reasoning inconsistencies: 0

CONFUSION MATRIX


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,11,6
Gold ANOMALOUS,6,45



CLASSIFICATION REPORT


...


PER EXACT CASE VARIANT


,case_variant,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate
0,lag_2sec,8,8,0,1,7,7,0.875000,0.875000,1.0
1,lag_3sec,9,9,0,4,5,5,0.555556,0.555556,1.0
2,normal,17,17,0,11,6,11,0.647059,0.647059,1.0
3,silent_partner,17,17,0,0,17,17,1.000000,1.000000,1.0
4,wrong_partner,17,17,0,1,16,16,0.941176,0.941176,1.0



ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count
0,participation_assessment,VALID,51
1,participation_assessment,INVALID,17
2,local_temporal_assessment,ANOMALOUS,43
3,local_temporal_assessment,NORMAL,24
4,local_temporal_assessment,LIMITED,1
5,global_temporal_assessment,ANOMALOUS,43
6,global_temporal_assessment,NORMAL,24
7,global_temporal_assessment,LIMITED,1
8,temporal_assessment,ANOMALOUS,43
9,temporal_assessment,NORMAL,24



Saved cache: /content/drive/MyDrive/final_test_completely_unseen_database/f1_frozen_vs_finetuned_final_unseen_evaluation/frozen_f1/structured_r1_improved_semantic_payload_ablation/f1/predictions_cache.json
Saved prompt: /content/drive/MyDrive/final_test_completely_unseen_database/f1_frozen_vs_finetuned_final_unseen_evaluation/frozen_f1/structured_r1_improved_semantic_payload_ablation/f1/prompt_template.txt
Saved prompt diff: /content/drive/MyDrive/final_test_completely_unseen_database/f1_frozen_vs_finetuned_final_unseen_evaluation/frozen_f1/structured_r1_improved_semantic_payload_ablation/f1/prompt_diff_vs_S_CF.txt
Saved predictions: /content/drive/MyDrive/final_test_completely_unseen_database/f1_frozen_vs_finetuned_final_unseen_evaluation/frozen_f1/structured_r1_improved_semantic_payload_ablation/f1/predictions_all_68.csv
Saved reasoning assessments: /content/drive/MyDrive/final_test_completely_unseen_database/f1_frozen_vs_finetuned_final_unseen_evaluation/frozen_f1/structured_r1_imp

In [ ]:
import pandas as pd
from IPython.display import display


# ============================================================
# LOAD FROZEN F1 — FINAL UNSEEN TEST RESULTS
# ============================================================

if "FROZEN_F1_EVALUATION" in globals():

    frozen_f1_df = (
        FROZEN_F1_EVALUATION[
            "results_df"
        ].copy()
    )

elif "F1_CONFIG" in globals():

    predictions_path = (
        F1_CONFIG[
            "paths"
        ][
            "predictions_csv"
        ]
    )

    frozen_f1_df = pd.read_csv(
        predictions_path
    )

else:

    raise RuntimeError(
        "Run the Frozen F1 configuration, inference "
        "and evaluation cells first."
    )


# ============================================================
# STRICT BASIC CHECKS
# ============================================================

assert len(
    frozen_f1_df
) == 68, (
    "Expected exactly 68 final-unseen Frozen F1 results, "
    f"but found {len(frozen_f1_df)}."
)


assert "case_id" in frozen_f1_df.columns
assert frozen_f1_df["case_id"].is_unique


# ============================================================
# NORMALIZE TEXT FIELDS
# ============================================================

upper_columns = [
    "gold_label",
    "prediction",
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]


for column in upper_columns:

    if column in frozen_f1_df.columns:

        frozen_f1_df[column] = (
            frozen_f1_df[column]
            .astype("string")
            .str.strip()
            .str.upper()
        )


for column in [
    "case_family",
    "case_variant",
]:

    if column in frozen_f1_df.columns:

        frozen_f1_df[column] = (
            frozen_f1_df[column]
            .astype("string")
            .str.strip()
            .str.lower()
            .str.replace(
                r"[\s\-]+",
                "_",
                regex=True,
            )
        )


frozen_f1_df["case_id"] = (
    frozen_f1_df["case_id"]
    .astype(str)
    .str.strip()
)


# ============================================================
# VALID PREDICTION AND CORRECTNESS
# ============================================================

if "valid_prediction" not in frozen_f1_df.columns:

    frozen_f1_df["valid_prediction"] = (
        frozen_f1_df["prediction"].isin([
            "NORMAL",
            "ANOMALOUS",
        ])
    )


if "correct" not in frozen_f1_df.columns:

    frozen_f1_df["correct"] = (
        frozen_f1_df["valid_prediction"]
        &
        (
            frozen_f1_df["gold_label"]
            ==
            frozen_f1_df["prediction"]
        )
    )


frozen_f1_df["correct"] = (
    frozen_f1_df["correct"]
    .fillna(False)
    .astype(bool)
)


# ============================================================
# EXPECTED STRUCTURED OUTPUT VALUES
# ============================================================

FROZEN_F1_ASSESSMENT_LEVELS = {

    "participation_assessment": [
        "VALID",
        "INVALID",
    ],

    "local_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "global_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "semantic_assessment": [
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    ],

    "decisive_dimension": [
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    ],
}


# ============================================================
# FREQUENCY TABLE
# ============================================================

def frozen_f1_frequency_table(
    subset,
    field,
    expected_values,
):

    if field not in subset.columns:

        return pd.DataFrame({
            "assessment_field": [field],
            "assessment_value": [
                "COLUMN_NOT_AVAILABLE"
            ],
            "count": [0],
            "percentage": [0.0],
        })


    values = (
        subset[field]
        .fillna("MISSING")
        .astype(str)
        .str.strip()
        .str.upper()
    )


    ordered_values = list(
        dict.fromkeys(
            list(expected_values)
            +
            ["MISSING"]
        )
    )


    unexpected_values = [
        value
        for value in values.unique().tolist()
        if value not in ordered_values
    ]


    ordered_values.extend(
        sorted(
            unexpected_values
        )
    )


    counts = (
        values
        .value_counts(
            dropna=False
        )
        .reindex(
            ordered_values,
            fill_value=0,
        )
    )


    table = pd.DataFrame({
        "assessment_field": field,
        "assessment_value": counts.index,
        "count": counts.values,
    })


    if len(subset) > 0:

        table["percentage"] = (
            100.0
            *
            table["count"]
            /
            len(subset)
        ).round(2)

    else:

        table["percentage"] = 0.0


    return table


# ============================================================
# SAFE CROSSTAB
# ============================================================

def display_frozen_f1_crosstab(
    subset,
    row_field,
    column_field,
    title,
):

    print(
        f"\n{title}"
    )


    if (
        row_field not in subset.columns
        or
        column_field not in subset.columns
    ):

        print(
            "Required columns are unavailable."
        )

        return


    display(
        pd.crosstab(
            subset[
                row_field
            ].fillna("MISSING"),

            subset[
                column_field
            ].fillna("MISSING"),

            margins=True,
        )
    )


# ============================================================
# SHOW INDIVIDUAL CASES
# ============================================================

def display_frozen_f1_individual_cases(
    subset,
):

    preferred_columns = [
        "case_id",
        "source_group_id",
        "case_family",
        "case_variant",
        "gold_label",
        "prediction",
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
        "schema_exact",
        "parse_mode",
        "input_token_count",
        "elapsed_seconds",
        "valid_prediction",
        "correct",
    ]


    available_columns = [
        column
        for column in preferred_columns
        if column in subset.columns
    ]


    if not available_columns:

        print(
            "No case-level display columns are available."
        )

        return


    sort_columns = [
        column
        for column in [
            "case_variant",
            "case_id",
        ]
        if column in available_columns
    ]


    print(
        "\nINDIVIDUAL CASES"
    )


    case_table = (
        subset[
            available_columns
        ]
        .sort_values(
            by=sort_columns
        )
        .reset_index(
            drop=True
        )
    )


    display(
        case_table
    )


# ============================================================
# DISPLAY RAW OUTPUT FOR A SPECIFIC CASE
# ============================================================

def inspect_frozen_f1_case(
    case_id,
):

    case_id = str(
        case_id
    ).strip()


    selected = frozen_f1_df[
        frozen_f1_df[
            "case_id"
        ]
        ==
        case_id
    ].copy()


    if selected.empty:

        raise KeyError(
            f"Case ID not found: {case_id}"
        )


    assert len(
        selected
    ) == 1


    row = selected.iloc[0]


    print(
        "\n"
        +
        "=" * 100
    )

    print(
        "FROZEN F1 INDIVIDUAL CASE:",
        case_id,
    )

    print(
        "=" * 100
    )


    fields = [
        "case_family",
        "case_variant",
        "gold_label",
        "prediction",
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
        "schema_exact",
        "parse_mode",
        "valid_prediction",
        "correct",
        "input_token_count",
        "elapsed_seconds",
        "generation_error",
    ]


    for field in fields:

        if field in selected.columns:

            print(
                f"{field}:",
                row[field],
            )


    if "schema_errors" in selected.columns:

        print(
            "\nschema_errors:"
        )

        print(
            row[
                "schema_errors"
            ]
        )


    if "raw_output" in selected.columns:

        print(
            "\nRAW MODEL OUTPUT"
        )

        print(
            "-" * 100
        )

        print(
            row[
                "raw_output"
            ]
        )


    return selected


# ============================================================
# COMPLETE SUBSET INSPECTION
# ============================================================

def inspect_frozen_f1_subset(
    subset,
    title,
    show_individual_cases=True,
):

    subset = subset.copy()


    print(
        "\n"
        +
        "=" * 100
    )

    print(
        title
    )

    print(
        "=" * 100
    )


    if subset.empty:

        print(
            "No cases found in this subset."
        )

        return subset


    overview = pd.DataFrame([{

        "cases": len(
            subset
        ),

        "gold_NORMAL": int(
            (
                subset[
                    "gold_label"
                ]
                ==
                "NORMAL"
            ).sum()
        ),

        "gold_ANOMALOUS": int(
            (
                subset[
                    "gold_label"
                ]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "pred_NORMAL": int(
            (
                subset[
                    "prediction"
                ]
                ==
                "NORMAL"
            ).sum()
        ),

        "pred_ANOMALOUS": int(
            (
                subset[
                    "prediction"
                ]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "invalid_predictions": int(
            (
                ~subset[
                    "valid_prediction"
                ]
            ).sum()
        ),

        "correct": int(
            subset[
                "correct"
            ].sum()
        ),

        "accuracy_percent": round(
            100.0
            *
            subset[
                "correct"
            ].mean(),
            2,
        ),
    }])


    print(
        "\nSUBSET OVERVIEW"
    )

    display(
        overview
    )


    if "case_variant" in subset.columns:

        print(
            "\nCASE VARIANT BREAKDOWN"
        )


        variant_counts = (
            subset[
                "case_variant"
            ]
            .fillna("MISSING")
            .value_counts()
            .rename_axis(
                "case_variant"
            )
            .reset_index(
                name="count"
            )
        )


        variant_counts[
            "percentage"
        ] = (
            100.0
            *
            variant_counts[
                "count"
            ]
            /
            len(subset)
        ).round(2)


        display(
            variant_counts
        )


    print(
        "\nASSESSMENT BREAKDOWN"
    )


    assessment_table = pd.concat(
        [
            frozen_f1_frequency_table(
                subset=subset,
                field=field,
                expected_values=expected_values,
            )

            for (
                field,
                expected_values,
            )
            in FROZEN_F1_ASSESSMENT_LEVELS.items()
        ],
        ignore_index=True,
    )


    display(
        assessment_table
    )


    display_frozen_f1_crosstab(
        subset,
        "participation_assessment",
        "decisive_dimension",
        (
            "PARTICIPATION ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_frozen_f1_crosstab(
        subset,
        "local_temporal_assessment",
        "global_temporal_assessment",
        (
            "LOCAL × GLOBAL "
            "TEMPORAL ASSESSMENT"
        ),
    )


    display_frozen_f1_crosstab(
        subset,
        "temporal_assessment",
        "semantic_assessment",
        (
            "TEMPORAL ASSESSMENT "
            "× SEMANTIC ASSESSMENT"
        ),
    )


    display_frozen_f1_crosstab(
        subset,
        "semantic_assessment",
        "decisive_dimension",
        (
            "SEMANTIC ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_frozen_f1_crosstab(
        subset,
        "temporal_assessment",
        "decisive_dimension",
        (
            "TEMPORAL ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    if show_individual_cases:

        display_frozen_f1_individual_cases(
            subset
        )


    return subset


# ============================================================
# FAMILY MASK
# ============================================================

def get_frozen_f1_family_mask(
    dataframe,
    family,
):

    values = (
        dataframe[
            "case_family"
        ]
        .fillna("")
        .astype(str)
        .str.lower()
    )


    if family == "normal":

        return values.isin([
            "normal",
            "normals",
        ])


    if family == "lag":

        return (
            values.isin([
                "lag",
                "lags",
            ])
            |
            values.str.contains(
                r"(?:^|_)lag(?:$|_)",
                regex=True,
            )
        )


    if family == "wrong_partner":

        return (
            values.isin([
                "wrong_partner",
                "wrong_partners",
            ])
            |
            (
                values.str.contains(
                    "wrong",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    if family == "silent_partner":

        return (
            values.isin([
                "silent_partner",
                "silent_partners",
            ])
            |
            (
                values.str.contains(
                    "silent",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    raise ValueError(
        f"Unknown family: {family}"
    )


# ============================================================
# CREATE CORRECT / MISSED SUBSET
# ============================================================

def get_frozen_f1_outcome_subset(
    family,
    outcome,
):

    family_mask = (
        get_frozen_f1_family_mask(
            dataframe=frozen_f1_df,
            family=family,
        )
    )


    if outcome == "invalid":

        return frozen_f1_df[
            family_mask
            &
            (
                ~frozen_f1_df[
                    "valid_prediction"
                ]
            )
        ].copy()


    if family == "normal":

        gold_label = "NORMAL"

        if outcome == "correct":

            prediction = "NORMAL"

        elif outcome == "missed":

            prediction = "ANOMALOUS"

        else:

            raise ValueError(
                "outcome must be 'correct', "
                "'missed' or 'invalid'."
            )

    else:

        gold_label = "ANOMALOUS"

        if outcome == "correct":

            prediction = "ANOMALOUS"

        elif outcome == "missed":

            prediction = "NORMAL"

        else:

            raise ValueError(
                "outcome must be 'correct', "
                "'missed' or 'invalid'."
            )


    return frozen_f1_df[
        family_mask
        &
        (
            frozen_f1_df[
                "gold_label"
            ]
            ==
            gold_label
        )
        &
        (
            frozen_f1_df[
                "prediction"
            ]
            ==
            prediction
        )
    ].copy()


# ============================================================
# INITIAL CHECK
# ============================================================

print(
    "Frozen F1 final-unseen cases loaded:",
    len(
        frozen_f1_df
    ),
)


print(
    "\nCASE FAMILY COUNTS"
)

display(
    frozen_f1_df[
        "case_family"
    ]
    .fillna("MISSING")
    .value_counts()
    .rename_axis(
        "case_family"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nCASE FAMILY × PREDICTION"
)

display(
    pd.crosstab(
        frozen_f1_df[
            "case_family"
        ],
        frozen_f1_df[
            "prediction"
        ],
        margins=True,
        dropna=False,
    )
)

Frozen F1 final-unseen cases loaded: 68

CASE FAMILY COUNTS


,case_family,count
0,normal,17
1,wrong_partner,17
2,lag,17
3,silent_partner,17



CASE FAMILY × PREDICTION


prediction,ANOMALOUS,NORMAL,All
case_family,,,
lag,12,5,17
normal,6,11,17
silent_partner,17,0,17
wrong_partner,16,1,17
All,51,17,68


In [ ]:
# ============================================================
# LAG — CORRECTLY DETECTED
# ============================================================

frozen_f1_lag_correct = (
    get_frozen_f1_outcome_subset(
        family="lag",
        outcome="correct",
    )
)

inspect_frozen_f1_subset(
    frozen_f1_lag_correct,
    (
        "FROZEN F1 FINAL UNSEEN — "
        "LAG CASES CORRECTLY DETECTED"
    ),
)


# ============================================================
# LAG — MISSED
# ============================================================

frozen_f1_lag_missed = (
    get_frozen_f1_outcome_subset(
        family="lag",
        outcome="missed",
    )
)

inspect_frozen_f1_subset(
    frozen_f1_lag_missed,
    (
        "FROZEN F1 FINAL UNSEEN — "
        "LAG CASES MISSED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# WRONG PARTNER — CORRECTLY DETECTED
# ============================================================

frozen_f1_wrong_partner_correct = (
    get_frozen_f1_outcome_subset(
        family="wrong_partner",
        outcome="correct",
    )
)

inspect_frozen_f1_subset(
    frozen_f1_wrong_partner_correct,
    (
        "FROZEN F1 FINAL UNSEEN — "
        "WRONG-PARTNER CASES "
        "CORRECTLY DETECTED"
    ),
)


# ============================================================
# WRONG PARTNER — MISSED
# ============================================================

frozen_f1_wrong_partner_missed = (
    get_frozen_f1_outcome_subset(
        family="wrong_partner",
        outcome="missed",
    )
)

inspect_frozen_f1_subset(
    frozen_f1_wrong_partner_missed,
    (
        "FROZEN F1 FINAL UNSEEN — "
        "WRONG-PARTNER CASES MISSED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# NORMAL — CORRECTLY DETECTED
# ============================================================

frozen_f1_normal_correct = (
    get_frozen_f1_outcome_subset(
        family="normal",
        outcome="correct",
    )
)

inspect_frozen_f1_subset(
    frozen_f1_normal_correct,
    (
        "FROZEN F1 FINAL UNSEEN — "
        "NORMAL CASES CORRECTLY "
        "PREDICTED AS NORMAL"
    ),
)


# ============================================================
# NORMAL — FALSELY FLAGGED
# ============================================================

frozen_f1_normal_missed = (
    get_frozen_f1_outcome_subset(
        family="normal",
        outcome="missed",
    )
)

inspect_frozen_f1_subset(
    frozen_f1_normal_missed,
    (
        "FROZEN F1 FINAL UNSEEN — "
        "NORMAL CASES FALSELY FLAGGED "
        "AS ANOMALOUS"
    ),
)


# ============================================================
# SILENT PARTNER — CORRECTLY DETECTED
# ============================================================

frozen_f1_silent_partner_correct = (
    get_frozen_f1_outcome_subset(
        family="silent_partner",
        outcome="correct",
    )
)

inspect_frozen_f1_subset(
    frozen_f1_silent_partner_correct,
    (
        "FROZEN F1 FINAL UNSEEN — "
        "SILENT-PARTNER CASES "
        "CORRECTLY DETECTED"
    ),
)


# ============================================================
# SILENT PARTNER — MISSED
# ============================================================

frozen_f1_silent_partner_missed = (
    get_frozen_f1_outcome_subset(
        family="silent_partner",
        outcome="missed",
    )
)

inspect_frozen_f1_subset(
    frozen_f1_silent_partner_missed,
    (
        "FROZEN F1 FINAL UNSEEN — "
        "SILENT-PARTNER CASES MISSED "
        "(PREDICTED NORMAL)"
    ),
)


FROZEN F1 FINAL UNSEEN — LAG CASES CORRECTLY DETECTED

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,invalid_predictions,correct,accuracy_percent
0,12,0,12,0,12,0,12,100.0



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,lag_2sec,7,58.33
1,lag_3sec,5,41.67



ASSESSMENT BREAKDOWN


...

decisive_dimension,PARTICIPATION,SEMANTIC,TEMPORAL,All
temporal_assessment,,,,
ANOMALOUS,0,8,8,16
LIMITED,1,0,0,1
All,1,8,8,17



INDIVIDUAL CASES


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,semantic_assessment,decisive_dimension,schema_exact,parse_mode,input_token_count,elapsed_seconds,valid_prediction,correct
0,consolidation_silent_partner_000,heldout_source_000,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,SEMANTIC,True,structured_json,5178,12.1672,True,True
1,consolidation_silent_partner_001,heldout_source_001,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True,structured_json,5268,12.2441,True,True
2,consolidation_silent_partner_002,heldout_source_002,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,SEMANTIC,True,structured_json,5312,12.1998,True,True
3,consolidation_silent_partner_003,heldout_source_003,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True,structured_json,5195,12.2183,True,True
4,consolidation_silent_partner_004,heldout_source_004,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,SEMANTIC,True,structured_json,5210,12.1424,True,True
5,consolidation_silent_partner_005,heldout_source_005,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,SEMANTIC,True,structured_json,5253,12.1620,True,True
6,consolidation_silent_partner_006,heldout_source_006,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True,structured_json,5287,12.1919,True,True
7,consolidation_silent_partner_007,heldout_source_007,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,SEMANTIC,True,structured_json,5117,12.2164,True,True
8,consolidation_silent_partner_008,heldout_source_008,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True,structured_json,5207,12.2553,True,True
9,consolidation_silent_partner_009,heldout_source_009,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True,structured_json,5207,12.1565,True,True



FROZEN F1 FINAL UNSEEN — SILENT-PARTNER CASES MISSED (PREDICTED NORMAL)
No cases found in this subset.


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency


## 13. Release the frozen model

In [ ]:

# ============================================================
# RELEASE THE FROZEN MODEL BEFORE LOADING THE SAVED ADAPTER
# ============================================================

if "model" in globals():
    del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Frozen base model released from GPU memory.")


Frozen base model released from GPU memory.


## 14. Load the already-saved best fine-tuned F1 adapter

In [ ]:

# ============================================================
# LOAD THE ALREADY-SAVED BEST FINE-TUNED F1 ADAPTER
#
# No training is performed.
# The adapter is loaded from the original completed run.
# ============================================================

import gc
import torch

from transformers import (
    BitsAndBytesConfig,
    Qwen2_5OmniProcessor,
    Qwen2_5OmniThinkerForConditionalGeneration,
)

from peft import PeftModel

assert torch.cuda.is_available(), (
    "A CUDA GPU runtime is required."
)

COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

MODEL_SOURCE = (
    str(MODEL_PATH)
    if MODEL_PATH.exists()
    else MODEL_ID
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

processor = Qwen2_5OmniProcessor.from_pretrained(
    MODEL_SOURCE
)

tokenizer = processor.tokenizer

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

original_split_manifest = json.loads(
    ORIGINAL_SPLIT_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    original_split_manifest[
        "f1_prompt_sha256"
    ]
    ==
    F1_CONFIG[
        "reasoning_prompt_sha256"
    ]
), (
    "The saved adapter was trained with a different F1 prompt."
)

base_model_for_evaluation = (
    Qwen2_5OmniThinkerForConditionalGeneration
    .from_pretrained(
        MODEL_SOURCE,
        torch_dtype=COMPUTE_DTYPE,
        quantization_config=quantization_config,
        device_map={"": 0},
        low_cpu_mem_usage=True,
    )
)

model = PeftModel.from_pretrained(
    base_model_for_evaluation,
    BEST_ADAPTER_DIR,
    is_trainable=False,
)

model.eval()
model.config.use_cache = True

print("=" * 100)
print("SAVED FINE-TUNED F1 ADAPTER LOADED")
print("=" * 100)
print("Base model:", MODEL_SOURCE)
print("Adapter:", BEST_ADAPTER_DIR)
print("Training will run:", False)
print(
    "Prompt SHA256:",
    F1_CONFIG["reasoning_prompt_sha256"],
)


Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniThinkerForConditionalGeneration LOAD REPORT from: /content/drive/MyDrive/Qwen2.5-Omni-7B
Key                                                                                                      | Status     |  | 
---------------------------------------------------------------------------------------------------------+------------+--+-
talker.model.layers.{0...23}.self_attn.q_proj.weight                                                     | UNEXPECTED |  | 
talker.model.layers.{0...23}.self_attn.q_proj.bias                                                       | UNEXPECTED |  | 
talker.model.layers.{0...23}.self_attn.v_proj.weight                                                     | UNEXPECTED |  | 
token2wav.code2wav_bigvgan_model.resblocks.{0...17}.activations.{0, 1, 2, 3, 4, 5}.act.alpha             | UNEXPECTED |  | 
token2wav.code2wav_bigvgan_model.resblocks.{0...17}.convs2.{0, 1, 2}.bias                                | UNEXPECTED |  | 
talker.model.laye

SAVED FINE-TUNED F1 ADAPTER LOADED
Base model: /content/drive/MyDrive/Qwen2.5-Omni-7B
Adapter: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning/qlora_r8_lambda_0_5_seed_42/best_adapter
Training will run: False
Prompt SHA256: bf50d017f027995c6eaae81adad6b04d263530d25f2f4361e3ee3d9f19791355


## 15. Run the saved fine-tuned F1 on the same 68 cases

In [ ]:

# ============================================================
# RUN THE SAVED FINE-TUNED F1 ON ALL 68 FINAL-UNSEEN CASES
# ============================================================

from datetime import datetime, timezone
from tqdm.auto import tqdm

FINE_TUNED_TEST_CACHE_PATH = (
    FINE_TUNED_RESULTS_DIR
    / "predictions_cache.json"
)

def utc_now_iso():
    return datetime.now(
        timezone.utc
    ).isoformat()

def atomic_write_json(path, value):
    temporary_path = Path(
        str(path) + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


final_test_case_ids = sorted(
    str(case["case_id"])
    for case in consolidation_cases
)

adapter_manifest_hash = sha256_text(
    json.dumps(
        {
            "best_adapter_dir": str(BEST_ADAPTER_DIR),
            "split_manifest": original_split_manifest,
            "preservation_lambda": PRESERVATION_LAMBDA,
            "lora_rank": LORA_RANK,
            "lora_alpha": LORA_ALPHA,
            "learning_rate": LEARNING_RATE,
        },
        sort_keys=True,
        ensure_ascii=False,
    )
)

expected_test_cache_header = {
    "experiment": (
        "F1_semantic_targeted_QLoRA_final_unseen_test"
    ),
    "f1_prompt_sha256": (
        F1_CONFIG["reasoning_prompt_sha256"]
    ),
    "normal_reference_sha256": (
        F1_CONFIG["normal_reference_sha256"]
    ),
    "adapter_manifest_hash": adapter_manifest_hash,
    "final_test_case_ids": final_test_case_ids,
    "max_new_tokens": MAX_NEW_TOKENS_REASONING,
}

if FINE_TUNED_TEST_CACHE_PATH.exists():
    FINE_TUNED_TEST_CACHE = json.loads(
        FINE_TUNED_TEST_CACHE_PATH.read_text(
            encoding="utf-8"
        )
    )

    for key, expected_value in (
        expected_test_cache_header.items()
    ):
        assert (
            FINE_TUNED_TEST_CACHE[key]
            == expected_value
        ), (
            f"Incompatible fine-tuned evaluation cache: {key}"
        )

else:
    FINE_TUNED_TEST_CACHE = {
        **expected_test_cache_header,
        "created_at_utc": utc_now_iso(),
        "updated_at_utc": utc_now_iso(),
        "records": {},
    }

    atomic_write_json(
        FINE_TUNED_TEST_CACHE_PATH,
        FINE_TUNED_TEST_CACHE,
    )


ordered_final_cases = sorted(
    consolidation_cases,
    key=lambda case: str(
        case["case_id"]
    ),
)

assert len(ordered_final_cases) == 68

for case in tqdm(
    ordered_final_cases,
    desc="Fine-tuned F1 final-unseen test",
):
    case_id = str(case["case_id"])

    existing = (
        FINE_TUNED_TEST_CACHE[
            "records"
        ].get(case_id)
    )

    if (
        existing is not None
        and existing.get("prediction") in LABELS
        and existing.get("schema_exact") is True
    ):
        prompt, payload = build_semantic_ablation_prompt(
            case,
            F1_CONFIG,
        )

        assert (
            existing["prompt_sha256"]
            == sha256_text(prompt)
        )

        assert (
            existing["input_payload_sha256"]
            == sha256_text(
                canonical_json(payload)
            )
        )

        continue

    prompt, payload = build_semantic_ablation_prompt(
        case,
        F1_CONFIG,
    )

    prompt_hash = sha256_text(prompt)
    payload_hash = sha256_text(
        canonical_json(payload)
    )

    started = time.perf_counter()

    try:
        raw_output, input_token_count = (
            qwen_text_only_binary(
                prompt,
                max_new_tokens=(
                    MAX_NEW_TOKENS_REASONING
                ),
            )
        )

        parsed = parse_structured_reasoning_prediction(
            raw_output
        )

        generation_error = None

    except Exception as exc:
        raw_output = ""
        input_token_count = None

        parsed = {
            "prediction": None,
            "schema_exact": False,
            "participation_assessment": None,
            "local_temporal_assessment": None,
            "global_temporal_assessment": None,
            "temporal_assessment": None,
            "semantic_assessment": None,
            "decisive_dimension": None,
            "schema_errors": [],
            "parsed_output": None,
            "parse_mode": "generation_error",
        }

        generation_error = (
            f"{type(exc).__name__}: {exc}"
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    elapsed = time.perf_counter() - started

    FINE_TUNED_TEST_CACHE[
        "records"
    ][case_id] = {
        "case_id": case_id,
        "source_group_id": str(
            case["source_group_id"]
        ),
        "case_family": get_case_family(case),
        "case_variant": str(case["case_variant"]),
        "gold_binary_label": str(
            case["gold_binary_label"]
        ).upper(),
        "prompt_sha256": prompt_hash,
        "input_payload_sha256": payload_hash,
        "input_token_count": input_token_count,
        "raw_output": raw_output,
        "prediction": parsed["prediction"],
        "participation_assessment": (
            parsed["participation_assessment"]
        ),
        "local_temporal_assessment": (
            parsed["local_temporal_assessment"]
        ),
        "global_temporal_assessment": (
            parsed["global_temporal_assessment"]
        ),
        "temporal_assessment": (
            parsed["temporal_assessment"]
        ),
        "semantic_assessment": (
            parsed["semantic_assessment"]
        ),
        "decisive_dimension": (
            parsed["decisive_dimension"]
        ),
        "schema_exact": bool(
            parsed["schema_exact"]
        ),
        "schema_errors": parsed["schema_errors"],
        "parse_mode": parsed["parse_mode"],
        "parsed_output": parsed["parsed_output"],
        "generation_error": generation_error,
        "elapsed_seconds": round(elapsed, 4),
        "completed_at_utc": utc_now_iso(),
    }

    FINE_TUNED_TEST_CACHE[
        "updated_at_utc"
    ] = utc_now_iso()

    atomic_write_json(
        FINE_TUNED_TEST_CACHE_PATH,
        FINE_TUNED_TEST_CACHE,
    )


assert len(
    FINE_TUNED_TEST_CACHE["records"]
) == 68

assert set(
    FINE_TUNED_TEST_CACHE["records"].keys()
) == set(final_test_case_ids)

print("=" * 100)
print("FINE-TUNED F1 FINAL-UNSEEN EVALUATION COMPLETE")
print("=" * 100)
print(
    "Predictions:",
    len(
        FINE_TUNED_TEST_CACHE["records"]
    ),
)
print("Cache:", FINE_TUNED_TEST_CACHE_PATH)


Fine-tuned F1 final-unseen test:   0%|          | 0/68 [00:00<?, ?it/s]

FINE-TUNED F1 FINAL-UNSEEN EVALUATION COMPLETE
Predictions: 68
Cache: /content/drive/MyDrive/final_test_completely_unseen_database/f1_frozen_vs_finetuned_final_unseen_evaluation/fine_tuned_f1/predictions_cache.json


## 16. Paired frozen-vs-fine-tuned comparison

In [ ]:

# ============================================================
# FINAL PAIRED COMPARISON:
# FROZEN F1 vs FINE-TUNED F1 ON THE SAME 68 FINAL-UNSEEN CASES
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from scipy.stats import binomtest

def cache_records_dataframe(records):
    return pd.DataFrame(
        list(records.values())
    )

fine_tuned_df = cache_records_dataframe(
    FINE_TUNED_TEST_CACHE["records"]
)

test_case_ids = {
    str(case["case_id"])
    for case in consolidation_cases
}

frozen_f1_test_df = pd.DataFrame([
    {
        **FROZEN_F1_CACHE["records"][case_id],
        "case_id": case_id,
    }
    for case_id in sorted(test_case_ids)
])

assert len(fine_tuned_df) == 68
assert len(frozen_f1_test_df) == 68

for frame in [
    fine_tuned_df,
    frozen_f1_test_df,
]:
    frame["gold_label"] = (
        frame["gold_binary_label"]
        .astype(str)
        .str.upper()
    )

    frame["prediction"] = (
        frame["prediction"]
        .astype(str)
        .str.upper()
    )

    frame["valid_prediction"] = (
        frame["prediction"].isin(LABELS)
    )

    frame["correct"] = (
        frame["valid_prediction"]
        &
        (
            frame["prediction"]
            == frame["gold_label"]
        )
    )

def summarize_model(frame, model_name):
    valid = frame[
        frame["valid_prediction"]
    ].copy()

    summary = {
        "model": model_name,
        "valid_predictions": int(len(valid)),
        "exact_schema_rate": float(
            frame["schema_exact"].mean()
        ),
        "accuracy": float(
            accuracy_score(
                valid["gold_label"],
                valid["prediction"],
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                valid["gold_label"],
                valid["prediction"],
            )
        ),
        "macro_f1": float(
            f1_score(
                valid["gold_label"],
                valid["prediction"],
                average="macro",
            )
        ),
    }

    for family in [
        "normal",
        "lag",
        "wrong_partner",
        "silent_partner",
    ]:
        family_frame = valid[
            valid["case_family"] == family
        ]

        summary[
            f"{family}_accuracy"
        ] = float(
            family_frame["correct"].mean()
        )

    for family in [
        "normal",
        "lag",
        "wrong_partner",
        "silent_partner",
    ]:
        family_frame = valid[
            valid["case_family"] == family
        ]

        semantic_counts = (
            family_frame[
                "semantic_assessment"
            ].value_counts()
        )

        family_total = max(
            len(family_frame),
            1,
        )

        for semantic_value in [
            "COMPATIBLE",
            "INCOMPATIBLE",
            "LIMITED",
        ]:
            summary[
                f"{family}_{semantic_value}_percent"
            ] = (
                100.0
                * int(
                    semantic_counts.get(
                        semantic_value,
                        0,
                    )
                )
                / family_total
            )

    return summary

comparison_summary_df = pd.DataFrame([
    summarize_model(
        frozen_f1_test_df,
        "Frozen F1",
    ),
    summarize_model(
        fine_tuned_df,
        "Fine-tuned F1",
    ),
])

print("=" * 100)
print("FINAL-UNSEEN COMPARISON SUMMARY")
print("=" * 100)

display(comparison_summary_df)

def display_confusion(frame, title):
    valid = frame[
        frame["valid_prediction"]
    ]

    matrix = confusion_matrix(
        valid["gold_label"],
        valid["prediction"],
        labels=LABELS,
    )

    confusion_df = pd.DataFrame(
        matrix,
        index=[
            "Gold NORMAL",
            "Gold ANOMALOUS",
        ],
        columns=[
            "Pred NORMAL",
            "Pred ANOMALOUS",
        ],
    )

    print("\n" + title)
    display(confusion_df)

display_confusion(
    frozen_f1_test_df,
    "FROZEN F1 CONFUSION MATRIX",
)

display_confusion(
    fine_tuned_df,
    "FINE-TUNED F1 CONFUSION MATRIX",
)

paired = (
    frozen_f1_test_df[
        [
            "case_id",
            "correct",
            "prediction",
            "participation_assessment",
            "local_temporal_assessment",
            "global_temporal_assessment",
            "temporal_assessment",
            "semantic_assessment",
            "decisive_dimension",
        ]
    ]
    .rename(columns={
        column: f"frozen_{column}"
        for column in [
            "correct",
            "prediction",
            "participation_assessment",
            "local_temporal_assessment",
            "global_temporal_assessment",
            "temporal_assessment",
            "semantic_assessment",
            "decisive_dimension",
        ]
    })
    .merge(
        fine_tuned_df[
            [
                "case_id",
                "case_family",
                "gold_label",
                "correct",
                "prediction",
                "participation_assessment",
                "local_temporal_assessment",
                "global_temporal_assessment",
                "temporal_assessment",
                "semantic_assessment",
                "decisive_dimension",
            ]
        ].rename(columns={
            column: f"fine_tuned_{column}"
            for column in [
                "correct",
                "prediction",
                "participation_assessment",
                "local_temporal_assessment",
                "global_temporal_assessment",
                "temporal_assessment",
                "semantic_assessment",
                "decisive_dimension",
            ]
        }),
        on="case_id",
        how="inner",
        validate="one_to_one",
    )
)

assert len(paired) == 68

branch_stability_rows = []

for field in [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "decisive_dimension",
    "prediction",
]:
    same = (
        paired[f"frozen_{field}"]
        == paired[f"fine_tuned_{field}"]
    )

    branch_stability_rows.append({
        "field": field,
        "same_count": int(same.sum()),
        "same_percent": float(
            100.0 * same.mean()
        ),
        "changed_count": int((~same).sum()),
    })

branch_stability_df = pd.DataFrame(
    branch_stability_rows
)

print("\n" + "=" * 100)
print("NON-SEMANTIC BRANCH PRESERVATION")
print("=" * 100)

display(branch_stability_df)

b = int(
    (
        paired["frozen_correct"]
        &
        ~paired["fine_tuned_correct"]
    ).sum()
)

c = int(
    (
        ~paired["frozen_correct"]
        &
        paired["fine_tuned_correct"]
    ).sum()
)

mcnemar_p = (
    float(
        binomtest(
            min(b, c),
            n=b + c,
            p=0.5,
            alternative="two-sided",
        ).pvalue
    )
    if b + c > 0
    else 1.0
)

print("\nPaired prediction changes")
print("Frozen correct / Fine-tuned wrong (b):", b)
print("Frozen wrong / Fine-tuned correct (c):", c)
print("Exact McNemar p-value:", mcnemar_p)

def reasoning_inconsistency_flags(frame):
    frame = frame.copy()

    frame["normal_with_invalid_participation"] = (
        (frame["prediction"] == "NORMAL")
        &
        (
            frame["participation_assessment"]
            == "INVALID"
        )
    )

    frame["normal_with_anomalous_temporal"] = (
        (frame["prediction"] == "NORMAL")
        &
        (
            frame["temporal_assessment"]
            == "ANOMALOUS"
        )
    )

    frame["normal_with_incompatible_semantics"] = (
        (frame["prediction"] == "NORMAL")
        &
        (
            frame["semantic_assessment"]
            == "INCOMPATIBLE"
        )
    )

    frame["anomalous_without_explicit_failure"] = (
        (frame["prediction"] == "ANOMALOUS")
        &
        (
            frame["participation_assessment"]
            != "INVALID"
        )
        &
        (
            frame["temporal_assessment"]
            != "ANOMALOUS"
        )
        &
        (
            frame["semantic_assessment"]
            != "INCOMPATIBLE"
        )
    )

    return frame

frozen_consistency_df = reasoning_inconsistency_flags(
    frozen_f1_test_df
)

fine_tuned_consistency_df = reasoning_inconsistency_flags(
    fine_tuned_df
)

consistency_summary_df = pd.DataFrame([
    {
        "model": "Frozen F1",
        **{
            column: int(
                frozen_consistency_df[column].sum()
            )
            for column in [
                "normal_with_invalid_participation",
                "normal_with_anomalous_temporal",
                "normal_with_incompatible_semantics",
                "anomalous_without_explicit_failure",
            ]
        },
    },
    {
        "model": "Fine-tuned F1",
        **{
            column: int(
                fine_tuned_consistency_df[column].sum()
            )
            for column in [
                "normal_with_invalid_participation",
                "normal_with_anomalous_temporal",
                "normal_with_incompatible_semantics",
                "anomalous_without_explicit_failure",
            ]
        },
    },
])

print("\n" + "=" * 100)
print("LOGICAL CONSISTENCY")
print("=" * 100)

display(consistency_summary_df)

comparison_summary_df.to_csv(
    PAIRED_RESULTS_DIR
    / "final_unseen_comparison_summary.csv",
    index=False,
)

branch_stability_df.to_csv(
    PAIRED_RESULTS_DIR
    / "branch_stability.csv",
    index=False,
)

consistency_summary_df.to_csv(
    PAIRED_RESULTS_DIR
    / "logical_consistency_summary.csv",
    index=False,
)

paired.to_csv(
    PAIRED_RESULTS_DIR
    / "paired_case_level_comparison.csv",
    index=False,
)

fine_tuned_df.to_csv(
    PAIRED_RESULTS_DIR
    / "fine_tuned_final_unseen_predictions.csv",
    index=False,
)

print("\nSaved final results to:", PAIRED_RESULTS_DIR)


# Save the frozen final-unseen case-level predictions as well.
frozen_f1_test_df.to_csv(
    PAIRED_RESULTS_DIR
    / "frozen_f1_final_unseen_predictions.csv",
    index=False,
)

# Save one compact JSON summary for reporting.
FINAL_UNSEEN_COMPARISON_JSON_PATH = (
    PAIRED_RESULTS_DIR
    / "final_unseen_comparison_summary.json"
)

FINAL_UNSEEN_COMPARISON_JSON_PATH.write_text(
    json.dumps(
        {
            "num_cases": 68,
            "num_source_groups": 17,
            "f1_prompt_sha256": (
                F1_CONFIG[
                    "reasoning_prompt_sha256"
                ]
            ),
            "normal_reference_sha256": (
                F1_CONFIG[
                    "normal_reference_sha256"
                ]
            ),
            "best_adapter_dir": str(
                BEST_ADAPTER_DIR
            ),
            "frozen_cache_path": str(
                F1_CONFIG[
                    "paths"
                ][
                    "prediction_cache"
                ]
            ),
            "fine_tuned_cache_path": str(
                FINE_TUNED_TEST_CACHE_PATH
            ),
            "comparison_rows": (
                comparison_summary_df
                .to_dict(
                    orient="records"
                )
            ),
            "mcnemar": {
                "frozen_correct_finetuned_wrong": b,
                "frozen_wrong_finetuned_correct": c,
                "exact_p_value": mcnemar_p,
            },
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    "Saved compact JSON summary:",
    FINAL_UNSEEN_COMPARISON_JSON_PATH,
)


FINAL-UNSEEN COMPARISON SUMMARY


,model,valid_predictions,exact_schema_rate,accuracy,balanced_accuracy,macro_f1,normal_accuracy,lag_accuracy,wrong_partner_accuracy,silent_partner_accuracy,...,normal_LIMITED_percent,lag_COMPATIBLE_percent,lag_INCOMPATIBLE_percent,lag_LIMITED_percent,wrong_partner_COMPATIBLE_percent,wrong_partner_INCOMPATIBLE_percent,wrong_partner_LIMITED_percent,silent_partner_COMPATIBLE_percent,silent_partner_INCOMPATIBLE_percent,silent_partner_LIMITED_percent
0,Frozen F1,68,1.0,0.823529,0.764706,0.764706,0.647059,0.705882,0.941176,1.0,...,0.0,100.000000,0.000000,0.000000,41.176471,35.294118,23.529412,11.764706,47.058824,41.176471
1,Fine-tuned F1,68,1.0,0.852941,0.862745,0.822917,0.882353,0.588235,0.941176,1.0,...,0.0,88.235294,5.882353,5.882353,17.647059,70.588235,11.764706,11.764706,52.941176,35.294118



FROZEN F1 CONFUSION MATRIX


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,11,6
Gold ANOMALOUS,6,45



FINE-TUNED F1 CONFUSION MATRIX


...


NON-SEMANTIC BRANCH PRESERVATION


,field,same_count,same_percent,changed_count
0,participation_assessment,68,100.000000,0
1,local_temporal_assessment,64,94.117647,4
2,global_temporal_assessment,65,95.588235,3
3,temporal_assessment,65,95.588235,3
4,decisive_dimension,59,86.764706,9
5,prediction,62,91.176471,6



Paired prediction changes
Frozen correct / Fine-tuned wrong (b): 2
Frozen wrong / Fine-tuned correct (c): 4
Exact McNemar p-value: 0.6875

LOGICAL CONSISTENCY


,model,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,anomalous_without_explicit_failure
0,Frozen F1,0,0,0,0
1,Fine-tuned F1,0,0,0,0



Saved final results to: /content/drive/MyDrive/final_test_completely_unseen_database/f1_frozen_vs_finetuned_final_unseen_evaluation/paired_comparison
Saved compact JSON summary: /content/drive/MyDrive/final_test_completely_unseen_database/f1_frozen_vs_finetuned_final_unseen_evaluation/paired_comparison/final_unseen_comparison_summary.json


In [ ]:
# ============================================================
# FINE-TUNED F1 — OUTCOME SUBSET HELPER
# ============================================================

fine_tuned_f1_df = fine_tuned_df.copy()


def get_fine_tuned_f1_outcome_subset(
    family,
    outcome,
):

    family_mask = (
        get_frozen_f1_family_mask(
            dataframe=fine_tuned_f1_df,
            family=family,
        )
    )


    if outcome == "invalid":

        return fine_tuned_f1_df[
            family_mask
            &
            (
                ~fine_tuned_f1_df[
                    "valid_prediction"
                ]
            )
        ].copy()


    if family == "normal":

        gold_label = "NORMAL"

        if outcome == "correct":

            prediction = "NORMAL"

        elif outcome == "missed":

            prediction = "ANOMALOUS"

        else:

            raise ValueError(
                "outcome must be 'correct', "
                "'missed' or 'invalid'."
            )

    else:

        gold_label = "ANOMALOUS"

        if outcome == "correct":

            prediction = "ANOMALOUS"

        elif outcome == "missed":

            prediction = "NORMAL"

        else:

            raise ValueError(
                "outcome must be 'correct', "
                "'missed' or 'invalid'."
            )


    return fine_tuned_f1_df[
        family_mask
        &
        (
            fine_tuned_f1_df[
                "gold_label"
            ]
            ==
            gold_label
        )
        &
        (
            fine_tuned_f1_df[
                "prediction"
            ]
            ==
            prediction
        )
    ].copy()


print(
    "Fine-tuned F1 final-unseen cases loaded:",
    len(
        fine_tuned_f1_df
    ),
)

Fine-tuned F1 final-unseen cases loaded: 68


In [ ]:
# ============================================================
# LAG — CORRECTLY DETECTED
# ============================================================

fine_tuned_f1_lag_correct = (
    get_fine_tuned_f1_outcome_subset(
        family="lag",
        outcome="correct",
    )
)

inspect_frozen_f1_subset(
    fine_tuned_f1_lag_correct,
    (
        "FINE-TUNED F1 FINAL UNSEEN — "
        "LAG CASES CORRECTLY DETECTED"
    ),
)


# ============================================================
# LAG — MISSED
# ============================================================

fine_tuned_f1_lag_missed = (
    get_fine_tuned_f1_outcome_subset(
        family="lag",
        outcome="missed",
    )
)

inspect_frozen_f1_subset(
    fine_tuned_f1_lag_missed,
    (
        "FINE-TUNED F1 FINAL UNSEEN — "
        "LAG CASES MISSED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# WRONG PARTNER — CORRECTLY DETECTED
# ============================================================

fine_tuned_f1_wrong_partner_correct = (
    get_fine_tuned_f1_outcome_subset(
        family="wrong_partner",
        outcome="correct",
    )
)

inspect_frozen_f1_subset(
    fine_tuned_f1_wrong_partner_correct,
    (
        "FINE-TUNED F1 FINAL UNSEEN — "
        "WRONG-PARTNER CASES "
        "CORRECTLY DETECTED"
    ),
)


# ============================================================
# WRONG PARTNER — MISSED
# ============================================================

fine_tuned_f1_wrong_partner_missed = (
    get_fine_tuned_f1_outcome_subset(
        family="wrong_partner",
        outcome="missed",
    )
)

inspect_frozen_f1_subset(
    fine_tuned_f1_wrong_partner_missed,
    (
        "FINE-TUNED F1 FINAL UNSEEN — "
        "WRONG-PARTNER CASES MISSED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# NORMAL — CORRECTLY DETECTED
# ============================================================

fine_tuned_f1_normal_correct = (
    get_fine_tuned_f1_outcome_subset(
        family="normal",
        outcome="correct",
    )
)

inspect_frozen_f1_subset(
    fine_tuned_f1_normal_correct,
    (
        "FINE-TUNED F1 FINAL UNSEEN — "
        "NORMAL CASES CORRECTLY "
        "PREDICTED AS NORMAL"
    ),
)


# ============================================================
# NORMAL — FALSELY FLAGGED
# ============================================================

fine_tuned_f1_normal_missed = (
    get_fine_tuned_f1_outcome_subset(
        family="normal",
        outcome="missed",
    )
)

inspect_frozen_f1_subset(
    fine_tuned_f1_normal_missed,
    (
        "FINE-TUNED F1 FINAL UNSEEN — "
        "NORMAL CASES FALSELY FLAGGED "
        "AS ANOMALOUS"
    ),
)


# ============================================================
# SILENT PARTNER — CORRECTLY DETECTED
# ============================================================

fine_tuned_f1_silent_partner_correct = (
    get_fine_tuned_f1_outcome_subset(
        family="silent_partner",
        outcome="correct",
    )
)

inspect_frozen_f1_subset(
    fine_tuned_f1_silent_partner_correct,
    (
        "FINE-TUNED F1 FINAL UNSEEN — "
        "SILENT-PARTNER CASES "
        "CORRECTLY DETECTED"
    ),
)


# ============================================================
# SILENT PARTNER — MISSED
# ============================================================

fine_tuned_f1_silent_partner_missed = (
    get_fine_tuned_f1_outcome_subset(
        family="silent_partner",
        outcome="missed",
    )
)

inspect_frozen_f1_subset(
    fine_tuned_f1_silent_partner_missed,
    (
        "FINE-TUNED F1 FINAL UNSEEN — "
        "SILENT-PARTNER CASES MISSED "
        "(PREDICTED NORMAL)"
    ),
)


FINE-TUNED F1 FINAL UNSEEN — LAG CASES CORRECTLY DETECTED

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,invalid_predictions,correct,accuracy_percent
0,10,0,10,0,10,0,10,100.0



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,lag_2sec,7,70.0
1,lag_3sec,3,30.0



ASSESSMENT BREAKDOWN


...

decisive_dimension,PARTICIPATION,SEMANTIC,TEMPORAL,All
temporal_assessment,,,,
ANOMALOUS,0,2,13,15
LIMITED,2,0,0,2
All,2,2,13,17



INDIVIDUAL CASES


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,semantic_assessment,decisive_dimension,schema_exact,parse_mode,input_token_count,elapsed_seconds,valid_prediction,correct
0,consolidation_silent_partner_000,heldout_source_000,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,SEMANTIC,True,structured_json,5178,25.1747,True,True
1,consolidation_silent_partner_001,heldout_source_001,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True,structured_json,5268,25.1926,True,True
2,consolidation_silent_partner_002,heldout_source_002,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True,structured_json,5312,25.2635,True,True
3,consolidation_silent_partner_003,heldout_source_003,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True,structured_json,5195,25.3129,True,True
4,consolidation_silent_partner_004,heldout_source_004,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True,structured_json,5210,25.2011,True,True
5,consolidation_silent_partner_005,heldout_source_005,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True,structured_json,5253,25.3393,True,True
6,consolidation_silent_partner_006,heldout_source_006,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True,structured_json,5287,25.3255,True,True
7,consolidation_silent_partner_007,heldout_source_007,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True,structured_json,5117,25.2080,True,True
8,consolidation_silent_partner_008,heldout_source_008,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True,structured_json,5207,25.0919,True,True
9,consolidation_silent_partner_009,heldout_source_009,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True,structured_json,5207,25.2355,True,True



FINE-TUNED F1 FINAL UNSEEN — SILENT-PARTNER CASES MISSED (PREDICTED NORMAL)
No cases found in this subset.


,case_id,source_group_id,case_family,case_variant,gold_binary_label,prompt_sha256,input_payload_sha256,input_token_count,raw_output,prediction,...,schema_exact,schema_errors,parse_mode,parsed_output,generation_error,elapsed_seconds,completed_at_utc,gold_label,valid_prediction,correct


## 17. Final exactness and completeness audit

In [ ]:

# ============================================================
# FINAL EXACTNESS AND COMPLETENESS AUDIT
# ============================================================

frozen_records = (
    FROZEN_F1_CACHE["records"]
)

fine_tuned_records = (
    FINE_TUNED_TEST_CACHE["records"]
)

expected_case_ids = {
    str(case["case_id"])
    for case in consolidation_cases
}

assert set(frozen_records) == expected_case_ids
assert set(fine_tuned_records) == expected_case_ids

for case in consolidation_cases:
    case_id = str(case["case_id"])

    prompt, payload = (
        build_semantic_ablation_prompt(
            case,
            F1_CONFIG,
        )
    )

    prompt_hash = sha256_text(prompt)
    payload_hash = sha256_text(
        canonical_json(payload)
    )

    for model_name, record in [
        (
            "frozen",
            frozen_records[case_id],
        ),
        (
            "fine_tuned",
            fine_tuned_records[case_id],
        ),
    ]:
        assert (
            record["prompt_sha256"]
            == prompt_hash
        ), (
            f"{model_name} prompt mismatch: {case_id}"
        )

        assert (
            record["input_payload_sha256"]
            == payload_hash
        ), (
            f"{model_name} payload mismatch: {case_id}"
        )

        assert record["prediction"] in LABELS
        assert record["schema_exact"] is True
        assert record["generation_error"] is None

assert (
    F1_CONFIG["reasoning_prompt_sha256"]
    ==
    ORIGINAL_F1_CACHE["reasoning_prompt_sha256"]
)

assert (
    F1_CONFIG["normal_reference_sha256"]
    ==
    ORIGINAL_F1_CACHE["normal_reference_sha256"]
)

print("=" * 100)
print("FINAL FROZEN-vs-FINE-TUNED UNSEEN EVALUATION AUDIT PASSED")
print("=" * 100)
print("Cases checked per model: 68")
print("Same exact prompt per paired case: YES")
print("Same exact input payload per paired case: YES")
print("Exact-schema outputs: 68 / 68 for both models")
print("Original F1 prompt hash preserved: YES")
print("Original frozen NORMAL references preserved: YES")
print("No training performed in this notebook: YES")
print("Old databases/caches modified: NO")
print("Results root:", FINAL_EVALUATION_ROOT)


FINAL FROZEN-vs-FINE-TUNED UNSEEN EVALUATION AUDIT PASSED
Cases checked per model: 68
Same exact prompt per paired case: YES
Same exact input payload per paired case: YES
Exact-schema outputs: 68 / 68 for both models
Original F1 prompt hash preserved: YES
Original frozen NORMAL references preserved: YES
No training performed in this notebook: YES
Old databases/caches modified: NO
Results root: /content/drive/MyDrive/final_test_completely_unseen_database/f1_frozen_vs_finetuned_final_unseen_evaluation


In [ ]:
import pandas as pd
from IPython.display import display


# ============================================================
# FINAL-UNSEEN CASE-LEVEL INSPECTION HELPERS
#
# Supports:
# - Frozen F1
# - Fine-tuned F1
# - correct/missed cases per anomaly family
# - paired Frozen vs Fine-tuned inspection
# ============================================================


# ------------------------------------------------------------
# Normalize the relevant fields
# ------------------------------------------------------------

for dataframe in [
    frozen_f1_test_df,
    fine_tuned_df,
]:

    for column in [
        "gold_label",
        "prediction",
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
    ]:

        if column in dataframe.columns:

            dataframe[column] = (
                dataframe[column]
                .astype("string")
                .str.strip()
                .str.upper()
            )


    if "case_family" in dataframe.columns:

        dataframe["case_family"] = (
            dataframe["case_family"]
            .astype("string")
            .str.strip()
            .str.lower()
            .str.replace(
                r"[\s\-]+",
                "_",
                regex=True,
            )
        )


    if "case_variant" in dataframe.columns:

        dataframe["case_variant"] = (
            dataframe["case_variant"]
            .astype("string")
            .str.strip()
            .str.lower()
            .str.replace(
                r"[\s\-]+",
                "_",
                regex=True,
            )
        )


    dataframe["case_id"] = (
        dataframe["case_id"]
        .astype(str)
        .str.strip()
    )


# ------------------------------------------------------------
# Strict checks
# ------------------------------------------------------------

assert len(
    frozen_f1_test_df
) == 68

assert len(
    fine_tuned_df
) == 68

assert frozen_f1_test_df[
    "case_id"
].is_unique

assert fine_tuned_df[
    "case_id"
].is_unique

assert set(
    frozen_f1_test_df[
        "case_id"
    ]
) == set(
    fine_tuned_df[
        "case_id"
    ]
)


# ============================================================
# FAMILY MASK
# ============================================================

def get_final_unseen_family_mask(
    dataframe,
    family,
):

    values = (
        dataframe[
            "case_family"
        ]
        .fillna("")
        .astype(str)
        .str.lower()
    )


    if family == "normal":

        return values.isin([
            "normal",
            "normals",
        ])


    if family == "lag":

        return (
            values.isin([
                "lag",
                "lags",
            ])
            |
            values.str.contains(
                r"(?:^|_)lag(?:$|_)",
                regex=True,
            )
        )


    if family == "wrong_partner":

        return (
            values.isin([
                "wrong_partner",
                "wrong_partners",
            ])
            |
            (
                values.str.contains(
                    "wrong",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    if family == "silent_partner":

        return (
            values.isin([
                "silent_partner",
                "silent_partners",
            ])
            |
            (
                values.str.contains(
                    "silent",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    raise ValueError(
        f"Unknown family: {family}"
    )


# ============================================================
# GET CORRECT / MISSED / INVALID SUBSET
# ============================================================

def get_final_unseen_outcome_subset(
    dataframe,
    family,
    outcome,
):

    family_mask = (
        get_final_unseen_family_mask(
            dataframe=dataframe,
            family=family,
        )
    )


    if outcome == "invalid":

        return dataframe[
            family_mask
            &
            (
                ~dataframe[
                    "valid_prediction"
                ]
            )
        ].copy()


    if family == "normal":

        expected_gold = "NORMAL"

        if outcome == "correct":

            expected_prediction = "NORMAL"

        elif outcome == "missed":

            expected_prediction = "ANOMALOUS"

        else:

            raise ValueError(
                "outcome must be correct, missed or invalid."
            )

    else:

        expected_gold = "ANOMALOUS"

        if outcome == "correct":

            expected_prediction = "ANOMALOUS"

        elif outcome == "missed":

            expected_prediction = "NORMAL"

        else:

            raise ValueError(
                "outcome must be correct, missed or invalid."
            )


    return dataframe[
        family_mask
        &
        (
            dataframe[
                "gold_label"
            ]
            ==
            expected_gold
        )
        &
        (
            dataframe[
                "prediction"
            ]
            ==
            expected_prediction
        )
    ].copy()


# ============================================================
# CASE-LEVEL TABLE
# ============================================================

def display_final_unseen_cases(
    subset,
    title,
):

    subset = subset.copy()


    print(
        "\n"
        +
        "=" * 110
    )

    print(
        title
    )

    print(
        "=" * 110
    )


    if subset.empty:

        print(
            "No cases found in this subset."
        )

        return subset


    preferred_columns = [
        "case_id",
        "source_group_id",
        "pair_index",
        "case_family",
        "case_variant",
        "gold_label",
        "prediction",
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
        "schema_exact",
        "parse_mode",
        "input_token_count",
        "elapsed_seconds",
        "valid_prediction",
        "correct",
    ]


    available_columns = [
        column
        for column in preferred_columns
        if column in subset.columns
    ]


    summary = pd.DataFrame([{

        "cases": len(
            subset
        ),

        "pred_NORMAL": int(
            (
                subset[
                    "prediction"
                ]
                ==
                "NORMAL"
            ).sum()
        ),

        "pred_ANOMALOUS": int(
            (
                subset[
                    "prediction"
                ]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "correct": int(
            subset[
                "correct"
            ].sum()
        ),

        "incorrect": int(
            (
                ~subset[
                    "correct"
                ]
            ).sum()
        ),
    }])


    print(
        "\nSUBSET SUMMARY"
    )

    display(
        summary
    )


    print(
        "\nINDIVIDUAL CASES"
    )


    case_table = (
        subset[
            available_columns
        ]
        .sort_values(
            by=[
                column
                for column in [
                    "case_variant",
                    "case_id",
                ]
                if column in available_columns
            ]
        )
        .reset_index(
            drop=True
        )
    )


    display(
        case_table
    )


    return subset


# ============================================================
# INSPECT ONE CASE FOR ONE MODEL
# ============================================================

def inspect_single_model_case(
    dataframe,
    case_id,
    model_name,
):

    case_id = str(
        case_id
    ).strip()


    selected = dataframe[
        dataframe[
            "case_id"
        ]
        ==
        case_id
    ].copy()


    if selected.empty:

        raise KeyError(
            f"Case not found: {case_id}"
        )


    assert len(
        selected
    ) == 1


    row = selected.iloc[0]


    print(
        "\n"
        +
        "=" * 110
    )

    print(
        f"{model_name} — CASE {case_id}"
    )

    print(
        "=" * 110
    )


    fields = [
        "case_family",
        "case_variant",
        "gold_label",
        "prediction",
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
        "schema_exact",
        "parse_mode",
        "valid_prediction",
        "correct",
        "input_token_count",
        "elapsed_seconds",
        "generation_error",
    ]


    for field in fields:

        if field in selected.columns:

            print(
                f"{field}:",
                row[field],
            )


    if "schema_errors" in selected.columns:

        print(
            "\nSCHEMA ERRORS"
        )

        print(
            row[
                "schema_errors"
            ]
        )


    if "raw_output" in selected.columns:

        print(
            "\nRAW MODEL OUTPUT"
        )

        print(
            "-" * 110
        )

        print(
            row[
                "raw_output"
            ]
        )


    return selected


# ============================================================
# PAIRED INSPECTION OF THE SAME CASE
# ============================================================

def inspect_frozen_vs_finetuned_case(
    case_id,
    show_raw_outputs=True,
):

    case_id = str(
        case_id
    ).strip()


    frozen_case = frozen_f1_test_df[
        frozen_f1_test_df[
            "case_id"
        ]
        ==
        case_id
    ].copy()


    fine_tuned_case = fine_tuned_df[
        fine_tuned_df[
            "case_id"
        ]
        ==
        case_id
    ].copy()


    if frozen_case.empty:

        raise KeyError(
            f"Frozen case not found: {case_id}"
        )


    if fine_tuned_case.empty:

        raise KeyError(
            f"Fine-tuned case not found: {case_id}"
        )


    assert len(
        frozen_case
    ) == 1

    assert len(
        fine_tuned_case
    ) == 1


    frozen_row = frozen_case.iloc[0]
    fine_tuned_row = fine_tuned_case.iloc[0]


    comparison_fields = [
        "prediction",
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
        "schema_exact",
        "parse_mode",
        "valid_prediction",
        "correct",
    ]


    comparison_rows = []


    for field in comparison_fields:

        frozen_value = (
            frozen_row[field]
            if field in frozen_case.columns
            else None
        )

        fine_tuned_value = (
            fine_tuned_row[field]
            if field in fine_tuned_case.columns
            else None
        )


        comparison_rows.append({

            "field": field,

            "frozen_F1": frozen_value,

            "fine_tuned_F1": fine_tuned_value,

            "changed": (
                frozen_value
                !=
                fine_tuned_value
            ),
        })


    print(
        "\n"
        +
        "=" * 110
    )

    print(
        "FROZEN F1 vs FINE-TUNED F1"
    )

    print(
        "CASE:",
        case_id,
    )

    print(
        "FAMILY:",
        fine_tuned_row.get(
            "case_family",
            "UNKNOWN",
        ),
    )

    print(
        "VARIANT:",
        fine_tuned_row.get(
            "case_variant",
            "UNKNOWN",
        ),
    )

    print(
        "GOLD:",
        fine_tuned_row[
            "gold_label"
        ],
    )

    print(
        "=" * 110
    )


    display(
        pd.DataFrame(
            comparison_rows
        )
    )


    if show_raw_outputs:

        if "raw_output" in frozen_case.columns:

            print(
                "\nFROZEN F1 RAW OUTPUT"
            )

            print(
                "-" * 110
            )

            print(
                frozen_row[
                    "raw_output"
                ]
            )


        if "raw_output" in fine_tuned_case.columns:

            print(
                "\nFINE-TUNED F1 RAW OUTPUT"
            )

            print(
                "-" * 110
            )

            print(
                fine_tuned_row[
                    "raw_output"
                ]
            )


    return pd.DataFrame(
        comparison_rows
    )


print(
    "Final-unseen inspection helpers ready."
)

Final-unseen inspection helpers ready.


In [ ]:
# ============================================================
# FINE-TUNED F1 — LAG CORRECT
# ============================================================

fine_tuned_lag_correct = (
    get_final_unseen_outcome_subset(
        dataframe=fine_tuned_df,
        family="lag",
        outcome="correct",
    )
)

display_final_unseen_cases(
    fine_tuned_lag_correct,
    "FINE-TUNED F1 — LAG CORRECTLY DETECTED",
)


# ============================================================
# FINE-TUNED F1 — LAG MISSED
# ============================================================

fine_tuned_lag_missed = (
    get_final_unseen_outcome_subset(
        dataframe=fine_tuned_df,
        family="lag",
        outcome="missed",
    )
)

display_final_unseen_cases(
    fine_tuned_lag_missed,
    "FINE-TUNED F1 — LAG MISSED",
)


# ============================================================
# FINE-TUNED F1 — WRONG PARTNER CORRECT
# ============================================================

fine_tuned_wrong_correct = (
    get_final_unseen_outcome_subset(
        dataframe=fine_tuned_df,
        family="wrong_partner",
        outcome="correct",
    )
)

display_final_unseen_cases(
    fine_tuned_wrong_correct,
    "FINE-TUNED F1 — WRONG PARTNER CORRECTLY DETECTED",
)


# ============================================================
# FINE-TUNED F1 — WRONG PARTNER MISSED
# ============================================================

fine_tuned_wrong_missed = (
    get_final_unseen_outcome_subset(
        dataframe=fine_tuned_df,
        family="wrong_partner",
        outcome="missed",
    )
)

display_final_unseen_cases(
    fine_tuned_wrong_missed,
    "FINE-TUNED F1 — WRONG PARTNER MISSED",
)


# ============================================================
# FINE-TUNED F1 — NORMAL CORRECT
# ============================================================

fine_tuned_normal_correct = (
    get_final_unseen_outcome_subset(
        dataframe=fine_tuned_df,
        family="normal",
        outcome="correct",
    )
)

display_final_unseen_cases(
    fine_tuned_normal_correct,
    "FINE-TUNED F1 — NORMAL CORRECTLY PREDICTED",
)


# ============================================================
# FINE-TUNED F1 — NORMAL FALSE POSITIVES
# ============================================================

fine_tuned_normal_missed = (
    get_final_unseen_outcome_subset(
        dataframe=fine_tuned_df,
        family="normal",
        outcome="missed",
    )
)

display_final_unseen_cases(
    fine_tuned_normal_missed,
    "FINE-TUNED F1 — NORMAL FALSELY FLAGGED",
)


# ============================================================
# FINE-TUNED F1 — SILENT PARTNER CORRECT
# ============================================================

fine_tuned_silent_correct = (
    get_final_unseen_outcome_subset(
        dataframe=fine_tuned_df,
        family="silent_partner",
        outcome="correct",
    )
)

display_final_unseen_cases(
    fine_tuned_silent_correct,
    "FINE-TUNED F1 — SILENT PARTNER CORRECTLY DETECTED",
)


# ============================================================
# FINE-TUNED F1 — SILENT PARTNER MISSED
# ============================================================

fine_tuned_silent_missed = (
    get_final_unseen_outcome_subset(
        dataframe=fine_tuned_df,
        family="silent_partner",
        outcome="missed",
    )
)

display_final_unseen_cases(
    fine_tuned_silent_missed,
    "FINE-TUNED F1 — SILENT PARTNER MISSED",
)


FINE-TUNED F1 — LAG CORRECTLY DETECTED

SUBSET SUMMARY


,cases,pred_NORMAL,pred_ANOMALOUS,correct,incorrect
0,10,0,10,10,0



INDIVIDUAL CASES


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,semantic_assessment,decisive_dimension,schema_exact,parse_mode,input_token_count,elapsed_seconds,valid_prediction,correct
0,consolidation_lag_2sec_001,heldout_source_001,lag,lag_2sec,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True,structured_json,5298,25.2968,True,True
1,consolidation_lag_2sec_002,heldout_source_002,lag,lag_2sec,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True,structured_json,5365,25.4050,True,True
2,consolidation_lag_2sec_003,heldout_source_003,lag,lag_2sec,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True,structured_json,5325,25.3124,True,True
3,consolidation_lag_2sec_004,heldout_source_004,lag,lag_2sec,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True,structured_json,5275,25.3001,True,True
4,consolidation_lag_2sec_006,heldout_source_006,lag,lag_2sec,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True,structured_json,5318,25.2175,True,True
5,consolidation_lag_2sec_009,heldout_source_009,lag,lag_2sec,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True,structured_json,5336,25.2674,True,True
6,consolidation_lag_2sec_012,heldout_source_012,lag,lag_2sec,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True,structured_json,5330,25.2408,True,True
7,consolidation_lag_3sec_005,heldout_source_005,lag,lag_3sec,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True,structured_json,5290,25.2431,True,True
8,consolidation_lag_3sec_008,heldout_source_008,lag,lag_3sec,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True,structured_json,5295,25.1856,True,True
9,consolidation_lag_3sec_013,heldout_source_013,lag,lag_3sec,ANOMALOUS,ANOMALOUS,VALID,NORMAL,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True,structured_json,5355,25.1085,True,True



FINE-TUNED F1 — LAG MISSED

SUBSET SUMMARY


...

,cases,pred_NORMAL,pred_ANOMALOUS,correct,incorrect
0,17,0,17,17,0



INDIVIDUAL CASES


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,semantic_assessment,decisive_dimension,schema_exact,parse_mode,input_token_count,elapsed_seconds,valid_prediction,correct
0,consolidation_silent_partner_000,heldout_source_000,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,SEMANTIC,True,structured_json,5178,25.1747,True,True
1,consolidation_silent_partner_001,heldout_source_001,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True,structured_json,5268,25.1926,True,True
2,consolidation_silent_partner_002,heldout_source_002,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True,structured_json,5312,25.2635,True,True
3,consolidation_silent_partner_003,heldout_source_003,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True,structured_json,5195,25.3129,True,True
4,consolidation_silent_partner_004,heldout_source_004,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True,structured_json,5210,25.2011,True,True
5,consolidation_silent_partner_005,heldout_source_005,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True,structured_json,5253,25.3393,True,True
6,consolidation_silent_partner_006,heldout_source_006,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True,structured_json,5287,25.3255,True,True
7,consolidation_silent_partner_007,heldout_source_007,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True,structured_json,5117,25.2080,True,True
8,consolidation_silent_partner_008,heldout_source_008,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True,structured_json,5207,25.0919,True,True
9,consolidation_silent_partner_009,heldout_source_009,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True,structured_json,5207,25.2355,True,True



FINE-TUNED F1 — SILENT PARTNER MISSED
No cases found in this subset.


,case_id,source_group_id,case_family,case_variant,gold_binary_label,prompt_sha256,input_payload_sha256,input_token_count,raw_output,prediction,...,schema_exact,schema_errors,parse_mode,parsed_output,generation_error,elapsed_seconds,completed_at_utc,gold_label,valid_prediction,correct


## 18. Optional Colab runtime disconnection

In [ ]:
from google.colab import runtime

runtime.unassign()
